In [ ]:
# =================================
# Common Utilities Functions
# =================================
import vectorbt as vbt
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os, re, json, glob
import time
import pandas as pd
import requests
from datetime import datetime, timedelta
from plotly.subplots import make_subplots
from email import encoders
from plotly.graph_objs import Figure as Figure
import ace_tools_open as tools
import os, shutil
from pathlib import Path
import itables

from typing import Dict, Iterable, List, Tuple, Optional, Mapping
from typing import Union, Dict, Any

import statsmodels.api as sm
import plotly.graph_objs as go
from functools import reduce
import calendar


# Struttura del progetto relativa a runtime o dev dir
_TSLAB_INPUNTS_DIR="../../inputs/"
_TSLAB_OUTPUTS_DIR="../../outputs"

_TSLAB_RUNTIME_T_WFO_RESULTS_DIR=f"{_TSLAB_INPUNTS_DIR}/WFO_T_RUN_RESULTS"
_TSLAB_RUNTIME_R_WFO_RESULTS_DIR=f"{_TSLAB_INPUNTS_DIR}/WFO_R_RUN_RESULTS"

_TSLAB_DEV_T_WFO_RESULTS_DIR=f"{_TSLAB_OUTPUTS_DIR}/WFO_T_DEV_RESULTS"
_TSLAB_DEV_R_WFO_RESULTS_DIR=f"{_TSLAB_OUTPUTS_DIR}/WFO_R_DEV_RESULTS"

_TSLAB_CACHE_DIR="../../cache"

# Dimensioni standard dei grafici prodotti dai metodi plot di vectorbt
vbt_plot_width=1100


#
# Cosmesi
# 
class Emoji:
    # 📌 Trading e Mercati
    UP = "📈"  # Mercato in crescita
    DOWN = "📉"  # Mercato in calo
    FIRE = "🔥"  # Movimento di mercato forte
    MONEY = "💰"  # Guadagni
    ALERT = "⚠️"  # Attenzione a condizioni critiche
    WARNING = "⚠️"  # Avviso di rischio
    INFO = "ℹ️"  # Informazioni generali
    CHECK = "✅"  # Controllo / Condizione soddisfatta
    SEARCH = "🔎"  # Analisi dati
    STAR = "⭐"  # Asset di valore / Performance eccellente
    CHART = "📊"  # Grafico e analisi dati
    STRATEGY = "🧪"  # Test strategia / Backtest
    RISK = "⚠️"  # Rischio alto / Drawdown
    WIN = "🏆"  # Ottima performance
    FAIL = "❌"  # Errore / Trade fallito
    ROCKET = "🚀"  # Forte crescita / Rally
    PARAMS = "🛠️"  # Parametri di strategia / Setup
    MACRO = "🏛️"  # Mercato macroeconomico / Fondamentali
    MOMENTUM = "⏳"  # Momentum e Timing
    REBALANCE = "🔄"  # Ribilanciamento del portafoglio
    DIVIETO = "🚫" # Divieto / Nessun dato valido
    
    # ▶️ Esecuzione e Setup (Run)
    RUN = "▶️"  # Avvio processo
    FAST_RUN = "⏩"  # Run veloce / Ottimizzazione
    POWER_RUN = "⚡"  # Computazione rapida
    LOOP = "🔄"  # Loop / Esecuzione ripetuta
    EXECUTE = "🚀"  # Run strategia con alte prestazioni
    PROCESS = "🏃"  # Processo in esecuzione
    SETUP = "🛠️"  # Setup e configurazione
    CALENDAR = "📅" # Data di esecuzione 

    
# === Stili base ===
RESET   = "\033[0m"
BOLD    = "\033[1m"
DIM     = "\033[2m"
ITALIC  = "\033[3m"
UNDER   = "\033[4m"
BLINK   = "\033[5m"
REVERSE = "\033[7m"

# === Foreground colors (ANSI) ===
BLACK   = "\033[30m"
RED     = "\033[31m"
GREEN   = "\033[32m"
YELLOW  = "\033[33m"
BLUE    = "\033[34m"
MAGENTA = "\033[35m"
CYAN    = "\033[36m"
WHITE   = "\033[37m"

BRIGHT_BLACK   = "\033[90m"
BRIGHT_RED     = "\033[91m"
BRIGHT_GREEN   = "\033[92m"
BRIGHT_YELLOW  = "\033[93m"
BRIGHT_BLUE    = "\033[94m"
BRIGHT_MAGENTA = "\033[95m"
BRIGHT_CYAN    = "\033[96m"
BRIGHT_WHITE   = "\033[97m"

# === helper YTD / anno corrente (robusto se mancano util esterni) ===
def _ytd_start():
    try:
        return ytd()  # se l'hai già definita
    except Exception:
        now = pd.Timestamp.now(tz=None)
        return pd.Timestamp(year=now.year, month=1, day=1)

def _now_year():
    try:
        return now().year  # se hai un helper now()
    except Exception:
        return pd.Timestamp.now(tz=None).year


def now(): return datetime.now()
today = now

def ytd_date (): return f"{now().year}-01-01"
ytd = ytd_date

###############################################################################
# Code Check
###############################################################################
def delete_paths(paths, dry_run=True):
    """
    paths: iterable di stringhe/Path
    dry_run=True -> non cancella, mostra soltanto cosa farebbe
    Ritorna: (deleted, failed) liste di Path
    """
    deleted, failed = [], []
    for p in map(Path, paths):
        try:
            if not p.exists():
                failed.append((p, "not found"))
                continue
            if dry_run:
                deleted.append((p, "would delete dir" if p.is_dir() else "would delete file"))
                continue
            if p.is_dir():
                shutil.rmtree(p)
            else:
                # se è un symlink, rimuove il link
                p.unlink()
            deleted.append((p, "deleted"))
        except Exception as e:
            failed.append((p, repr(e)))
    return deleted, failed


# match di una def top-level: "def nome(...):"
_DEF_RE = re.compile(r'^def\s+([A-Za-z_]\w*)\s*\(')

def _load_ipynb_code_lines(path):
    """Ritorna lista di (linea, cell_idx, line_in_cell) per un .ipynb."""
    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)
    lines = []
    for ci, cell in enumerate(nb.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue
        src = cell.get("source", [])
        if isinstance(src, str):
            src = src.splitlines(True)
        for li, line in enumerate(src, start=1):
            # ignora magics/shell
            if line.lstrip().startswith(("%", "!", "?")):
                continue
            lines.append((line.rstrip("\n"), ci, li))
    return lines

def _load_py_code_lines(path):
    """Ritorna lista di (linea, None, lineno) per un .py."""
    with open(path, "r", encoding="utf-8") as f:
        src = f.read().splitlines()
    return [(line, None, i+1) for i, line in enumerate(src)]

def find_duplicate_function_defs(path, prefix=None, top_level_only=True, show=True):
    """
    Scansiona un .py o .ipynb e segnala funzioni definite più volte.
    - prefix: filtra per prefisso (es. 'strategy_'); None = tutte.
    - top_level_only: True = considera solo 'def' senza indentazione (spazio globale).
    Ritorna: dict {nome: [posizioni]} con posizioni 'C<cell>:L<line>' o 'L<line>'.
    """
    path = os.fspath(path)
    if path.endswith(".ipynb"):
        lines = _load_ipynb_code_lines(path)
    else:
        lines = _load_py_code_lines(path)

    seen = {}   # nome -> list di posizioni
    for line, cell_idx, line_no in lines:
        if top_level_only and (line.startswith(" ") or line.startswith("\t")):
            continue
        m = _DEF_RE.match(line)
        if not m:
            continue
        name = m.group(1)
        if prefix and not name.startswith(prefix):
            continue
        pos = f"C{cell_idx}:L{line_no}" if cell_idx is not None else f"L{line_no}"
        seen.setdefault(name, []).append(pos)

    duplicates = {n: locs for n, locs in seen.items() if len(locs) > 1}

    if show:
        base = os.path.basename(path)
        if duplicates:
            print(f"❌ Duplicati trovati in {base}:")
            for n, locs in sorted(duplicates.items()):
                print(f"  {n}: definita {len(locs)} volte a {locs}")
        else:
            print(f"✅ Nessuna funzione duplicata in {base}")

    return duplicates

def _scan_file(path, prefix=None, top_level_only=True, collect_all=False):
    """Ritorna (duplicates, name_positions) per un singolo file."""
    if path.endswith(".ipynb"):
        lines = _load_ipynb_code_lines(path)
    else:
        lines = _load_py_code_lines(path)

    seen = {}          # name -> [pos]
    for line, cell_idx, line_no in lines:
        if top_level_only and (line.startswith(" ") or line.startswith("\t")):
            continue
        m = _DEF_RE.match(line)
        if not m:
            continue
        name = m.group(1)
        if prefix and not name.startswith(prefix):
            continue
        pos = f"C{cell_idx}:L{line_no}" if cell_idx is not None else f"L{line_no}"
        seen.setdefault(name, []).append(pos)

    duplicates = {n: locs for n, locs in seen.items() if len(locs) > 1}
    return duplicates, seen if collect_all else None

def find_duplicate_function_defs_multi(patterns, prefix=None, top_level_only=True,
                                       recursive=True, show=True):
    """
    patterns: stringa con wildcard e/o più pattern separati da virgola o spazi,
              oppure lista di pattern (es. ['**/*strategies.ipynb','**/*functions.ipynb']).
    Ritorna: (per_file_duplicates, cross_file_repeats)
    """
    # normalizza i pattern
    if isinstance(patterns, str):
        pats = [p.strip() for p in re.split(r'[,\s]+', patterns) if p.strip()]
    else:
        pats = list(patterns)

    # espandi i file
    files = []
    for p in pats:
        files.extend(glob.glob(p, recursive=recursive))
    files = sorted(set(f for f in files if f.endswith((".ipynb", ".py"))))

    per_file = {}
    name_positions_global = {}  # name -> [(file, pos), ...]

    for path in files:
        dups, name_pos = _scan_file(path, prefix=prefix, top_level_only=top_level_only, collect_all=True)
        per_file[path] = dups
        for name, positions in name_pos.items():
            for pos in positions:
                name_positions_global.setdefault(name, []).append((path, pos))

    # funzioni definite in più file
    cross_file = {n: locs for n, locs in name_positions_global.items() if len({fp for fp, _ in locs}) > 1}

    if show:
        # print(f"Scansione di {len(files)} file.")
        print("Verifica naming di funzioni...\n")
        for path in files:
            base = os.path.basename(path)
            d = per_file[path]
            if d:
                print(f"❌ Duplicati in {base}:")
                for n, locs in sorted(d.items()):
                    print(f"  {n}: {locs}")
            else:
                print(f"✅ Nessun duplicato in {base}")
        if cross_file:
            print("\n⚠️ Funzioni con LO STESSO NOME presenti in file diversi:")
            for n, locs in sorted(cross_file.items()):
                pretty = [f"{os.path.basename(fp)}:{pos}" for fp, pos in locs]
                print(f"  {n}: {pretty}")
        else:
            print("\n✅ Nessun nome di funzione ripetuto su file diversi.")

    return per_file, cross_file

## Funzioni di download

In [ ]:
###############################################################################
# Download Financial Data 
###############################################################################
def load_ohlcv(symbol: str, 
               start: str =  None, 
               end: str = None, 
               show_progress: bool = False,
               auto_adjust: bool = True,
               multi_level_index: bool = False,
               interval: str = "1d",) -> pd.DataFrame:
    
    """
    Scarica dati OHLCV da yfinance con indice DatetimeIndex.

    Params
    ------
    symbol : str
    start  : str | None
    end    : str | None
    show_progress : bool
        Se True, mostra la barra di progresso durante il download (quando supportato).
    interval : str
        Intervallo yfinance (default "1d").

    Ritorna
    -------
    pd.DataFrame con colonne tipiche: ['Open','High','Low','Close','Adj Close','Volume']
    """
    
    
    df = yf.download(symbol, 
                 start=start, 
                 end=end, 
                 multi_level_index=multi_level_index, 
                 auto_adjust=auto_adjust,
                 progress=show_progress,
                 interval=interval)
    

    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df


get_clean_financial_data = load_ohlcv
    
def download_data(tickers, start_date=None, end_date=None, auto_adjust=True, show_progress=False):
    """Scarica i dati storici dei prezzi per una lista di ticker."""
    # return vbt.YFData.download(tickers, start=start_date, end=end_date).get('Close')
    return load_ohlcv(tickers, start=start_date, end=end_date,auto_adjust=auto_adjust,show_progress=show_progress).Close
def load_isin_overrides(path: str = None) -> dict:
    """Carica mapping manuale ticker → ISIN da CSV statico.

    Formato CSV: due colonne 'Ticker' e 'ISIN'.
    Gli override manuali hanno priorità anche sulla cache esistente.
    """
    if path is None:
        path = f"{_TSLAB_CACHE_DIR}/ticker_isin_overrides.csv"

    if not os.path.exists(path):
        return {}

    try:
        df = pd.read_csv(path, dtype=str)

        if "Ticker" not in df.columns or "ISIN" not in df.columns:
            return {}

        df["Ticker"] = df["Ticker"].str.strip()
        df["ISIN"] = df["ISIN"].str.strip()

        df = df.dropna(subset=["Ticker", "ISIN"])
        df = df[(df["Ticker"] != "") & (df["ISIN"] != "")]

        return dict(zip(df["Ticker"], df["ISIN"]))

    except Exception:
        return {}


def build_company_df_with_cache(
    tickers,
    cache_path: str = None,
    expire_days=250,
    max_retries=3,
    backoff_factor=1.5
):
    """
    Costruisce o ricarica da cache un DataFrame con Company, marketCap e ISIN.

    Priorità ISIN:
    1) ticker_isin_overrides.csv
    2) cache esistente
    3) yfinance.isin

    Gli override manuali vengono applicati SEMPRE alla cache prima di calcolare
    i ticker da rifetchare, quindi hanno priorità anche su valori già presenti
    in company_cache.csv.
    """
    if cache_path is None:
        cache_path = f"{_TSLAB_CACHE_DIR}/company_cache.csv"

    isin_overrides = load_isin_overrides()

    # 1. Carica la cache se esiste
    if os.path.exists(cache_path):
        cache = pd.read_csv(cache_path, parse_dates=["DateFetched"], index_col="Ticker")

        if "Company" not in cache.columns:
            cache["Company"] = pd.NA
        if "marketCap" not in cache.columns:
            cache["marketCap"] = pd.NA
        if "ISIN" not in cache.columns:
            cache["ISIN"] = pd.NA
        if "DateFetched" not in cache.columns:
            cache["DateFetched"] = pd.NaT

    else:
        cache = pd.DataFrame(columns=["Company", "marketCap", "ISIN", "DateFetched"])
        cache.index.name = "Ticker"

    # 2. Applica SEMPRE gli override manuali alla cache già caricata
    cache_changed_by_overrides = False

    for tk in tickers:
        if tk in isin_overrides and tk in cache.index:
            override_isin = isin_overrides[tk]

            if override_isin and pd.notna(override_isin):
                old_isin = cache.at[tk, "ISIN"]

                if pd.isna(old_isin) or old_isin != override_isin:
                    cache.at[tk, "ISIN"] = override_isin
                    cache_changed_by_overrides = True

    if cache_changed_by_overrides:
        cache.to_csv(cache_path)

    # 3. Identifica quali ticker rifetchare
    cutoff = pd.Timestamp.now() - pd.Timedelta(days=expire_days)

    to_fetch = [
        tk for tk in tickers
        if tk not in cache.index
        or pd.isna(cache.at[tk, "DateFetched"])
        or cache.at[tk, "DateFetched"] < cutoff
        or pd.isna(cache.at[tk, "ISIN"])
    ]

    # 4. Funzione di retry per fetch info
    def fetch_info_with_retry(tk_obj):
        delay = 1.0

        for attempt in range(max_retries):
            try:
                return tk_obj.info
            except (HTTPError, Exception):
                time.sleep(delay)
                delay *= backoff_factor

        return {}

    # 5. Funzione ISIN con priorità override → yfinance
    def fetch_isin_safe(tk_obj, ticker_str):
        if ticker_str in isin_overrides:
            override_isin = isin_overrides[ticker_str]

            if override_isin and pd.notna(override_isin):
                return override_isin

        if tk_obj is None:
            return None

        try:
            isin = tk_obj.isin

            if isin and isin != "-":
                return isin

        except Exception:
            pass

        return None

    # 6. Batch fetch con fallback
    if to_fetch:
        try:
            tk_objs = yf.Tickers(" ".join(to_fetch)).tickers
            fetched = {}

            for tk in to_fetch:
                obj = tk_objs.get(tk)

                if obj is None:
                    fetched[tk] = {
                        "Company": pd.NA,
                        "marketCap": pd.NA,
                        "ISIN": fetch_isin_safe(None, tk),
                        "DateFetched": pd.Timestamp.now()
                    }
                    continue

                info = fetch_info_with_retry(obj) or {}

                fetched[tk] = {
                    "Company": info.get("longName") or info.get("shortName"),
                    "marketCap": info.get("marketCap"),
                    "ISIN": fetch_isin_safe(obj, tk),
                    "DateFetched": pd.Timestamp.now()
                }

        except Exception:
            fetched = {}

            for tk in to_fetch:
                try:
                    obj = yf.Ticker(tk)
                    info = fetch_info_with_retry(obj) or {}
                except Exception:
                    obj = None
                    info = {}

                fetched[tk] = {
                    "Company": info.get("longName") or info.get("shortName"),
                    "marketCap": info.get("marketCap"),
                    "ISIN": fetch_isin_safe(obj, tk),
                    "DateFetched": pd.Timestamp.now()
                }

        # 7. Aggiorna cache
        fetched_df = pd.DataFrame.from_dict(fetched, orient="index")
        fetched_df.index.name = "Ticker"

        cache.update(fetched_df)

        new_idx = fetched_df.index.difference(cache.index)
        if not new_idx.empty:
            cache = pd.concat([cache, fetched_df.loc[new_idx]])

        # Riapplica override dopo il fetch, così vincono anche su yfinance
        for tk in tickers:
            if tk in isin_overrides and tk in cache.index:
                override_isin = isin_overrides[tk]

                if override_isin and pd.notna(override_isin):
                    cache.at[tk, "ISIN"] = override_isin

        cache.to_csv(cache_path)

    # 8. Prepara il risultato ordinato come tickers in ingresso
    result = cache.reindex(tickers)[["Company", "marketCap", "ISIN"]]

    return result
    
def load_isin_overrides_BAD(path: str = None) -> dict:
    """Carica mapping manuale ticker → ISIN da CSV statico.
    
    Formato CSV: due colonne 'Ticker' e 'ISIN'.
    Usato come override prioritario quando yfinance.isin non funziona
    (tipicamente ticker europei non-anglosassoni).
    """
    if path is None:
        path = f"{_TSLAB_CACHE_DIR}/ticker_isin_overrides.csv"
    if not os.path.exists(path):
        return {}
    try:
        df = pd.read_csv(path, dtype=str)
        return dict(zip(df["Ticker"], df["ISIN"]))
    except Exception:
        return {}

def build_company_df_with_cache_BAD(
    tickers,
    cache_path: str = None,
    expire_days=250,
    max_retries=3,
    backoff_factor=1.5
):
    if cache_path is None:
        cache_path = f"{_TSLAB_CACHE_DIR}/company_cache.csv"
    
    """
    Costruisce o ricarica da cache un DataFrame con Company, marketCap e ISIN.
    
    ISIN viene risolto con strategia gerarchica:
    1) Lookup in ticker_isin_overrides.csv (override manuale, prioritario)
    2) Fallback a yfinance.isin (esperimentale, funziona bene su US/UK
       ma fallisce su molti ticker europei continentali)
    
    Per aggiungere ISIN a ticker che yfinance non risolve, aggiungere
    una riga in ticker_isin_overrides.csv (in _TSLAB_CACHE_DIR).
    """
    # Carica override manuali una volta
    isin_overrides = load_isin_overrides()
    
    # 1. Carica la cache se esiste
    if os.path.exists(cache_path):
        cache = pd.read_csv(cache_path, parse_dates=["DateFetched"], index_col="Ticker")
        # Retro-compat: aggiungi ISIN se cache pre-esistente non lo aveva
        if "ISIN" not in cache.columns:
            cache["ISIN"] = pd.NA
    else:
        cache = pd.DataFrame(columns=["Company", "marketCap", "ISIN", "DateFetched"])
        cache.index.name = "Ticker"
    
    # 2. Identifica quali ticker (ri)fetchare
    cutoff = pd.Timestamp.now() - pd.Timedelta(days=expire_days)
    to_fetch = [tk for tk in tickers
                if tk not in cache.index 
                or cache.at[tk, "DateFetched"] < cutoff
                or pd.isna(cache.at[tk, "ISIN"])]
    
    # 3. Funzione di retry per fetch info
    def fetch_info_with_retry(tk_obj):
        delay = 1.0
        for attempt in range(max_retries):
            try:
                return tk_obj.info
            except (HTTPError, Exception):
                time.sleep(delay)
                delay *= backoff_factor
        return {}
    
    # 3b. Funzione ISIN con strategia gerarchica (override → yfinance)
    def fetch_isin_safe(tk_obj, ticker_str):
        # Priorità 1: override manuale
        if ticker_str in isin_overrides:
            isin_val = isin_overrides[ticker_str]
            if isin_val and pd.notna(isin_val):
                return isin_val
        # Priorità 2: yfinance (esperimentale)
        if tk_obj is None:
            return None
        try:
            isin = tk_obj.isin
            if isin and isin != '-':
                return isin
        except Exception:
            pass
        return None
    
    # 4. Batch fetch con fallback
    if to_fetch:
        try:
            tk_objs = yf.Tickers(" ".join(to_fetch)).tickers
            fetched = {}
            for tk, obj in tk_objs.items():
                info = fetch_info_with_retry(obj) or {}
                fetched[tk] = {
                    "Company": info.get("longName") or info.get("shortName"),
                    "marketCap": info.get("marketCap"),
                    "ISIN": fetch_isin_safe(obj, tk),
                    "DateFetched": pd.Timestamp.now()
                }
        except Exception:
            fetched = {}
            for tk in to_fetch:
                try:
                    obj = yf.Ticker(tk)
                    info = fetch_info_with_retry(obj) or {}
                except Exception:
                    obj = None
                    info = {}
                fetched[tk] = {
                    "Company": info.get("longName") or info.get("shortName"),
                    "marketCap": info.get("marketCap"),
                    "ISIN": fetch_isin_safe(obj, tk),
                    "DateFetched": pd.Timestamp.now()
                }
        
        # 5. Aggiorna cache
        fetched_df = pd.DataFrame.from_dict(fetched, orient="index")
        fetched_df.index.name = "Ticker"
        cache.update(fetched_df)
        new_idx = fetched_df.index.difference(cache.index)
        if not new_idx.empty:
            cache = pd.concat([cache, fetched_df.loc[new_idx]])
        cache.to_csv(cache_path)
    
    # 6. Prepara il risultato ordinato come tickers in ingresso
    result = cache.reindex(tickers)[["Company", "marketCap", "ISIN"]]
    return result
     
# def build_company_df_with_cache(
#     tickers,
#     cache_path: str = None,
#     expire_days=250,
#     max_retries=3,
#     backoff_factor=1.5
# ):
#     if cache_path is None:
#         cache_path=f"{_TSLAB_CACHE_DIR}/company_cache.csv"
        
#     # 1. Carica la cache se esiste
#     if os.path.exists(cache_path):
#         cache = pd.read_csv(cache_path, parse_dates=["DateFetched"], index_col="Ticker")
#         # Retro-compat: aggiungi ISIN se cache pre-esistente non lo aveva
#         if "ISIN" not in cache.columns:
#             cache["ISIN"] = pd.NA
#     else:
#         cache = pd.DataFrame(columns=["Company", "marketCap", "ISIN", "DateFetched"])
#         cache.index.name = "Ticker"
    
#     # 2. Identifica quali ticker (ri)fetchare
#     cutoff = pd.Timestamp.now() - pd.Timedelta(days=expire_days)
#     to_fetch = [tk for tk in tickers
#                 if tk not in cache.index 
#                 or cache.at[tk, "DateFetched"] < cutoff
#                 or pd.isna(cache.at[tk, "ISIN"])]   # rifetch se ISIN manca
    
#     # 3. Funzione di retry per fetch info di un oggetto yf.Ticker
#     def fetch_info_with_retry(tk_obj):
#         delay = 1.0
#         for attempt in range(max_retries):
#             try:
#                 return tk_obj.info
#             except (HTTPError, Exception):
#                 time.sleep(delay)
#                 delay *= backoff_factor
#         return {}
    
#     # 3b. Funzione separata per ISIN (esperimentale in yfinance, può fallire)
#     def fetch_isin_safe(tk_obj):
#         try:
#             isin = tk_obj.isin
#             if isin and isin != '-':
#                 return isin
#         except Exception:
#             pass
#         return None
    
#     # 4. Batch fetch con fallback
#     if to_fetch:
#         try:
#             tk_objs = yf.Tickers(" ".join(to_fetch)).tickers
#             fetched = {}
#             for tk, obj in tk_objs.items():
#                 info = fetch_info_with_retry(obj) or {}
#                 fetched[tk] = {
#                     "Company": info.get("longName") or info.get("shortName"),
#                     "marketCap": info.get("marketCap"),
#                     "ISIN": fetch_isin_safe(obj),
#                     "DateFetched": pd.Timestamp.now()
#                 }
#         except Exception:
#             fetched = {}
#             for tk in to_fetch:
#                 try:
#                     obj = yf.Ticker(tk)
#                     info = fetch_info_with_retry(obj) or {}
#                 except Exception:
#                     obj = None
#                     info = {}
#                 fetched[tk] = {
#                     "Company": info.get("longName") or info.get("shortName"),
#                     "marketCap": info.get("marketCap"),
#                     "ISIN": fetch_isin_safe(obj) if obj is not None else None,
#                     "DateFetched": pd.Timestamp.now()
#                 }
        
#         # 5. Aggiorna cache
#         fetched_df = pd.DataFrame.from_dict(fetched, orient="index")
#         fetched_df.index.name = "Ticker"
#         cache.update(fetched_df)
#         new_idx = fetched_df.index.difference(cache.index)
#         if not new_idx.empty:
#             cache = pd.concat([cache, fetched_df.loc[new_idx]])
#         cache.to_csv(cache_path)
    
#     # 6. Prepara il risultato ordinato come tickers in ingresso
#     result = cache.reindex(tickers)[["Company", "marketCap", "ISIN"]]
#     return result
    
def build_company_df_with_cache_R1(
    tickers,
    cache_path: str = None,
    expire_days=250,
    max_retries=3,
    backoff_factor=1.5
):

    if cache_path is None:
        cache_path=f"{_TSLAB_CACHE_DIR}/company_cache.csv"
        
    """
    Costruisce o ricarica da cache un DataFrame con Company e marketCap per ciascun ticker.
    La cache è un CSV con colonne: Ticker, Company, marketCap, DateFetched.
    Se il record è più vecchio di `expire_days`, lo rifetchiamo.
    In caso di errori HTTP tenta retry esponenziale; se fallisce batch, ricade su singolo ticker.
    """
    # 1. Carica la cache se esiste
    if os.path.exists(cache_path):
        cache = pd.read_csv(cache_path, parse_dates=["DateFetched"], index_col="Ticker")
    else:
        cache = pd.DataFrame(columns=["Company", "marketCap", "DateFetched"])
        cache.index.name = "Ticker"

    # 2. Identifica quali ticker (ri)fetchare
    cutoff = pd.Timestamp.now() - pd.Timedelta(days=expire_days)
    to_fetch = [tk for tk in tickers
                if tk not in cache.index or cache.at[tk, "DateFetched"] < cutoff]

    # 3. Funzione di retry per fetch info di un oggetto yf.Ticker
    def fetch_info_with_retry(tk_obj):
        delay = 1.0
        for attempt in range(max_retries):
            try:
                return tk_obj.info
            except (HTTPError, Exception):
                time.sleep(delay)
                delay *= backoff_factor
        return {}

    # 4. Batch fetch con fallback
    if to_fetch:
        try:
            # Proviamo in batch
            tk_objs = yf.Tickers(" ".join(to_fetch)).tickers
            fetched = {}
            for tk, obj in tk_objs.items():
                info = fetch_info_with_retry(obj) or {}
                fetched[tk] = {
                    "Company": info.get("longName") or info.get("shortName"),
                    "marketCap": info.get("marketCap"),
                    "DateFetched": pd.Timestamp.now()
                }
        except Exception:
            # Batch fallito → fallback ticker-per-ticker
            fetched = {}
            for tk in to_fetch:
                try:
                    info = fetch_info_with_retry(yf.Ticker(tk)) or {}
                except Exception:
                    info = {}
                fetched[tk] = {
                    "Company": info.get("longName") or info.get("shortName"),
                    "marketCap": info.get("marketCap"),
                    "DateFetched": pd.Timestamp.now()
                }

        # 5. Aggiorna cache
        fetched_df = pd.DataFrame.from_dict(fetched, orient="index")
        fetched_df.index.name = "Ticker"
        cache.update(fetched_df)
        # Aggiungi eventuali ticker completamente nuovi
        new_idx = fetched_df.index.difference(cache.index)
        if not new_idx.empty:
            cache = pd.concat([cache, fetched_df.loc[new_idx]])
        # Salva
        cache.to_csv(cache_path)

    # 6. Prepara il risultato ordinato come tickers in ingresso
    result = cache.reindex(tickers)[["Company", "marketCap"]]
    return result

def fetch_data_and_companies(
    tickers,
    start_date=None,
    end_date=None,
    show_progress=False,
    normalize: bool = False,
    min_overlap_cols: int = 2,
    verbose: bool = True,
    auto_adjust: bool = True,
    # ----------------------------
    # NEW (legacy-safe)
    enrich_companies: bool = False,
    enrich_fields: tuple = ("marketCap", "priceToBook", "bookValue", "trailingPE", "forwardPE", "enterpriseValue"),
    enrich_pause: float = 0.15,
    enrich_max_retries: int = 2
):
    """
    Scarica dati e anagrafica società.

    Se normalize=True:
      - prova intersection-mode sull'intervallo comune
      - se non esiste overlap, elimina progressivamente colonne “incompatibili”
      - se ancora non c'è overlap con ≥ min_overlap_cols, fallback union-mode (solo fill)
      - in ogni caso esegue bfill() poi ffill()
      - stampa il ticker responsabile del taglio iniziale (max first_valid_index) e finale (min last_valid_index)

    NEW (legacy-safe):
      - enrich_companies=True: arricchisce company_data con campi fondamentali extra (se disponibili)
      - di default enrich_companies=False => comportamento IDENTICO a prima

    Return:
      - se normalize=False: (stocks_data, company_data)
      - se normalize=True:  (stocks_data, company_data, common_start_date, common_end_date)
        dove common_start_date/common_end_date sono le date COMUNI finali (intersection) se trovate,
        altrimenti (union fallback) sono (df.index.min(), df.index.max()) dopo fill.
        Se df è vuoto: (None, None).
    """

    import time
    import numpy as np
    import pandas as pd
    import yfinance as yf

    stocks_data = download_data(
        tickers, start_date=start_date, end_date=end_date, show_progress=show_progress, auto_adjust=auto_adjust
    )

    common_start_date = None
    common_end_date = None

    if normalize and isinstance(stocks_data, pd.DataFrame):
        df = stocks_data.copy().sort_index()

        # 1) Rimuovi colonne totalmente vuote
        first_valid = df.apply(lambda s: s.first_valid_index())
        last_valid  = df.apply(lambda s: s.last_valid_index())
        fully_nan_cols = first_valid[first_valid.isna()].index.tolist()
        if fully_nan_cols:
            if verbose:
                print(f"[Normalizzazione] Colonne senza dati totali rimosse: {fully_nan_cols}")
            df = df.drop(columns=fully_nan_cols)

            if df.empty:
                if verbose:
                    print("[Normalizzazione] Nessuna colonna residua dopo la rimozione. Salto normalizzazione.")
                company_data = build_company_df_with_cache(tickers)
                # normalize=True => ritorna anche le date comuni (None, None)
                return df, company_data, None, None

            first_valid = df.apply(lambda s: s.first_valid_index())
            last_valid  = df.apply(lambda s: s.last_valid_index())

        # 2) Tentativo: intersection mode con rimozione iterativa
        keep_cols = df.columns.tolist()
        changed = True
        while changed and len(keep_cols) >= min_overlap_cols:
            changed = False
            fv = first_valid[keep_cols]
            lv = last_valid[keep_cols]

            # calcolo finestra comune
            common_start = max(fv.dropna())
            common_end   = min(lv.dropna())

            if common_start <= common_end:
                break  # overlap trovato

            # identifica le colonne peggiori
            worst_fv_col = fv.idxmax()  # inizia più tardi → stringe lo start
            worst_lv_col = lv.idxmin()  # finisce prima → stringe la fine

            # euristica: rimuovi quella che peggiora di più
            to_drop = worst_fv_col if fv[worst_fv_col] >= lv[worst_lv_col] else worst_lv_col
            keep_cols.remove(to_drop)
            changed = True
            if verbose:
                print(f"[Normalizzazione] Nessun overlap: rimuovo '{to_drop}' e riprovo. Colonne rimaste: {len(keep_cols)}")

        if len(keep_cols) >= min_overlap_cols:
            # Overlap trovato → stampa “responsabili del taglio”
            fv = first_valid[keep_cols]
            lv = last_valid[keep_cols]
            common_start = max(fv.dropna())
            common_end   = min(lv.dropna())

            culprit_start = fv.idxmax()  # responsabile del taglio iniziale
            culprit_end   = lv.idxmin()  # responsabile del taglio finale

            orig_start, orig_end = df.index.min(), df.index.max()
            if verbose:
                print(
                    f"[Normalizzazione] Intervallo comune: {common_start.date()} → {common_end.date()} "
                    f"(prima: {orig_start.date()} → {orig_end.date()}). Colonne mantenute: {len(keep_cols)}"
                )
                print(
                    f"[Normalizzazione] Responsabile del taglio START: '{culprit_start}' "
                    f"(first_valid_index = {fv[culprit_start].date()})"
                )
                print(
                    f"[Normalizzazione] Responsabile del taglio END:   '{culprit_end}' "
                    f"(last_valid_index  = {lv[culprit_end].date()})"
                )

            df = df[keep_cols].loc[common_start:common_end]

            # fill
            na_before = int(df.isna().sum().sum())
            if na_before > 0:
                df = df.bfill().ffill()
                na_after = int(df.isna().sum().sum())
                if verbose:
                    print(f"[Normalizzazione] Fill eseguito (intersection): NaN prima={na_before}, dopo={na_after}.")

            # date comuni finali (intersection)
            common_start_date = df.index.min() if not df.empty else None
            common_end_date   = df.index.max() if not df.empty else None

        else:
            # 3) Fallback: union mode (solo fill) + stampa “colpevoli” informativa sulle colonne originali
            if verbose:
                try:
                    culprit_start_all = first_valid.idxmax()
                    culprit_end_all   = last_valid.idxmin()
                    print(
                        f"[Normalizzazione] Fallback UNION: impossibile trovare overlap con ≥{min_overlap_cols} colonne."
                        f" Candidati che avrebbero tagliato di più → START: '{culprit_start_all}' ({first_valid[culprit_start_all].date()}), "
                        f"END: '{culprit_end_all}' ({last_valid[culprit_end_all].date()})."
                    )
                except Exception:
                    pass

            na_before = int(df.isna().sum().sum())
            df = df.bfill().ffill()
            na_after = int(df.isna().sum().sum())
            if verbose:
                print(f"[Normalizzazione] Fill eseguito (union): NaN prima={na_before}, dopo={na_after}.")

            # in union mode non c'è "intervallo comune" stretto: ritorniamo l'intervallo effettivo del df finale
            common_start_date = df.index.min() if not df.empty else None
            common_end_date   = df.index.max() if not df.empty else None

        stocks_data = df

    # --- base company data (legacy) ---
    company_data = build_company_df_with_cache(tickers)

    # ------------------------------------------------------------
    # NEW: optional enrichment (legacy-safe)
    # ------------------------------------------------------------
    if enrich_companies:
        # company_data expected indexed by ticker (your current format)
        # We'll add columns only; no breaking changes.
        if verbose:
            print(f"[Companies] Enrichment enabled. Fields={enrich_fields}")

        # make sure index are tickers
        if 'Ticker' in company_data.columns:
            company_data = company_data.set_index('Ticker')

        for tk in tickers:
            for attempt in range(enrich_max_retries):
                try:
                    info = yf.Ticker(tk).info or {}
                    row_updates = {}
                    for fld in enrich_fields:
                        val = info.get(fld, np.nan)
                        row_updates[fld] = val
                    for fld, val in row_updates.items():
                        company_data.loc[tk, fld] = val
                    break
                except Exception as e:
                    if attempt + 1 >= enrich_max_retries and verbose:
                        print(f"[Companies] Enrich failed for {tk}: {e}")
                    time.sleep(enrich_pause)
            time.sleep(enrich_pause)

        # try to coerce numerics where possible
        for c in company_data.columns:
            if c in enrich_fields or c == 'marketCap':
                company_data[c] = pd.to_numeric(company_data[c], errors='ignore')

    if normalize:
        return stocks_data, company_data, common_start_date, common_end_date
    return stocks_data, company_data

## Reportistica, Analisi della performance

In [ ]:
###############################################################################
# Reportistica e Analisi Performance/Statistiche
###############################################################################

#
# Nuove funzioni di performance/statistiche
#

# ======================================================================
# Unified refactor + full plotting + rolling alpha
# ======================================================================

# import statsmodels.api as sm

# -------------------------
# Helpers: normalizzazione indici e conversioni
# -------------------------
def _normalize_series_idx(s: pd.Series) -> pd.Series:
    s = s.copy()
    if getattr(s.index, "tz", None) is not None:
        s.index = s.index.tz_localize(None)
    s.index = pd.to_datetime(s.index).normalize()
    s = s[~s.index.duplicated(keep="last")]
    return s.sort_index()

def _pf_returns_series(pf) -> pd.Series:
    """Aggregated daily simple returns series from a vectorbt Portfolio."""
    r = pf.returns(group_by=True)
    # If DataFrame, try to aggregate sensibly (should usually be Series)
    if isinstance(r, pd.DataFrame):
        # If grouped returns appear as columns (unlikely), sum as fallback
        try:
            # If it's multi-column, but represents groups, prefer single-column behaviour
            if r.shape[1] == 1:
                r = r.iloc[:, 0]
            else:
                # try pf.value() fallback
                v = pf.value()
                if isinstance(v, pd.DataFrame):
                    v_sum = v.sum(axis=1)
                else:
                    v_sum = v
                r = _normalize_series_idx(v_sum).pct_change().dropna()
                return r
        except Exception:
            v = pf.value()
            if isinstance(v, pd.DataFrame):
                v_sum = v.sum(axis=1)
            else:
                v_sum = v
            r = _normalize_series_idx(v_sum).pct_change().dropna()
            return r
    r = r.dropna()
    return _normalize_series_idx(r.astype(float))

def _prices_to_returns_align(prices: pd.Series, idx: pd.DatetimeIndex) -> pd.Series:
    """Convert price series to daily pct returns and reindex/ffill on idx."""
    p = _normalize_series_idx(prices.dropna().astype(float))
    br = p.pct_change().dropna()
    br = br.reindex(idx, method="ffill").fillna(0.0)
    return br

# -------------------------
# Resolve benchmark returns (single source of truth)
# -------------------------
def resolve_benchmark_returns(
    pf,
    benchmark_mode: str = "internal",         # "internal" | "external" | "portfolio"
    benchmark_name: str = "Benchmark",
    benchmark_data: Optional[pd.Series] = None, # prices (Close) for external
    benchmark_portfolio=None
) -> Tuple[pd.Series, dict]:
    """
    Returns (benchmark_returns_series_aligned_to_pf_index, meta)
    meta contains benchmark_source and notes.
    """
    pf_ret = _pf_returns_series(pf)
    idx = pf_ret.index
    meta = {"benchmark_source": None, "benchmark_name": benchmark_name, "notes": ""}

    if benchmark_mode == "portfolio":
        if benchmark_portfolio is None:
            raise ValueError("benchmark_mode='portfolio' requires benchmark_portfolio.")
        bm = benchmark_portfolio.returns(group_by=True)
        if isinstance(bm, pd.DataFrame):
            bm = bm.sum(axis=1)
        bm = _normalize_series_idx(bm.dropna().astype(float))
        bm = bm.reindex(idx, method="ffill").fillna(0.0)
        meta["benchmark_source"] = "portfolio"
        return bm, meta

    if benchmark_mode == "external":
        if benchmark_data is None:
            raise ValueError("benchmark_mode='external' requires benchmark_data (price series).")
        bm = _prices_to_returns_align(benchmark_data, idx)
        meta["benchmark_source"] = "external"
        return bm, meta

    # default internal
    try:
        bm = pf.benchmark_returns(group_by=True)
    except Exception:
        # fallback: try to build B&H equal weighted from pf.assets if available
        try:
            assets = pf.assets()
            if isinstance(assets, pd.DataFrame):
                assets_cols = assets.columns.tolist()
            else:
                assets_cols = None
            # not attempting heavy fallback here; keep it simple
            bm = pf.benchmark_returns(group_by=True)
        except Exception:
            raise
    if isinstance(bm, pd.DataFrame):
        bm = bm.sum(axis=1)
    bm = _normalize_series_idx(bm.dropna().astype(float))
    bm = bm.reindex(idx, method="ffill").fillna(0.0)
    meta["benchmark_source"] = "internal(pf.benchmark_returns)"
    return bm, meta

# -------------------------
# CAPM single-window
# -------------------------
def capm_alpha_beta(ret: pd.Series, bm: pd.Series, risk_free_rate: float = 0.02,
                    annualization: int = 252, min_obs: int = 30) -> dict:
    rf_daily = (1 + risk_free_rate) ** (1 / annualization) - 1
    df = pd.concat({"ret": ret, "bm": bm}, axis=1).dropna()
    if df.shape[0] < min_obs:
        return {"alpha_daily": np.nan, "alpha_ann_pct": np.nan, "beta": np.nan,
                "t_alpha": np.nan, "p_alpha": np.nan, "te_ann": np.nan, "corr": np.nan, "n_obs": int(df.shape[0])}
    df["excess_ret"] = df["ret"] - rf_daily
    df["excess_bm"] = df["bm"] - rf_daily
    X = sm.add_constant(df["excess_bm"])
    y = df["excess_ret"]
    model = sm.OLS(y, X).fit()
    alpha = float(model.params["const"])
    beta = float(model.params["excess_bm"])
    t_a = float(model.tvalues["const"])
    p_a = float(model.pvalues["const"])
    te = float(model.resid.std() * np.sqrt(annualization))
    corr = float(df["ret"].corr(df["bm"]))
    return {"alpha_daily": alpha, "alpha_ann_pct": alpha * annualization * 100.0, "beta": beta,
            "t_alpha": t_a, "p_alpha": p_a, "te_ann": te, "corr": corr, "n_obs": int(df.shape[0])}

# -------------------------
# Rolling CAPM
# -------------------------
def rolling_capm_alpha_beta(ret: pd.Series, bm: pd.Series, window: int = 252,
                            min_periods: Optional[int] = None, risk_free_rate: float = 0.02,
                            annualization: int = 252) -> pd.DataFrame:
    if min_periods is None:
        min_periods = max(60, window // 3)
    df = pd.concat({"ret": ret, "bm": bm}, axis=1).dropna()
    if df.empty:
        return pd.DataFrame(columns=["alpha_ann_pct", "beta", "t_alpha", "p_alpha"])
    idx = df.index
    rows = []
    rf_daily = (1 + risk_free_rate) ** (1 / annualization) - 1
    for end in range(window, len(idx) + 1):
        widx = idx[end - window:end]
        sub = df.loc[widx]
        if sub.shape[0] < min_periods:
            continue
        y = sub["ret"] - rf_daily
        x = sub["bm"] - rf_daily
        X = sm.add_constant(x)
        m = sm.OLS(y, X).fit()
        alpha_daily = float(m.params["const"])
        alpha_ann_pct = alpha_daily * annualization * 100.0
        beta = float(m.params[x.name] if x.name in m.params else m.params.iloc[1])
        t_alpha = float(m.tvalues["const"])
        p_alpha = float(m.pvalues["const"])
        rows.append([widx[-1], alpha_ann_pct, beta, t_alpha, p_alpha])
    df_roll = pd.DataFrame(rows, columns=["date", "alpha_ann_pct", "beta", "t_alpha", "p_alpha"]).set_index("date")
    return df_roll

# -------------------------
# Rolling alpha plotting helper
# -------------------------
def rolling_alpha_section(pf, benchmark_returns: pd.Series, window: int = 252,
                          risk_free_rate: float = 0.02, annualization: int = 252,
                          show_plot: bool = False) -> Tuple[pd.DataFrame, Optional[plt.Figure]]:
    df_roll = rolling_capm_alpha_beta(_pf_returns_series(pf), benchmark_returns,
                                     window=window, risk_free_rate=risk_free_rate, annualization=annualization)
    fig = None
    if show_plot and not df_roll.empty:
        fig, ax = plt.subplots(figsize=(11, 3.5))
        ax.plot(df_roll.index, df_roll["alpha_ann_pct"], linewidth=1)
        ax.axhline(0, linestyle="--", linewidth=0.8)
        sig_mask = df_roll["p_alpha"] < 0.05
        if sig_mask.any():
            ax.scatter(df_roll.index[sig_mask], df_roll["alpha_ann_pct"][sig_mask], s=22, zorder=3)
        ax.set_ylabel("Alpha annuo (%)")
        ax.set_title("Rolling CAPM Alpha vs Benchmark")
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
    return df_roll, fig

# -------------------------
# create_portfolio_summary_refactored (keeps compatibility with original names)
# -------------------------

def create_portfolio_summary_refactored(
    pf,
    *,
    sel_tickers=None,
    alpha_analysis: bool = True,
    risk_free_rate: float = 0.02,
    benchmark_mode: str = "internal",            # internal|external|portfolio
    benchmark_name: str = "Benchmark",
    benchmark_data: Optional[pd.Series] = None,  # prices (Close) if external
    benchmark_portfolio=None,                    # vbt.Portfolio if portfolio
    annualization: int = 252,
    rolling_window: Optional[int] = 252,
    show: bool = False,
    return_formatted: bool = False
):
    from tabulate import tabulate
    import numpy as np
    import pandas as pd

    stats = pf.stats()
    start_date = stats.get("Start")
    end_date = stats.get("End")
    init_invested = stats.get("Start Value")
    final_value = stats.get("End Value")

    try:
        start_str = pd.to_datetime(start_date).date().isoformat()
    except Exception:
        start_str = str(start_date)
    try:
        end_str = pd.to_datetime(end_date).date().isoformat()
    except Exception:
        end_str = str(end_date)
    period_str = f"{start_str} → {end_str}"

    period_val = stats.get("Period")
    period_days = period_val.days if hasattr(period_val, "days") else int(period_val)
    period_days = max(int(period_days), 1)

    # --- returns portfolio (Series, daily) ---
    ret_pf = _pf_returns_series(pf)

    # --- base perf ---
    cagr = (final_value / init_invested) ** (annualization / period_days) - 1
    total_return = float(pf.total_return(group_by=True))
    volatility = float(pf.annualized_volatility(group_by=True, freq="1D"))
    max_dd = float(abs(pf.max_drawdown(group_by=True)))

    rf_daily = (1 + risk_free_rate) ** (1 / annualization) - 1
    mean_daily = float(ret_pf.mean())
    std_daily = float(ret_pf.std())
    sharpe = (mean_daily - rf_daily) / std_daily * np.sqrt(annualization) if std_daily != 0 else np.nan
    adj_car_sharpe = sharpe
    adj_car_calmar = (cagr / max_dd) if max_dd != 0 else np.nan

    # --- trade frequency ---
    if sel_tickers is None:
        total_trades = stats.get("Total Trades", np.nan)
    else:
        total_trades = 0
        dates = sel_tickers.index
        for i in range(1, len(dates)):
            # prev_set = set(sel_tickers.loc[dates[i - 1], "Top_Tickers"])
            # curr_set = set(sel_tickers.loc[dates[i], "Top_Tickers"])
            prev_set = set(sel_tickers.loc[dates[i - 1], "tickers"])
            curr_set = set(sel_tickers.loc[dates[i], "tickers"])
            total_trades += len(curr_set - prev_set) + len(prev_set - curr_set)

    total_months = period_days / 21
    trades_per_month = float(total_trades) / total_months if total_months > 0 else np.nan
    month_op = trades_per_month * (2 if sel_tickers is None else 1)

    # --- exposure ---
    try:
        exposure = float(pf.gross_exposure().mean())
    except Exception:
        # fallback events method
        try:
            records = pf.trades.records_readable
            idx = pf.returns(group_by=False).index
            if idx.tz is not None:
                idx = idx.tz_localize(None)
            idx = idx.normalize()
            events = pd.Series(0, index=idx, dtype="int64")
            for _, row in records.iterrows():
                entry_ts = row["Entry Timestamp"]
                exit_ts = row["Exit Timestamp"] if pd.notnull(row["Exit Timestamp"]) else end_date
                if hasattr(entry_ts, "tzinfo") and entry_ts.tzinfo is not None:
                    entry_ts = entry_ts.tz_localize(None)
                if hasattr(exit_ts, "tzinfo") and exit_ts.tzinfo is not None:
                    exit_ts = exit_ts.tz_localize(None)
                try:
                    entry_ts = pd.Timestamp(entry_ts).normalize()
                except Exception:
                    pass
                try:
                    exit_ts = pd.Timestamp(exit_ts).normalize()
                except Exception:
                    pass
                entry_pos = idx.searchsorted(entry_ts, side="left")
                if entry_pos < len(idx):
                    events.iloc[entry_pos] += 1
                exit_pos = idx.searchsorted(exit_ts, side="left")
                if exit_pos < len(idx):
                    events.iloc[exit_pos] -= 1
            exposure = float(events.cumsum().astype(int).gt(0).mean())
        except Exception:
            exposure = np.nan

    # -----------------------------
    # NEW: profit/recovery diagnostics
    # -----------------------------
    def _min_days_to_positive(ret: pd.Series):
        # legacy behavior: find_min_positive_period expects returns
        try:
            L = find_min_positive_period(ret)
            return "nessuno" if L is None else L
        except Exception:
            return "n/a"

    def _underwater_durations_from_returns(ret: pd.Series):
        """
        Calcola drawdown durations da equity cumulata.
        Ritorna:
          - uw_max: max durata underwater (giorni) considerando anche DD ancora aperto
          - rec_lengths: lista durate di recovery SOLO per DD chiusi (quando equity torna al max precedente)
          - uw_now: True se l'ultimo punto è underwater
          - uw_now_days: durata del DD corrente (se aperto), altrimenti 0
        """
        try:
            r = ret.dropna()
            if r.empty:
                return ("n/a", [], "n/a", "n/a")

            eq = (1.0 + r).cumprod()
            peak = eq.cummax()
            dd = eq / peak - 1.0
            is_uw = dd < 0

            idx = r.index
            # posizioni start/end underwater
            starts = []
            ends = []
            prev = False
            for i, flag in enumerate(is_uw.values):
                if flag and not prev:
                    starts.append(i)
                if (not flag) and prev:
                    ends.append(i)
                prev = flag

            # max underwater duration (include open DD)
            uw_max = 0
            for s_pos in starts:
                # se c'è un end successivo
                e_pos = None
                for cand in ends:
                    if cand > s_pos:
                        e_pos = cand
                        break
                if e_pos is None:
                    # open DD -> fino a fine serie
                    e_pos = len(idx) - 1
                uw_max = max(uw_max, int(e_pos - s_pos))

            # recovery lengths: SOLO drawdown chiusi (serve end)
            rec_lengths = []
            pairs = min(len(starts), len(ends))
            for k in range(pairs):
                s_pos = starts[k]
                e_pos = ends[k]
                if e_pos > s_pos:
                    rec_lengths.append(int(e_pos - s_pos))

            uw_now = bool(is_uw.iloc[-1])
            if uw_now:
                # durata DD corrente: last start -> end (last)
                last_start = None
                for s_pos in reversed(starts):
                    if s_pos <= len(idx) - 1:
                        last_start = s_pos
                        break
                uw_now_days = int((len(idx) - 1) - last_start) if last_start is not None else int("0")
            else:
                uw_now_days = 0

            return (uw_max, rec_lengths, uw_now, uw_now_days)
        except Exception:
            return ("n/a", [], "n/a", "n/a")

    L_min_profit = _min_days_to_positive(ret_pf)
    uw_max, rec_lengths, uw_now, uw_now_days = _underwater_durations_from_returns(ret_pf)

    if isinstance(rec_lengths, list) and len(rec_lengths) > 0:
        rec_min = int(np.min(rec_lengths))
        rec_med = int(np.median(rec_lengths))
        rec_p90 = int(np.percentile(rec_lengths, 90))
        rec_max = int(np.max(rec_lengths))
    else:
        rec_min = rec_med = rec_p90 = rec_max = "nessuno"

    # -----------------------------
    # Resolve benchmark returns
    # -----------------------------
    bm_ret, bm_meta = resolve_benchmark_returns(
        pf,
        benchmark_mode=benchmark_mode,
        benchmark_name=benchmark_name,
        benchmark_data=benchmark_data,
        benchmark_portfolio=benchmark_portfolio
    )

    capm = capm_alpha_beta(ret_pf, bm_ret, risk_free_rate=risk_free_rate, annualization=annualization) if alpha_analysis else None
    rolling_df = None
    if alpha_analysis and rolling_window:
        rolling_df = rolling_capm_alpha_beta(
            ret_pf, bm_ret, window=int(rolling_window),
            risk_free_rate=risk_free_rate, annualization=annualization
        )

    data_entries = [
        ("Periodo", period_str, period_str),
        ("Benchmark source", bm_meta.get("benchmark_source"), bm_meta.get("benchmark_source")),
        ("Benchmark name", bm_meta.get("benchmark_name"), bm_meta.get("benchmark_name")),
        ("Importo investito (€)", init_invested, f"{init_invested:.2f}"),
        ("Valore patrimoniale netto (€)", final_value, f"{final_value:.2f}"),
        ("Giorni di trading", period_days, period_days),
        ("Ritorno totale", total_return, f"{total_return:.2%}"),
        ("CAGR", cagr, f"{cagr:.2%}"),
        ("Max Drawdown", max_dd, f"{max_dd:.2%}"),
        ("Volatilità annua", volatility, f"{volatility:.2%}"),
        ("Rapporto di Sharpe", sharpe, round(sharpe, 2) if pd.notna(sharpe) else "n/a"),
        ("Adjusted CAR (Sharpe-style)", adj_car_sharpe, round(adj_car_sharpe, 2) if pd.notna(adj_car_sharpe) else "n/a"),
        ("Adjusted CAR (Calmar-style)", adj_car_calmar, round(adj_car_calmar, 2) if pd.notna(adj_car_calmar) else "n/a"),
        ("Operazioni al mese", month_op, round(month_op, 2) if pd.notna(month_op) else "n/a"),
        ("Market Exposure (avg gross)", exposure, f"{exposure:.2%}" if pd.notna(exposure) else "n/a"),

        # --- NEW: recovery / underwater diagnostics ---
        ("Durata minima in guadagno (giorni)", L_min_profit, L_min_profit),
        ("Max Underwater Duration (giorni)", uw_max, uw_max),
        ("Recovery Duration Min (giorni)", rec_min, rec_min),
        ("Recovery Duration Median (giorni)", rec_med, rec_med),
        ("Recovery Duration P90 (giorni)", rec_p90, rec_p90),
        ("Recovery Duration Max (giorni)", rec_max, rec_max),
        ("Underwater Now?", uw_now, uw_now),
        ("Underwater Days Now", uw_now_days, uw_now_days),
    ]

    if alpha_analysis and capm is not None:
        data_entries.extend([
            ("Alpha (giornaliero, assoluto)", capm["alpha_daily"],
             f"{capm['alpha_daily']:.6f}" if pd.notna(capm["alpha_daily"]) else "n/a"),
            ("Alpha annualizzato (%)", capm["alpha_ann_pct"],
             f"{capm['alpha_ann_pct']:.2f}%" if pd.notna(capm["alpha_ann_pct"]) else "n/a"),
            ("Beta (vs benchmark)", capm["beta"],
             round(capm["beta"], 4) if pd.notna(capm["beta"]) else "n/a"),
            ("Correlazione (vs benchmark)", capm["corr"],
             round(capm["corr"], 4) if pd.notna(capm["corr"]) else "n/a"),
            ("T-stat Alpha", capm["t_alpha"],
             round(capm["t_alpha"], 2) if pd.notna(capm["t_alpha"]) else "n/a"),
            ("P-value Alpha", capm["p_alpha"],
             float(f"{capm['p_alpha']:.2e}") if pd.notna(capm["p_alpha"]) else "n/a"),
            ("Tracking Error (ann)", capm["te_ann"],
             round(capm["te_ann"], 4) if pd.notna(capm["te_ann"]) else "n/a"),
            ("Obs (alpha reg)", capm["n_obs"], capm["n_obs"]),
        ])

    df_raw = pd.DataFrame({"Valore": [v for _, v, _ in data_entries]},
                          index=[k for k, _, _ in data_entries])
    df_fmt = pd.DataFrame({"Valore": [pv for _, _, pv in data_entries]},
                          index=[k for k, _, _ in data_entries])
    out = {
        "stats_df_raw": df_raw,
        "stats_df": (df_fmt if return_formatted else df_raw),
        "capm": capm,
        "rolling_capm": rolling_df,
        "benchmark_meta": bm_meta,
        "benchmark_returns": bm_ret
    }

    if show:
        display(df_fmt)
    return out


# -------------------------
# Unified core refactored: plotting grouped in blocks
# -------------------------
def _generate_portfolio_performance_core_refactored(
    # Portfolios
    pf: 'vbt.Portfolio',
    portfolio_title: str,
    pf_b_h: 'vbt.Portfolio' = None,
    
    # switch comportamento
    mode: str = "standard",  # standard|rotational|lazy
    
    # caratteristiche "standard"
    portfolio_ts: list[dict] | None = None,
    
    # caratteristiche "rotational"
    sel_tickers: list | None = None,

    # benchmark
    benchmark_mode: str = "internal",
    benchmark: str = "Benchmark",
    benchmark_data: Optional[pd.Series] = None,   # prices if external
    
    alpha_analysis: bool = True,
    risk_free_rate: float = 0.02,
    rolling_window: Optional[int] = 252,
    show_report: bool = True,
    show_plots: bool = False,
    vbt_plot_width: Optional[int] = 1000,
    universe: Optional[list] = None,
):
    """
    Refactoring a blocchi:
      - header PRIMA di tutto
      - STATISTICHE complete + SINTESI boxed
      - Rolling CAPM alpha+beta (Plotly) subito dopo i plot vbt
      - Ripristina rolling panels (cum/rolling + heatmap + prob loss + triangle)
      - Ripristina grafico rendimenti totali per titolo
      - Benchmark interno: evita yfinance nei contributions (crea benchmark_data sintetico da bm_ret)
    """
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    from tabulate import tabulate

    # -------------------------
    # Helpers: show
    # -------------------------
    def _maybe_show_fig(fig):
        if not show_plots:
            return
        try:
            fig.show()
        except Exception:
            try:
                import matplotlib.pyplot as plt
                plt.show()
            except Exception:
                pass

    # -------------------------
    # Helpers: dates
    # -------------------------
    def _safe_period_str(stats_obj) -> tuple[str, str, str]:
        try:
            s = pd.to_datetime(stats_obj.get("Start")).date().isoformat()
        except Exception:
            s = str(stats_obj.get("Start", ""))
        try:
            e = pd.to_datetime(stats_obj.get("End")).date().isoformat()
        except Exception:
            e = str(stats_obj.get("End", ""))
        return s, e, f"{s} → {e}"

    def _ytd_start():
        now = pd.Timestamp.now()
        return pd.Timestamp(now.year, 1, 1)

    # -------------------------
    # Helpers: benchmark “prices” from returns (to avoid yfinance)
    # -------------------------
    def _returns_to_price(ret: pd.Series, base: float = 100.0) -> pd.Series:
        r = ret.copy().dropna()
        if r.empty:
            return r
        try:
            if getattr(r.index, "tz", None) is not None:
                r.index = r.index.tz_localize(None)
        except Exception:
            pass
        r.index = pd.to_datetime(r.index).normalize()
        return (1.0 + r).cumprod() * float(base)

    def is_one_ticker(pf) -> bool:
        """
        Ritorna True se il Portfolio contiene un solo ticker/colonna.
        Robusto a pf.assets() che può essere Series o DataFrame.
        """
        assets = pf.assets()
    
        # Caso più comune: DataFrame (multi-ticker)
        if isinstance(assets, pd.DataFrame):
            return assets.shape[1] == 1
    
        # Caso: Series (spesso single-ticker o single-group)
        if isinstance(assets, pd.Series):
            # Se è una Series con name che rappresenta un ticker -> considerala 1 ticker.
            # Se è una Series "group" aggregata, comunque lato tickers è 1 (perché non hai breakdown).
            return True
    
        # Fallback: prova ad inferire da pf.close (di solito è coerente)
        close = getattr(pf, "close", None)
        if close is not None:
            if isinstance(close, pd.DataFrame):
                return close.shape[1] == 1
            if isinstance(close, pd.Series):
                return True
    
        # Ultimo fallback: non lo sappiamo => assume multi per prudenza
        return False

    # -------------------------
    # Block 0: header + summary build + print tables
    # -------------------------
    def _build_summary_and_print():
        stats_tmp = pf.stats()
        start_str, end_str, period_str = _safe_period_str(stats_tmp)

        one_ticker = is_one_ticker(pf)
        portfolio_type = "Titolo" if one_ticker else "Portfolio"

    
        header = f"📈 Statistiche {portfolio_type} {portfolio_title} ({period_str})"

        # header = f"📈 Statistiche Portfolio {portfolio_title} ({period_str})"
        
        if show_report:
            print(header)

        summary = create_portfolio_summary_refactored(
            pf,
            sel_tickers=sel_tickers,
            alpha_analysis=alpha_analysis,
            risk_free_rate=risk_free_rate,
            # benchmark_mode=benchmark_mode if mode != "standard" else ("portfolio" if pf_b_h is not None else benchmark_mode),
            benchmark_mode=benchmark_mode,
            benchmark_name=(f"{benchmark}" if benchmark else "Benchmark"),
            benchmark_data=benchmark_data,
            benchmark_portfolio=pf_b_h,
            annualization=252,
            rolling_window=rolling_window,
            show=False,
            return_formatted=True
        )

        stats_df = summary["stats_df"]
        stats_df_raw = summary.get("stats_df_raw", stats_df)
        bm_ret = summary.get("benchmark_returns", None)
        bm_meta = summary.get("benchmark_meta", {"benchmark_source": None, "benchmark_name": benchmark})
        capm = summary.get("capm")
        rolling_capm = summary.get("rolling_capm")

        # STATISTICHE complete
        if show_report:
            print("\n================= STATISTICHE ==================")
            try:
                display(stats_df.rename(columns={"Valore": ""}))
            except Exception:
                print(stats_df.rename(columns={"Valore": ""}))

        # SINTESI boxed (stesso formato)
        if show_report:
            print("\n================= SINTESI ==================")
            df_printing = stats_df.copy()

            headers_raw = [
                "Importo investito", "Valore finale netto", "CAGR (252)",
                "Max Drawdown", "Deviazione standard", "Rapporto di Sharpe",
                "Operazioni al mese", "Durata minima in guadagno"
            ]

            def _find(key, default="n/a"):
                return df_printing.loc[key, "Valore"] if key in df_printing.index else default

            vals_ordered = [
                _find("Importo investito (€)"),
                _find("Valore patrimoniale netto (€)"),
                _find("CAGR"),
                _find("Max Drawdown"),
                _find("Volatilità annua"),
                _find("Rapporto di Sharpe"),
                _find("Operazioni al mese"),
                _find("Durata minima in guadagno (giorni)")
            ]
            tab = tabulate([vals_ordered], headers=headers_raw, tablefmt="fancy_grid",
                           colalign=("center",) * len(headers_raw))
            print(tab)

        # sintesi_df (come prima)
        def _pick(key):
            return stats_df.loc[key, "Valore"] if key in stats_df.index else "n/a"

        sintesi_df = pd.DataFrame([{
            "Importo investito": _pick("Importo investito (€)"),
            "Valore finale netto": _pick("Valore patrimoniale netto (€)"),
            "CAGR (252)": _pick("CAGR"),
            "Max Drawdown": _pick("Max Drawdown"),
            "Deviazione standard": _pick("Volatilità annua"),
            "Rapporto di Sharpe": _pick("Rapporto di Sharpe"),
            "Operazioni al mese": _pick("Operazioni al mese"),
            "Durata minima in guadagno": _pick("Durata minima in guadagno (giorni)"),
        }])

        return {
            "header": header,
            "stats_tmp": stats_tmp,
            "stats_df": stats_df,
            "stats_df_raw": stats_df_raw,
            "sintesi_df": sintesi_df,
            "bm_ret": bm_ret,
            "bm_meta": bm_meta,
            "capm": capm,
            "rolling_capm": rolling_capm,
        }

    # -------------------------
    # Block 1: vbt base plots
    # -------------------------
    def _plot_vbt_base(figs: list):
        try:
            subplots = ['orders', 'trade_pnl'] if mode == "standard" else []    
            subplots.extend(['cum_returns', 'drawdowns', 'underwater', 'gross_exposure'])
            fig_value = pf.plot(
                width=vbt_plot_width,
                subplots=subplots
            )
            figs.append(fig_value)
            _maybe_show_fig(fig_value)
        except Exception:
            pass
            
    # -------------------------
    # Block 1.1: Time-series portfolio (solo standard) — RIPRISTINATO
    # -------------------------
    def _plot_ts_portfolio_standard(figs: list):
        if mode != "standard":
            return
        if not portfolio_ts:
            return

        try:
            fig_ts = plot_ts_portfolio(
                final_portfolio=pf,
                portfolio_ts=portfolio_ts,
                portfolio_title=portfolio_title,
                # width=vbt_plot_width,
                width=None,
                start_date=ytd(),          # come originale
                # show_report=False
            )
            figs.append(fig_ts)
            _maybe_show_fig(fig_ts)
        except Exception as e:
            if show_report:
                print("Warning: plot_ts_portfolio failed:", str(e))
    # -------------------------
    # Block 2: rolling CAPM alpha+beta (Plotly) right after vbt
    # -------------------------
    def _plot_rolling_capm_alpha_beta(figs: list, bm_ret: pd.Series | None):
    
        rolling_alpha_df = None  # <-- FIX: evita UnboundLocalError in ogni path
    
        def _diag_bm(bm: pd.Series | None) -> str:
            if bm is None:
                return "bm_ret=None"
            try:
                n = int(bm.shape[0])
                nn = int(bm.dropna().shape[0])
                idx = bm.index
                i0 = pd.to_datetime(idx.min()) if n else None
                i1 = pd.to_datetime(idx.max()) if n else None
                return f"bm_ret: n={n}, nonnull={nn}, range={i0}→{i1}"
            except Exception:
                return "bm_ret: (diagnostica non disponibile)"
    
        def _diag_df(df: pd.DataFrame | None, window: int) -> str:
            if df is None:
                return "rolling_alpha_df=None"
            if getattr(df, "empty", True):
                return "rolling_alpha_df empty=True"
    
            cols = list(df.columns)
            msg = [f"rolling_alpha_df: n={len(df)}, cols={cols}"]
            try:
                msg.append(f"range={pd.to_datetime(df.index.min())}→{pd.to_datetime(df.index.max())}")
            except Exception:
                pass
    
            expected = ["alpha_ann_pct", "p_alpha", "beta"]
            missing = [c for c in expected if c not in df.columns]
            if missing:
                msg.append(f"missing_cols={missing}")
    
            for c in ["alpha_ann_pct", "p_alpha", "beta"]:
                if c in df.columns:
                    s = df[c]
                    msg.append(f"{c}: nonnull={int(s.notna().sum())}/{len(s)}")
    
            if "alpha_ann_pct" in df.columns and "beta" in df.columns:
                valid = df[["alpha_ann_pct", "beta"]].dropna()
                msg.append(f"valid_rows(alpha+beta)= {len(valid)}/{len(df)}")
    
            msg.append(f"rolling_window={int(window)}")
            return " | ".join(msg)
    
        # -------------------------
        # Guard-rail benchmark
        # -------------------------
        if bm_ret is None:
            if show_report:
                print(f"ℹ️  Rolling CAPM skipped: benchmark returns missing ({_diag_bm(bm_ret)})")
            return None
    
        try:
            bm_len = int(len(bm_ret))
        except Exception:
            bm_len = 0
    
        if bm_len == 0:
            if show_report:
                print(f"ℹ️  Rolling CAPM skipped: benchmark returns empty ({_diag_bm(bm_ret)})")
            return None
    
        # Normalizza index benchmark
        try:
            bm_ret = bm_ret.copy()
            if bm_ret.index.tz is not None:
                bm_ret.index = bm_ret.index.tz_localize(None)
            bm_ret.index = pd.to_datetime(bm_ret.index).normalize()
        except Exception:
            pass
    
        w = int(rolling_window or 252)
    
        try:
            bm_nonnull = int(bm_ret.dropna().shape[0])
        except Exception:
            bm_nonnull = 0
    
        if bm_nonnull < max(30, w):
            if show_report:
                print(
                    "ℹ️  Rolling CAPM skipped: benchmark too short for rolling window. "
                    f"need≥{max(30, w)} non-null obs, got {bm_nonnull}. ({_diag_bm(bm_ret)})"
                )
            return None
    
        # -------------------------
        # Compute rolling alpha/beta
        # -------------------------
        try:
            rolling_alpha_df, _ = rolling_alpha_section(
                pf,
                bm_ret,
                window=w,
                risk_free_rate=risk_free_rate,
                annualization=252,
                show_plot=False
            )
        except Exception as e:
            if show_report:
                print("⚠️  Rolling CAPM failed inside rolling_alpha_section.")
                print("    Error:", str(e))
                print("    " + _diag_bm(bm_ret))
            return None
    
        # -------------------------
        # Validate output before plotting
        # -------------------------
        if rolling_alpha_df is None or rolling_alpha_df.empty:
            if show_report:
                print("ℹ️  Rolling CAPM skipped: rolling_alpha_df is None/empty.")
                print("    " + _diag_bm(bm_ret))
                print("    " + _diag_df(rolling_alpha_df, w))
            return rolling_alpha_df
    
        needed_cols = ["alpha_ann_pct", "p_alpha", "beta"]
        missing = [c for c in needed_cols if c not in rolling_alpha_df.columns]
        if missing:
            if show_report:
                print("ℹ️  Rolling CAPM skipped: rolling_alpha_df missing required columns:", missing)
                print("    " + _diag_df(rolling_alpha_df, w))
            return rolling_alpha_df
    
        valid_df = rolling_alpha_df[["alpha_ann_pct", "beta", "p_alpha"]].dropna(subset=["alpha_ann_pct", "beta"])
        if valid_df.empty:
            if show_report:
                print("ℹ️  Rolling CAPM skipped: no valid rows after dropna(alpha_ann_pct,beta).")
                print("    " + _diag_df(rolling_alpha_df, w))
                try:
                    display(rolling_alpha_df.tail(5))
                except Exception:
                    pass
            return rolling_alpha_df
    
        # -------------------------
        # Plot (plotly)
        # -------------------------
        try:
            fig_roll = go.Figure()
    
            fig_roll.add_trace(go.Scatter(
                x=rolling_alpha_df.index,
                y=rolling_alpha_df["alpha_ann_pct"],
                mode="lines",
                name="Alpha annuo (%)",
                line=dict(width=2),
                hovertemplate="Date: %{x}<br>Alpha annuo: %{y:.3f}%<br>p: %{customdata[0]:.3g}",
                customdata=rolling_alpha_df[["p_alpha"]].values
            ))
    
            sig_mask = (rolling_alpha_df["p_alpha"] < 0.05) & rolling_alpha_df["p_alpha"].notna()
            if sig_mask.any():
                fig_roll.add_trace(go.Scatter(
                    x=rolling_alpha_df.index[sig_mask],
                    y=rolling_alpha_df.loc[sig_mask, "alpha_ann_pct"],
                    mode="markers",
                    name="Alpha signif. (p<0.05)",
                    marker=dict(size=8, symbol="circle-open"),
                    hovertemplate="Date: %{x}<br>Alpha annuo: %{y:.3f}%<br>p: %{customdata[0]:.3g}",
                    customdata=rolling_alpha_df.loc[sig_mask, ["p_alpha"]].values
                ))
    
            fig_roll.add_trace(go.Scatter(
                x=rolling_alpha_df.index,
                y=rolling_alpha_df["beta"],
                mode="lines",
                name="Beta (rolling)",
                line=dict(width=1, dash="dot"),
                yaxis="y2",
                hovertemplate="Date: %{x}<br>Beta: %{y:.3f}"
            ))
    
            fig_roll.update_layout(
                title="Rolling CAPM — Alpha annuo (%) e Beta (rolling)",
                xaxis=dict(title="Date"),
                yaxis=dict(title="Alpha annuo (%)", zeroline=True),
                yaxis2=dict(title="Beta (rolling)", overlaying="y", side="right", showgrid=False),
                legend=dict(
                    orientation="v",
                    yanchor="top",
                    y=0.99,
                    xanchor="left",
                    x=0.01,
                    bgcolor="rgba(255,255,255,0.85)",
                    bordercolor="rgba(0,0,0,0.2)",
                    borderwidth=1,
                    font=dict(size=12),
                    itemsizing="constant"
                ),
                height=420,
                margin=dict(l=50, r=70, t=60, b=40)
            )
    
            fig_roll.add_hline(y=0, line=dict(dash="dash", width=1),
                               annotation_text="0", annotation_position="top left")
    
            figs.append(fig_roll)
            _maybe_show_fig(fig_roll)
    
        except Exception as e:
            if show_report:
                print("⚠️  Rolling CAPM plotly build failed.")
                print("    Error:", str(e))
                print("    " + _diag_df(rolling_alpha_df, w))
    
        return rolling_alpha_df  
    

    # -------------------------
    # Block 3: alpha diagnostics text (after rolling plot)
    # -------------------------
    def _print_alpha_diagnostics(capm, rolling_alpha_df):
        if show_report and alpha_analysis:
            print("\n================= ANALISI ALPHA ==================")
            print(comment_alpha_diagnostics(capm, rolling_alpha_df))

    # -------------------------
    # Block 4: build portfolios_returns + vs benchmark plots (incl YTD)
    # -------------------------
    def _plot_vs_benchmark(figs: list, stats_tmp, bm_ret: pd.Series | None, bm_meta: dict):
        # returns strategy
        try:
            returns_strat = pf.returns()
        except Exception:
            returns_strat = _pf_returns_series(pf)
    
        portfolios_returns = {f"{portfolio_title} (Strategy)": returns_strat}
    
        # optional B&H portfolio
        if pf_b_h is not None:
            try:
                portfolios_returns[f"{portfolio_title} (B&H)"] = pf_b_h.returns()
            except Exception:
                pass
    
        # add benchmark RETURNS directly (no yfinance)
        if bm_ret is not None and len(bm_ret) > 0:
            bench_name = (bm_meta.get("benchmark_name") or "Benchmark")
            portfolios_returns[str(bench_name)] = bm_ret
    
        # --- normalize full-period window (for de-dup YTD) ---
        p_start = stats_tmp.get("Start")
        p_end   = stats_tmp.get("End")
        
        try:
            p_start_n = pd.to_datetime(p_start).tz_localize(None) if getattr(pd.to_datetime(p_start), "tzinfo", None) else pd.to_datetime(p_start)
            p_end_n   = pd.to_datetime(p_end).tz_localize(None)   if getattr(pd.to_datetime(p_end), "tzinfo", None)   else pd.to_datetime(p_end)
            p_start_n = pd.Timestamp(p_start_n).normalize()
            p_end_n   = pd.Timestamp(p_end_n).normalize()
        except Exception:
            p_start_n = None
            p_end_n = None
    
        # full period plot
        try:
            start_year = getattr(p_start, "year", None)
            curr_year = pd.Timestamp.now().year
            base = 0.0 if (start_year == curr_year) else 1.0
    
            fig_vs = plot_multiple_portfolios(
                portfolios_returns,
                benchmark=None,
                title="Performance vs Benchmark",
                benchmark_data=None,
                # start_date=p_start,
                # end_date=p_end,
                base=base
            )
            figs.append(fig_vs)
            _maybe_show_fig(fig_vs)
        except Exception:
            pass
    
        # YTD plot (only if effective window differs from full-period effective window)
        try:
            strat_series = portfolios_returns.get(f"{portfolio_title} (Strategy)", None)
            if strat_series is not None:
                idx = pd.to_datetime(strat_series.index)
                try:
                    if getattr(idx, "tz", None) is not None:
                        idx = idx.tz_localize(None)
                except Exception:
                    pass
                idx = pd.DatetimeIndex(idx).normalize()
    
                # requested windows
                ytd_start_req = _ytd_start().normalize()
                ytd_end_req   = idx.max()
    
                # full requested window (fallback to idx bounds)
                full_start_req = p_start_n if p_start_n is not None else idx.min()
                full_end_req   = p_end_n   if p_end_n   is not None else idx.max()
    
                # ---- EFFECTIVE windows after clamp to available data ----
                eff_full_start = max(full_start_req, idx.min())
                eff_full_end   = min(full_end_req,   idx.max())
    
                eff_ytd_start  = max(ytd_start_req,  idx.min())
                eff_ytd_end    = min(ytd_end_req,    idx.max())
    
                same_as_full = (eff_full_start == eff_ytd_start) and (eff_full_end == eff_ytd_end)
    
                if (eff_ytd_end >= eff_ytd_start) and (not same_as_full):
                    fig_vs_ytd = plot_multiple_portfolios(
                        portfolios_returns,
                        title="Performance vs Benchmark (YTD)",
                        benchmark=None,
                        benchmark_data=None,
                        start_date=eff_ytd_start,
                        end_date=eff_ytd_end,
                        base=base
                    )
                    figs.append(fig_vs_ytd)
                    _maybe_show_fig(fig_vs_ytd)
    
        except Exception as e:
            if show_report:
                print("Warning: YTD plot failed:", str(e))    
                
        return portfolios_returns

    

    # -------------------------
    # Block 5: periodic plots (annual/monthly)
    # -------------------------
    def _plot_periodic(figs: list, portfolios_returns: dict):
        try:
            fig_annual = plot_annual_performance(portfolios_returns, benchmark=None, benchmark_data=None)
            figs.append(fig_annual); _maybe_show_fig(fig_annual)
        except Exception:
            pass

        try:
            fig_annual_hist = plot_year_returns_histogram(
                pf,
                title=f"Istogramma dei rendimenti annuali – {portfolio_title}",
                panel_width=0.34, gap=0.02, hist_fill=0.95, min_years=2
            )
            if fig_annual_hist is not None:
                figs.append(fig_annual_hist); _maybe_show_fig(fig_annual_hist)
        except Exception:
            pass

        try:
            fig_monthly = plot_monthly_returns(
                pf,
                eoy=True,
                title=f"Portfolio {portfolio_title} - Rendimenti mensili (%)",
                width=vbt_plot_width if mode == "standard" else None
            )
            figs.append(fig_monthly); _maybe_show_fig(fig_monthly)
        except Exception:
            pass

        try:
            fig_monthly_hist = plot_monthly_returns_histogram(
                pf,
                title=f"Istogramma dei rendimenti mensili – {portfolio_title}"
            )
            figs.append(fig_monthly_hist); _maybe_show_fig(fig_monthly_hist)
        except Exception:
            pass

    # -------------------------
    # Block 6: rolling panels (restore original “rolling” section)
    # -------------------------
    def _plot_rolling_panels(figs: list, stats_tmp):
        # solo rotational/lazy e solo se non siamo all’anno corrente (come tua logica originale)
        if mode not in ("rotational", "lazy"):
            return

        try:
            p_start = stats_tmp.get("Start")
            start_year = getattr(p_start, "year", None)
            curr_year = pd.Timestamp.now().year
            if start_year == curr_year:
                return
        except Exception:
            # se non riesce, proviamo comunque
            pass

        try:
            out_roll = plot_cumulative_and_rolling_returns(
                pf,
                horizons_years=[1, 2, 3, 4, 5],
                add_fan_chart=False,
                add_heatmap=True,
                return_extras=True,
                add_horizon_analysis=True,
            )
            if isinstance(out_roll, dict) and out_roll.get("fig") is not None:
                figs.append(out_roll["fig"]); _maybe_show_fig(out_roll["fig"])
            if isinstance(out_roll, dict) and out_roll.get("fig_hm") is not None:
                figs.append(out_roll["fig_hm"]); _maybe_show_fig(out_roll["fig_hm"])
            if isinstance(out_roll, dict) and out_roll.get("fig_loss") is not None:
                figs.append(out_roll["fig_loss"]); _maybe_show_fig(out_roll["fig_loss"])
        except Exception:
            pass

        try:
            fig_triangle, _ = plot_annual_return_triangle(pf, resample_freq="YE")
            figs.append(fig_triangle); _maybe_show_fig(fig_triangle)
        except Exception:
            pass

    # -------------------------
    # Block 7: selection frequencies (rotational)
    # -------------------------
    def _plot_selection_freq(figs: list, stats_tmp):
        freq_df = None
        if mode == "rotational" and sel_tickers is not None:
            try:
                fig_sel_tickers, freq_df, freq_full_df, unselected_info = plot_ticker_frequencies(
                    sel_tickers,
                    start_date=stats_tmp.get("Start"),
                    end_date=stats_tmp.get("End"),
                    include_prev=True,
                    universe=universe or []
                )
                figs.append(fig_sel_tickers); _maybe_show_fig(fig_sel_tickers)

                unselected_df, n_unselected, pct_unselected = unselected_info
                if show_report:
                    print(f"{n_unselected} tickers su {len(universe or [])} mai selezionati ({pct_unselected:.1%})")
            except Exception:
                pass
        return freq_df

    # -------------------------
    # Block 8: total return per ticker — RIPRISTINATO (standard + rotational)
    # -------------------------
    def _plot_total_return_per_ticker(figs: list):
        try:
            if mode == "rotational":
                performance = pf.total_return(group_by=False) * 100
                fig_tickers_perf = plot_total_return_per_ticker(performance[performance != 0])
                figs.append(fig_tickers_perf)
                _maybe_show_fig(fig_tickers_perf)

            elif mode == "standard":
                if portfolio_ts is None or len(portfolio_ts) == 0:
                    return

                # portfolio_ts atteso: list[dict] con chiavi "symbol" e "returns"
                # Esempio: [{"symbol":"XYZ","returns":0.12}, ...]
                returns = pd.Series({d["symbol"]: d["returns"] for d in portfolio_ts if "symbol" in d and "returns" in d})

                if returns.empty:
                    return

                fig_tickers_perf = plot_total_return_per_ticker(
                    returns,
                    start_date=_ytd_start(),
                    end_date=None
                )
                figs.append(fig_tickers_perf)
                _maybe_show_fig(fig_tickers_perf)

        except Exception as e:
            if show_report:
                print("Warning: plot_total_return_per_ticker failed:", str(e))    
                
    # -------------------------
    # Block 9: contributions (fix internal benchmark without changing signature)
    # -------------------------
    def _plot_contributions(figs: list, stats_tmp, bm_ret: pd.Series | None, bm_meta: dict):
        # Se benchmark_data non è fornito e benchmark_mode è internal, costruisci prezzi sintetici da bm_ret
        _bench_data = benchmark_data
        _bench_name = benchmark

        if (_bench_data is None) and (bm_ret is not None) and (benchmark_mode == "internal"):
            # price-like series -> evita yfinance
            _bench_data = _returns_to_price(bm_ret, base=100.0)
            _bench_name = str(bm_meta.get("benchmark_name") or benchmark or "Internal BM")

        try:
            fig_assets = build_and_plot_portfolio_contributions(
                pf,
                title=portfolio_title,
                benchmark=_bench_name,
                benchmark_data=_bench_data,
                start_date=stats_tmp.get("Start"),
                end_date=stats_tmp.get("End"),
                show_report=show_report
            )
            if fig_assets is not None:
                figs.append(fig_assets); _maybe_show_fig(fig_assets)
        except Exception:
            pass

        # YTD contributions (come prima)
        try:
            p_start = stats_tmp.get("Start")
            start_year = getattr(p_start, "year", None)
            curr_year = pd.Timestamp.now().year
            if start_year != curr_year:
                fig_assets_ytd = build_and_plot_portfolio_contributions(
                    pf,
                    title=portfolio_title,
                    benchmark=_bench_name,
                    benchmark_data=_bench_data,
                    start_date=_ytd_start(),
                    end_date=stats_tmp.get("End"),
                    show_report=show_report
                )
                if fig_assets_ytd is not None:
                    figs.append(fig_assets_ytd); _maybe_show_fig(fig_assets_ytd)
        except Exception:
            pass

    # -------------------------
    # Block 10: weights
    # -------------------------
    def _plot_weights(figs: list,benchmark: str):
        
        if str(mode).lower() != "standard" or benchmark == None:
            try:
                fig_weights = visualize_portfolio_weights(pf=pf, title=portfolio_title)
                if fig_weights is not None:
                    figs.append(fig_weights); _maybe_show_fig(fig_weights)
            except Exception:
                pass

    # -------------------------
    # RUN: pipeline
    # -------------------------
    summary_pack = _build_summary_and_print()

    header = summary_pack["header"]
    stats_tmp = summary_pack["stats_tmp"]
    stats_df = summary_pack["stats_df"]
    stats_df_raw = summary_pack["stats_df_raw"]
    sintesi_df = summary_pack["sintesi_df"]
    bm_ret = summary_pack["bm_ret"]
    bm_meta = summary_pack["bm_meta"]
    capm = summary_pack["capm"]
    rolling_capm = summary_pack["rolling_capm"]

    figs: list = []

    _plot_vbt_base(figs)
    _plot_ts_portfolio_standard(figs)

    rolling_alpha_df = _plot_rolling_capm_alpha_beta(figs, bm_ret)    
    _print_alpha_diagnostics(capm, rolling_alpha_df)

    portfolios_returns = _plot_vs_benchmark(figs, stats_tmp, bm_ret, bm_meta)
    _plot_periodic(figs, portfolios_returns)

    _plot_rolling_panels(figs, stats_tmp)
    freq_df = _plot_selection_freq(figs, stats_tmp)

    _plot_total_return_per_ticker(figs)
    _plot_contributions(figs, stats_tmp, bm_ret, bm_meta)
    _plot_weights(figs,benchmark)

    out = {
        "figs": figs,
        "header": header,
        "stats_df": stats_df,
        "stats_df_raw": stats_df_raw,
        "sintesi_df": sintesi_df,
        "capm": capm,
        "rolling_capm": rolling_capm,
        "rolling_alpha": rolling_alpha_df,
        "benchmark_meta": bm_meta,
        "benchmark_returns": bm_ret
    }

    if mode == "rotational" and freq_df is not None:
        try:
            out.update({
                "performance_info": "Informazioni sulla selezione dei titoli:",
                "performance_tables": {
                    "Sequenza di selezioni": sel_tickers,
                    "Frequenze di selezione": freq_df.set_index("Ticker")
                }
            })
        except Exception:
            pass

    return out
    
# -------------------------
# Backwards-compatible wrappers matching your original names
# -------------------------
def generate_portfolio_performance(
    pf: 'vbt.Portfolio',
    portfolio_title: str,
    pf_b_h : Optional[vbt.Portfolio] = None,
    portfolio_ts: list[dict] | None = None,
    alpha_analysis: bool = True,
    benchmark: str | None = None,  # <-- consenti None
    benchmark_data: Optional[pd.Series] = None,
    plot_start_date: Optional[pd.Timestamp] = None,
    plot_end_date: Optional[pd.Timestamp] = None,
    show_report: bool = True,
    show_plots: bool = False
) -> dict:

    # # Regola: pf_b_h è benchmark SOLO se benchmark=None
    # if benchmark is None:
    #     bm_mode = "portfolio"     # benchmark = pf_b_h
    #     # bm_name = "B&H"           # etichetta (opzionale, puoi metterci portfolio_title)
    #     bm_name = f"{portfolio_title} (B&H as Benchmark)"
    #     bm_data = None
    # else:
    #     # benchmark significativo -> pf_b_h NON è benchmark
    #     bm_mode = "external" if benchmark_data is not None else "internal"
    #     bm_name = benchmark
    #     bm_data = benchmark_data

    # return _generate_portfolio_performance_core_refactored(
    #     pf=pf,
    #     portfolio_title=portfolio_title,
    #     mode="standard",
    #     portfolio_ts=portfolio_ts,
    #     # pf_b_h=pf_b_h,                 # <-- rimane per plottarlo come linea extra
    #     pf_b_h=None,
    #     benchmark_mode=bm_mode,
    #     benchmark=bm_name,
    #     benchmark_data=bm_data,
    #     alpha_analysis=alpha_analysis,
    #     show_report=show_report,
    #     show_plots=show_plots
    # )   
    
    return _generate_portfolio_performance_core_refactored(
        pf=pf,
        portfolio_title=portfolio_title,
        mode="standard",
        portfolio_ts=portfolio_ts,
        pf_b_h=pf_b_h,
        benchmark_mode=("external" if benchmark_data is not None else "internal"),
        benchmark=benchmark,
        benchmark_data=benchmark_data,
        alpha_analysis=alpha_analysis,
        show_report=show_report,
        show_plots=show_plots
    )   

def generate_rotational_portfolio_performance(
    pf: 'vbt.Portfolio',
    portfolio_title: str,
    sel_tickers: list | None = None,
    benchmark: str = 'SPY',
    benchmark_data: Optional[pd.Series] = None,
    plot_start_date: Optional[pd.Timestamp] = None,
    plot_end_date: Optional[pd.Timestamp] = None,
    method: Optional[str] = None,
    freq: Optional[str] = None,
    alpha_analysis: bool = True,
    show_report: bool = True,
    show_plots: bool = False,
    universe=[]
) -> dict:
    return _generate_portfolio_performance_core_refactored(
        pf=pf,
        portfolio_title=portfolio_title,
        mode="rotational",
        sel_tickers=sel_tickers,
        pf_b_h=None,
        benchmark_mode=("external" if benchmark_data is not None else "internal"),
        benchmark=benchmark,
        benchmark_data=benchmark_data,
        alpha_analysis=alpha_analysis,
        show_report=show_report,
        show_plots=show_plots,
        universe=universe
    )

def generate_lazy_portfolio_performance(
    pf: 'vbt.Portfolio',
    portfolio_title: str,
    benchmark: str = 'SPY',
    benchmark_data: Optional[pd.Series] = None,
    method: Optional[str] = None,
    freq: Optional[str] = None,
    alpha_analysis: bool = True,
    show_report: bool = True,
    show_plots: bool = False
) -> dict:
    return _generate_portfolio_performance_core_refactored(
        pf=pf,
        portfolio_title=portfolio_title,
        mode="lazy",
        sel_tickers=None,
        pf_b_h=None,
        benchmark_mode=("external" if benchmark_data is not None else "internal"),
        benchmark=benchmark,
        benchmark_data=benchmark_data,
        alpha_analysis=alpha_analysis,
        show_report=show_report,
        show_plots=show_plots
    )


        
def comment_alpha_diagnostics(capm: dict | None, roll: pd.DataFrame | None) -> str:
    import numpy as np
    import pandas as pd

    # --- helper formatting ---
    def _fmt(x, fmt):
        try:
            if x is None or (isinstance(x, float) and np.isnan(x)):
                return "n/a"
            return format(float(x), fmt)
        except Exception:
            return "n/a"

    def _is_num(x):
        try:
            return x is not None and not (isinstance(x, float) and np.isnan(x))
        except Exception:
            return False

    if capm is None:
        return f"{BOLD}Alpha CAPM non disponibile{RESET} (regressione non eseguita o dati insufficienti)."

    # --- inputs (robusti a naming diversi) ---
    a = capm.get("alpha_ann_pct", capm.get("alpha_ann", np.nan))          # %
    t = capm.get("t_alpha", capm.get("alpha_tstat", np.nan))
    p = capm.get("p_alpha", capm.get("alpha_pvalue", np.nan))
    b = capm.get("beta", capm.get("beta_mkt", np.nan))
    te = capm.get("tracking_error_ann", capm.get("te", np.nan))
    nobs = capm.get("nobs", capm.get("obs", capm.get("n", np.nan)))

    # -----------------------------
    # Diagnosi statica (CAPM)
    # -----------------------------
    lines = []

    if pd.isna(a) or pd.isna(p) or pd.isna(t):
        lines.append(f"{BOLD}Alpha/Beta statici non stimabili{RESET} (NaN).")
    else:
        sig_05 = (p < 0.05)
        sig_10 = (p < 0.10)

        # Alpha scenario
        if a > 0 and sig_05:
            base = f"Alpha {BOLD}positivo e significativo{RESET}: evidenza di extra-rendimento vs benchmark."
        elif a > 0 and sig_10:
            base = f"Alpha {BOLD}positivo e debolmente significativo{RESET}: segnale promettente ma non pienamente robusto."
        elif a > 0 and not sig_10:
            base = f"Alpha {BOLD}positivo ma non significativo{RESET}: extra-rendimento non statisticamente confermato."
        elif a < 0 and sig_05:
            base = f"Alpha {BOLD}negativo e significativo{RESET}: sotto-performance vs benchmark con evidenza."
        elif a < 0 and sig_10:
            base = f"Alpha {BOLD}negativo e debolmente significativo{RESET}: sotto-performance probabile."
        elif a < 0 and not sig_10:
            base = f"Alpha {BOLD}negativo ma non significativo{RESET}: sotto-performance non confermata."
        else:
            base = f"Alpha {BOLD}~0 o poco informativo{RESET}."

        # Beta context
        beta_msg = ""
        if pd.notna(b):
            if b > 1.1:
                beta_msg = f" Beta {BOLD}> 1{RESET} (β={b:.2f}): portafoglio più direzionale del benchmark."
            elif b < 0.9:
                beta_msg = f" Beta {BOLD}< 1{RESET} (β={b:.2f}): esposizione al mercato più contenuta."
            else:
                beta_msg = f" Beta {BOLD}~ 1{RESET} (β={b:.2f}): esposizione simile al benchmark."

        # Stats details
        stats_msg = (
            f" Dettagli: alpha_ann={BOLD}{a:.2f}%{RESET}, "
            f"T-stat={BOLD}{t:.2f}{RESET}, "
            f"P-value={BOLD}{_fmt(p, '.2e')}{RESET}, "
            f"TE_ann={BOLD}{_fmt(te, '.3f')}{RESET}, "
            f"Obs={BOLD}{_fmt(nobs, '.0f')}{RESET}."
        )

        lines.append(base + beta_msg + stats_msg)

    # -----------------------------
    # Rolling (trend + “robustezza”)
    # -----------------------------
    rolling_block = ""
    roll_score = 0.0  # contribuirà alla valutazione complessiva

    if roll is None or roll.empty:
        rolling_block = f"Rolling alpha {BOLD}non disponibile{RESET} (finestra troppo corta o benchmark mancante)."
    else:
        r = roll.dropna()
        if r.empty:
            rolling_block = f"Rolling alpha presente ma {BOLD}tutto NaN{RESET} (controllare allineamenti/dati)."
        else:
            last_a = float(r["alpha_ann_pct"].iloc[-1]) if "alpha_ann_pct" in r.columns else np.nan
            med_a  = float(r["alpha_ann_pct"].median()) if "alpha_ann_pct" in r.columns else np.nan
            sig_pct = float((r["p_alpha"] < 0.05).mean() * 100.0) if "p_alpha" in r.columns else np.nan

            # Trend
            if _is_num(last_a) and _is_num(med_a):
                if last_a > med_a + 0.5:
                    trend = f"Rolling alpha {BOLD}in miglioramento{RESET} (ultimo vs mediana)."
                    roll_score += 1.0
                elif last_a < med_a - 0.5:
                    trend = f"Rolling alpha {BOLD}in peggioramento{RESET} (ultimo vs mediana)."
                    roll_score -= 1.0
                else:
                    trend = f"Rolling alpha {BOLD}stabile{RESET} rispetto al regime recente."
            else:
                trend = f"Rolling alpha: {BOLD}trend non stimabile{RESET}."

            # Significatività rolling
            sig_msg = ""
            if _is_num(sig_pct):
                if sig_pct >= 60:
                    sig_msg = f" Significatività rolling {BOLD}alta{RESET} (p<0.05 nel {sig_pct:.1f}% dei giorni)."
                    roll_score += 1.0
                elif sig_pct >= 30:
                    sig_msg = f" Significatività rolling {BOLD}moderata{RESET} (p<0.05 nel {sig_pct:.1f}% dei giorni)."
                    roll_score += 0.5
                else:
                    sig_msg = f" Significatività rolling {BOLD}bassa{RESET} (p<0.05 nel {sig_pct:.1f}% dei giorni)."
                    roll_score -= 0.5

            # Beta rolling hint
            beta_roll_msg = ""
            if "beta" in r.columns:
                last_b = float(r["beta"].iloc[-1])
                med_b  = float(r["beta"].median())
                if last_b > med_b + 0.1:
                    beta_roll_msg = f" Beta rolling {BOLD}in aumento{RESET} (ultimo {last_b:.2f} vs mediana {med_b:.2f})."
                elif last_b < med_b - 0.1:
                    beta_roll_msg = f" Beta rolling {BOLD}in calo{RESET} (ultimo {last_b:.2f} vs mediana {med_b:.2f})."
                else:
                    beta_roll_msg = f" Beta rolling {BOLD}stabile{RESET} (ultimo {last_b:.2f})."

            # Numbers
            nums = ""
            if _is_num(last_a) and _is_num(med_a):
                nums = f" Ultimo alpha={BOLD}{last_a:.2f}%{RESET}, mediana={med_a:.2f}%."

            rolling_block = trend + nums + sig_msg + beta_roll_msg

    lines.append(rolling_block)

    # -----------------------------
    # Valutazione complessiva (decisionale)
    # -----------------------------
    score = 0.0

    # Alpha statico: contributo principale
    if _is_num(a) and _is_num(p) and _is_num(t):
        if a > 0 and p < 0.05 and abs(t) >= 2.0:
            score += 2.5
        elif a > 0 and p < 0.10:
            score += 1.5
        elif a > 0:
            score += 0.5
        elif a < 0 and p < 0.05 and abs(t) >= 2.0:
            score -= 2.5
        elif a < 0 and p < 0.10:
            score -= 1.5
        elif a < 0:
            score -= 0.5

    # Tracking error: “qualità” dell’alpha (più TE = più difficile monetizzare/replicare)
    if _is_num(te):
        if te <= 0.12:
            score += 0.5
        elif te >= 0.25:
            score -= 0.5

    # N obs: affidabilità statistica
    if _is_num(nobs):
        if nobs >= 252:
            score += 0.5
        elif nobs < 126:
            score -= 0.5

    # Rolling contribution
    score += roll_score

    # Decision label
    if score >= 3.0:
        verdict = f"{BOLD}VALUTAZIONE COMPLESSIVA: FORTE{RESET} – evidenza robusta di alpha e/o miglioramento strutturale."
    elif score >= 1.5:
        verdict = f"{BOLD}VALUTAZIONE COMPLESSIVA: PROMETTENTE{RESET} – segnali coerenti, serve ulteriore conferma."
    elif score >= 0.0:
        verdict = f"{BOLD}VALUTAZIONE COMPLESSIVA: NEUTRA{RESET} – nessuna evidenza chiara; può essere regime-driven."
    elif score >= -1.5:
        verdict = f"{BOLD}VALUTAZIONE COMPLESSIVA: DEBOLE{RESET} – segnali sfavorevoli o non robusti."
    else:
        verdict = f"{BOLD}VALUTAZIONE COMPLESSIVA: NEGATIVA{RESET} – evidenza di sotto-performance vs benchmark."

    # Action note (molto operativo)
    action = (
        "Indicazione operativa: "
        "se la valutazione è FORTE/PROMETTENTE, ha senso considerare overlay (es. beta-neutral/hedged) "
        "e monitorare il rolling alpha/beta come “early warning”. "
        "Se NEUTRA/DEBOLE/NEGATIVA, evitare conclusioni su edge e lavorare su benchmark, fattori e robustezza (MC, WFO, OOS)."
    )

    lines.append(verdict)
    lines.append(action)

    return "\n".join(lines)
    
# ======================================================================
# End of integrated module
# ======================================================================


# def create_portfolio_summary(
#     portfolio,
#     benchmark_portfolio=None,
#     sel_tickers=None,
#     alpha_analysis=False,
#     risk_free_rate=0.02,
#     show=False,
#     run_as_app=False,
#     return_formatted: bool = False
# ):
#     import numpy as np
#     import pandas as pd
#     from tabulate import tabulate
#     import statsmodels.api as sm

#     # --- Base statistics ---
#     stats = portfolio.stats()
#     start_date = stats["Start"]
#     end_date = stats["End"]
#     init_invested = stats["Start Value"]
#     final_value = stats["End Value"]

#     # === PERIODO (UNA SOLA RIGA) ===
#     try:
#         start_ts = pd.to_datetime(start_date)
#         start_str = start_ts.date().isoformat()
#     except Exception:
#         start_str = str(start_date)

#     try:
#         end_ts = pd.to_datetime(end_date)
#         end_str = end_ts.date().isoformat()
#     except Exception:
#         end_str = str(end_date)

#     period_str = f"{start_str} → {end_str}"

#     # Giorni di trading (Period può essere timedelta o int)
#     period_val = stats["Period"]
#     period = period_val.days if hasattr(period_val, 'days') else int(period_val)
#     period = max(int(period), 1)  # guard-rail

#     # === CAGR unico (trading year = 252) ===
#     cagr = (final_value / init_invested) ** (252 / period) - 1

#     # --- Returns & benchmark alignment ---
#     portfolio_returns = portfolio.returns(group_by=True)

#     # normalizza index portfolio (per coerenza con benchmark e regressioni)
#     if hasattr(portfolio_returns, "index"):
#         if portfolio_returns.index.tz is not None:
#             portfolio_returns.index = portfolio_returns.index.tz_localize(None)
#         portfolio_returns.index = portfolio_returns.index.normalize()

#     if benchmark_portfolio is not None:
#         benchmark_returns = benchmark_portfolio.returns(group_by=True).copy()
#         if benchmark_returns.index.tz is not None:
#             benchmark_returns.index = benchmark_returns.index.tz_localize(None)
#         benchmark_returns.index = benchmark_returns.index.normalize()
#     else:
#         benchmark_returns = portfolio.benchmark_returns(group_by=True)
#         if benchmark_returns.index.tz is not None:
#             benchmark_returns.index = benchmark_returns.index.tz_localize(None)
#         benchmark_returns.index = benchmark_returns.index.normalize()

#     total_return  = portfolio.total_return(group_by=True)
#     volatility    = portfolio.annualized_volatility(group_by=True, freq='1D')
#     max_dd        = abs(portfolio.max_drawdown(group_by=True))

#     # === Sharpe coerente (base giornaliera) ===
#     rf_daily   = (1 + risk_free_rate) ** (1 / 252) - 1
#     mean_daily = portfolio_returns.mean()
#     std_daily  = portfolio_returns.std()
#     sharpe_ratio = (mean_daily - rf_daily) / std_daily * np.sqrt(252) if std_daily != 0 else np.nan

#     # === Adjusted metrics (coerenti) ===
#     adj_car_sharpe = sharpe_ratio
#     adj_car_calmar = (cagr / max_dd) if max_dd != 0 else np.nan

#     # --- Trade frequency (ops/mese) ---
#     if sel_tickers is None:
#         total_trades = stats.get('Total Trades', np.nan)
#     else:
#         total_trades = 0
#         dates = sel_tickers.index
#         for i in range(1, len(dates)):
#             prev_set = set(sel_tickers.loc[dates[i - 1], 'Top_Tickers'])
#             curr_set = set(sel_tickers.loc[dates[i], 'Top_Tickers'])
#             total_trades += len(curr_set - prev_set) + len(prev_set - curr_set)

#     total_months = period / 21
#     trades_per_month = total_trades / total_months if total_months > 0 else np.nan
#     month_op = trades_per_month * (2 if sel_tickers is None else 1)

#     # === Alpha analysis (opzionale) ===
#     alpha = beta = alpha_tstat = alpha_pvalue = te = corr = np.nan
#     alpha_ann_pct = np.nan
#     if alpha_analysis:
#         df_reg = pd.concat({"ret": portfolio_returns, "bm": benchmark_returns}, axis=1).dropna()

#         if df_reg.shape[0] >= 30:
#             df_reg["excess_ret"] = df_reg["ret"] - rf_daily
#             df_reg["excess_bm"]  = df_reg["bm"]  - rf_daily

#             X = sm.add_constant(df_reg["excess_bm"])
#             y = df_reg["excess_ret"]

#             model = sm.OLS(y, X).fit()
#             alpha = float(model.params["const"])
#             beta  = float(model.params["excess_bm"])
#             alpha_tstat  = float(model.tvalues["const"])
#             alpha_pvalue = float(model.pvalues["const"])
#             te = float(model.resid.std() * np.sqrt(252))
#             corr = float(df_reg["ret"].corr(df_reg["bm"]))
#             alpha_ann_pct = alpha * 252 * 100
#         else:
#             alpha = beta = alpha_tstat = alpha_pvalue = te = corr = np.nan
#             alpha_ann_pct = np.nan

#     # === Market Time Exposure (vettoriale) ===
#     try:
#         records = portfolio.trades.records_readable
#         idx = portfolio.returns(group_by=False).index
#         if idx.tz is not None:
#             idx = idx.tz_localize(None)
#         idx = idx.normalize()

#         events = pd.Series(0, index=idx, dtype='int64')

#         for _, row in records.iterrows():
#             entry_ts = row['Entry Timestamp']
#             exit_ts  = row['Exit Timestamp'] if pd.notnull(row['Exit Timestamp']) else end_date

#             if hasattr(entry_ts, 'tzinfo') and entry_ts.tzinfo is not None:
#                 entry_ts = entry_ts.tz_localize(None)
#             if hasattr(exit_ts, 'tzinfo') and exit_ts.tzinfo is not None:
#                 exit_ts = exit_ts.tz_localize(None)

#             try:
#                 entry_ts = pd.Timestamp(entry_ts).normalize()
#             except Exception:
#                 pass
#             try:
#                 exit_ts = pd.Timestamp(exit_ts).normalize()
#             except Exception:
#                 pass

#             entry_pos = idx.searchsorted(entry_ts, side='left')
#             if entry_pos < len(idx):
#                 events.iloc[entry_pos] += 1

#             exit_pos = idx.searchsorted(exit_ts, side='left')
#             if exit_pos < len(idx):
#                 events.iloc[exit_pos] -= 1

#         state = events.cumsum().astype(int) > 0
#         exposure = float(state.mean())
#     except Exception:
#         exposure = np.nan

#     # --- Min holding period in profit ---
#     L_min = find_min_positive_period(portfolio_returns)
#     if L_min is None:
#         L_min = "nessuno"

#     # === Assemble results ===
#     data_entries = [
#         ("Periodo", period_str, period_str),
#         ("Importo investito (€)", init_invested, f"{init_invested:.2f}"),
#         ("Valore patrimoniale netto (€)", final_value, f"{final_value:.2f}"),
#         ("Giorni di trading", period, period),
#         ("Ritorno totale", total_return, f"{total_return:.2%}"),
#         ("CAGR", cagr, f"{cagr:.2%}"),
#         ("Max Drawdown", max_dd, f"{max_dd:.2%}"),
#         ("Adjusted CAR (Sharpe-style)", adj_car_sharpe, round(adj_car_sharpe, 2) if pd.notna(adj_car_sharpe) else "n/a"),
#         ("Adjusted CAR (Calmar-style)", adj_car_calmar, round(adj_car_calmar, 2) if pd.notna(adj_car_calmar) else "n/a"),
#         ("Volatilità annua", volatility, f"{volatility:.2%}"),
#         ("Rapporto di Sharpe", sharpe_ratio, round(sharpe_ratio, 2) if pd.notna(sharpe_ratio) else "n/a"),
#         ("Operazioni al mese", month_op, round(month_op, 2) if pd.notna(month_op) else "n/a"),
#         ("Market Time Exposure", exposure, f"{exposure:.2%}" if pd.notna(exposure) else "n/a")
#     ]

#     if alpha_analysis:
#         data_entries.extend([
#             ("Alpha (giornaliero, assoluto)", alpha, f"{alpha:.6f}" if pd.notna(alpha) else "n/a"),
#             ("Alpha annualizzato (%)", alpha_ann_pct, f"{alpha_ann_pct:.2f}%" if pd.notna(alpha_ann_pct) else "n/a"),
#             ("Beta (rispetto a B&H)", beta, round(beta, 4) if pd.notna(beta) else "n/a"),
#             ("Correlazione (rispetto a B&H)", corr, round(corr, 4) if pd.notna(corr) else "n/a"),
#             ("T-stat Alpha", alpha_tstat, round(alpha_tstat, 2) if pd.notna(alpha_tstat) else "n/a"),
#             ("P-value Alpha", alpha_pvalue, float(f"{alpha_pvalue:.2e}") if pd.notna(alpha_pvalue) else "n/a"),
#             ("Tracking Error", te, round(te, 4) if pd.notna(te) else "n/a"),
#         ])

#     data_entries.append(("Durata minima in guadagno (giorni)", L_min, L_min))

#     # --- grezzo (numerico) ---
#     df_sommario = pd.DataFrame(
#         {'Valore': [v for _, v, _ in data_entries]},
#         index=[k for k, _, _ in data_entries]
#     )

#     # --- formattato (stringhe) ---
#     df_printing = None
#     if show or return_formatted:
#         df_printing = pd.DataFrame(
#             {'Valore': [pv for _, _, pv in data_entries]},
#             index=[k for k, _, _ in data_entries]
#         )

#     if show:
#         display(df_printing)

#         headers_raw = [
#             "Importo investito", "Valore finale netto", "CAGR (252)",
#             "Max Drawdown", "Deviazione standard", "Rapporto di Sharpe",
#             "Operazioni al mese", "Durata minima in guadagno"
#         ]
#         max_width = 13
#         wrapped_headers = ["\n".join(pd.Series(h).str.wrap(max_width).tolist()) for h in headers_raw]

#         try:
#             vals = [
#                 f"{BOLD}{init_invested:.2f}€{RESET}",
#                 f"{BOLD}{final_value:.2f}€{RESET}",
#                 f"{BOLD}{cagr:.2%}{RESET}",
#                 f"{BOLD}{max_dd:.2%}{RESET}",
#                 f"{BOLD}{volatility:.2%}{RESET}",
#                 f"{BOLD}{sharpe_ratio:.2f}{RESET}" if pd.notna(sharpe_ratio) else "n/a",
#                 f"{BOLD}{month_op:.2f}{RESET}" if pd.notna(month_op) else "n/a",
#                 f"{BOLD}{L_min}{RESET}"
#             ]
#         except NameError:
#             vals = [
#                 f"{init_invested:.2f}€",
#                 f"{final_value:.2f}€",
#                 f"{cagr:.2%}",
#                 f"{max_dd:.2%}",
#                 f"{volatility:.2%}",
#                 f"{sharpe_ratio:.2f}" if pd.notna(sharpe_ratio) else "n/a",
#                 f"{month_op:.2f}" if pd.notna(month_op) else "n/a",
#                 f"{L_min}"
#             ]

#         tab = tabulate([vals], headers=wrapped_headers, tablefmt="fancy_grid",
#                        colalign=("center",)*len(wrapped_headers))
#         print("\n================= SINTESI ==================")
#         print(tab)

#         return df_printing

#     if return_formatted and df_printing is not None:
#         return df_printing

#     return df_sommario
    
def print_summary(
    portfolio,
    benchmark_portfolio=None,
    sel_tickers=None,
    alpha_analysis=True,
    risk_free_rate=0.02,
    # run_as_app=False
):
    return create_portfolio_summary_refactored(
        portfolio,
        benchmark_portfolio=benchmark_portfolio,
        sel_tickers=sel_tickers,
        alpha_analysis=alpha_analysis,
        # run_as_app=run_as_app,
        risk_free_rate=risk_free_rate,
        show=True
    )
    
    

# # === CORE UNIFICATO ===

# def _generate_portfolio_performance_core(
#     *,
#     pf: 'vbt.Portfolio',
#     portfolio_title: str,
#     pf_b_h: 'vbt.Portfolio' = None,
#     # caratteristiche "standard"
#     portfolio_ts: list[dict] | None = None,
#     alpha_analysis: bool | None = None,
#     # caratteristiche "rotational"
#     sel_tickers: list | None = None,
#     method: str | None = None,
#     freq: str | None = None,
#     # benchmark
#     benchmark: str = 'SPY',
#     benchmark_data: pd.Series | None = None,
#     # finestre dei plot
#     plot_start_date: 'pd.Timestamp | str | None' = None,
#     plot_end_date: 'pd.Timestamp | str | None' = None,
#     # switch comportamento
#     mode: str = "standard",
#     # mostrare le figure
#     show_report: bool = True,
#     universe: str | None = None,
# ) -> dict:
#     """
#     Core condiviso: costruisce statistiche + grafici.
#     Ritorna SEMPRE un dict:
#       - figs
#       - header
#       - stats_df  (FORMATTATO come a video)
#       - sintesi_df (FORMATTATO come a video)
#     """
#     import pandas as pd
#     import matplotlib.pyplot as _plt

#     # ------------------------------------------------------------------
#     # Utility
#     # ------------------------------------------------------------------
#     def _maybe_show(fig):
#         if not show_report:
#             return
#         try:
#             fig.show()
#         except Exception:
#             try:
#                 _plt.show()
#             except Exception:
#                 pass
                
#     def _to_ts(x):
#         if x is None:
#             return None
#         try:
#             return pd.to_datetime(x)
#         except Exception:
#             return None

#     def _resolve_plot_window(idx: pd.Index, start, end):
#         if idx is None or len(idx) == 0:
#             return (None, None)

#         start_ts = _to_ts(start)
#         end_ts   = _to_ts(end)

#         idx_min = pd.to_datetime(idx.min())
#         idx_max = pd.to_datetime(idx.max())

#         if start_ts is None:
#             start_ts = idx_min
#         if end_ts is None:
#             end_ts = idx_max

#         if start_ts > end_ts:
#             start_ts, end_ts = end_ts, start_ts

#         if start_ts < idx_min:
#             start_ts = idx_min
#         if end_ts > idx_max:
#             end_ts = idx_max

#         if start_ts > idx_max or end_ts < idx_min:
#             return (None, None)

#         return (start_ts, end_ts)
        
#     def _has_data_in_window(pf: 'vbt.Portfolio', start, end) -> bool:
#         """True se esistono returns non-NaN nella finestra [start, end]."""
#         try:
#             r = pf.returns()
#             s, e = _resolve_plot_window(r.index, start, end)
#             if s is None:
#                 return False
#             return not r.loc[s:e].dropna().empty
#         except Exception:
#             return False

#     def is_one_ticker(pf) -> bool:
#         """
#         Ritorna True se il Portfolio contiene un solo ticker/colonna.
#         Robusto a pf.assets() che può essere Series o DataFrame.
#         """
#         assets = pf.assets()
    
#         # Caso più comune: DataFrame (multi-ticker)
#         if isinstance(assets, pd.DataFrame):
#             return assets.shape[1] == 1
    
#         # Caso: Series (spesso single-ticker o single-group)
#         if isinstance(assets, pd.Series):
#             # Se è una Series con name che rappresenta un ticker -> considerala 1 ticker.
#             # Se è una Series "group" aggregata, comunque lato tickers è 1 (perché non hai breakdown).
#             return True
    
#         # Fallback: prova ad inferire da pf.close (di solito è coerente)
#         close = getattr(pf, "close", None)
#         if close is not None:
#             if isinstance(close, pd.DataFrame):
#                 return close.shape[1] == 1
#             if isinstance(close, pd.Series):
#                 return True
    
#         # Ultimo fallback: non lo sappiamo => assume multi per prudenza
#         return False


#     stats = pf.stats()

#     start_dt = stats.get("Start", None)
#     end_dt   = stats.get("End", None)

#     try:
#         start_ts = pd.to_datetime(start_dt)
#     except Exception:
#         start_ts = None
#     try:
#         end_ts = pd.to_datetime(end_dt)
#     except Exception:
#         end_ts = None

#     start_str = start_ts.date().isoformat() if start_ts is not None else str(start_dt)
#     end_str   = end_ts.date().isoformat() if end_ts is not None else str(end_dt)
#     period_str = f"{start_str} → {end_str}"
    

#     # ------------------------------------------------------------------
#     # 1) HEADER + PRINT (come prima)
#     # ------------------------------------------------------------------
#     one_ticker = is_one_ticker(pf)
#     portfolio_type = "Titolo" if one_ticker else "Portfolio"

    
#     header = f"📈 Statistiche {portfolio_type} {portfolio_title} ({period_str})"

#     if mode == "standard":
#         if show_report:
#             print(header)
#             print_summary(pf, pf_b_h, alpha_analysis=alpha_analysis)
#     else:
#         if method:
#             header += f" | method: {method}"
#             if freq:
#                 header += f" freq: {freq}"
#         if show_report:
#             print(header)
#             print_summary(pf, sel_tickers=sel_tickers,alpha_analysis=alpha_analysis)

#     # ------------------------------------------------------------------
#     # 2) STATS_DF FORMATTATO (identico a video) SENZA show
#     # ------------------------------------------------------------------
#     if mode == "standard":
#         stats_df = create_portfolio_summary(
#             pf,
#             benchmark_portfolio=pf_b_h,
#             alpha_analysis=bool(alpha_analysis),
#             show=False,
#             return_formatted=True
#         )
#     else:
#         stats_df = create_portfolio_summary(
#             pf,
#             benchmark_portfolio=None,
#             sel_tickers=sel_tickers,
#             alpha_analysis=bool(alpha_analysis),
#             show=False,
#             return_formatted=True
#         )

#     # ------------------------------------------------------------------
#     # 3) SINTESI_DF FORMATTATO
#     # ------------------------------------------------------------------
#     def _pick_row(df, key):
#         return df.loc[key, "Valore"] if key in df.index else "n/a"

#     sintesi_df = pd.DataFrame([{
#         "Importo investito": _pick_row(stats_df, "Importo investito (€)"),
#         "Valore finale netto": _pick_row(stats_df, "Valore patrimoniale netto (€)"),
#         "CAGR (252)": _pick_row(stats_df, "CAGR"),
#         "Max Drawdown": _pick_row(stats_df, "Max Drawdown"),
#         "Deviazione standard": _pick_row(stats_df, "Volatilità annua"),
#         "Rapporto di Sharpe": _pick_row(stats_df, "Rapporto di Sharpe"),
#         "Operazioni al mese": _pick_row(stats_df, "Operazioni al mese"),
#         "Durata minima in guadagno": _pick_row(stats_df, "Durata minima in guadagno (giorni)"),
#     }])

#     # ------------------------------------------------------------------
#     # 4) PLOTS
#     # ------------------------------------------------------------------
#     figs: list = []
    
#     fig_value = pf.plot(
#         width=vbt_plot_width,
#         subplots=[
#             'cum_returns',
#             'drawdowns',
#             'underwater',
#             # luca
#             'gross_exposure'
#             ]
#     )
#     _maybe_show(fig_value); figs.append(fig_value)
    
#     # 3) Time-series portfolio (solo standard)
#     if mode == "standard" and portfolio_ts:
#         # if plot_start_date is None:
#         #     plot_start_date = pd.Timestamp(f"{_now_year()}-01-01")
#         fig_ts = plot_ts_portfolio(
#             final_portfolio=pf,
#             portfolio_ts=portfolio_ts,
#             portfolio_title=portfolio_title,
#             width=vbt_plot_width,
#             # start_date=plot_start_date,
#             start_date=ytd(),
#             # show_report=show_report
#         )
#         figs.append(fig_ts)

#     # ------------------------------------------------------------------
#     # 4) Strategy vs Benchmark (AGGREGATO ROBUSTO)
#     # ------------------------------------------------------------------
    
#     def _portfolio_total_returns_series(pf: 'vbt.Portfolio') -> pd.Series:
#         """
#         Ritorna i daily returns del PORTAFOGLIO aggregato come Series,
#         indipendente da come pf.returns(...) si comporta (Series o DataFrame).
#         """
#         v = pf.value()
#         # pf.value() può essere Series o DataFrame (per asset). Noi vogliamo il totale.
#         if isinstance(v, pd.DataFrame):
#             v = v.sum(axis=1)
    
#         v = v.dropna().copy()
    
#         try:
#             if v.index.tz is not None:
#                 v.index = v.index.tz_localize(None)
#         except Exception:
#             pass
    
#         v.index = pd.to_datetime(v.index).normalize()
#         r = v.pct_change().dropna()
#         return r
    
    
#     def _benchmark_returns_series(benchmark_data: pd.Series, idx: pd.DatetimeIndex) -> pd.Series:
#         """
#         Ritorna i daily returns benchmark riallineati sull'indice del portafoglio.
#         NOTA: benchmark_data è SERIE PREZZI (Close). Qui facciamo pct_change e poi reindex/ffill.
#         """
#         b = benchmark_data.dropna().copy()
    
#         try:
#             if b.index.tz is not None:
#                 b.index = b.index.tz_localize(None)
#         except Exception:
#             pass
    
#         b.index = pd.to_datetime(b.index).normalize()
#         b = b.sort_index()
#         if b.index.duplicated().any():
#             b = b[~b.index.duplicated(keep="last")]
    
#         br = b.pct_change().dropna()
    
#         # Riallinea ESATTAMENTE alle date del portafoglio (coerenza con plot_multiple_portfolios)
#         br = br.reindex(idx, method="ffill").fillna(0.0)
#         return br
    
    
#     # --- returns PORTAFOGLIO (Series) - sempre
#     returns_strat = pf.returns()
    
#     portfolios_returns = {f"{portfolio_title} (Strategy)": returns_strat}
    
#     # --- B&H se presente: stesso metodo robusto
#     if pf_b_h is not None:
#         # returns_bh = _portfolio_total_returns_series(pf_b_h)
#         returns_bh = pf_b_h.returns()
#         portfolios_returns[f"{portfolio_title} (B&H)"] = returns_bh
    
#     # --- BENCHMARK: costruiscilo UNA VOLTA e mettilo DENTRO portfolios_returns
#     bench_name = None
#     if (mode == "rotational" or mode == "lazy") and benchmark_data is not None:
#         bench_name = f"Benchmark ({benchmark})" if benchmark else "Benchmark"
#         bench_ret = _benchmark_returns_series(benchmark_data, returns_strat.index)
#         portfolios_returns[bench_name] = bench_ret
    
#     p_start = getattr(stats, "Start")
#     p_end   = getattr(stats, "End")
#     # print(f"p_start: {p_start} p_end: {p_end}")
#     start_year = p_start.year
#     end_year   = p_end.year
#     curr_year = _now_year()


#     # NB: benchmark già dentro portfolios_returns -> NON passare benchmark/benchmark_data
#     if (start_year == curr_year): 
#         base=0
#         # base=1
#     else:
#         base=1

#     fig_vs = plot_multiple_portfolios(
#         portfolios_returns,
#         benchmark=None,
#         title="Performance vs Benchmark",
#         benchmark_data=None,
#         # benchmark_data=benchmark_data,
#         start_date=p_start,
#         end_date=p_end,
#         base=base
#     )
#     _maybe_show(fig_vs); figs.append(fig_vs)

    
#     # 4.1) Confronto Portfolio vs Benchmark anno corrente

#     if (start_year != curr_year) and (end_year >= curr_year):
#         ytd_start = _ytd_start()
#         if _has_data_in_window(pf, ytd_start, plot_end_date):
#             base=0
#             # base=1
#             fig_vs_ytd = plot_multiple_portfolios(
#                 portfolios_returns,
#                 title="Performance vs Benchmark (YTD)",
#                 benchmark=None,
#                 benchmark_data=None,
#                 start_date=_to_ts(ytd_start),
#                 end_date=_to_ts(plot_end_date),
#                 base=base
#             )
#             _maybe_show(fig_vs_ytd); figs.append(fig_vs_ytd)
    
    
#     # ------------------------------------------------------------------
#     # 5) Annual returns
#     # ------------------------------------------------------------------
#     # NB: benchmark già dentro portfolios_returns -> NON passare benchmark/benchmark_data
#     fig_annual = plot_annual_performance(
#         portfolios_returns,
#         benchmark=None,
#         benchmark_data=None,
#         # title="Confronto rendimenti annuali (%)"
#     )
#     _maybe_show(fig_annual); figs.append(fig_annual)

#     fig_annual_hist = plot_year_returns_histogram(
#         pf,
#         title=f"Istogramma dei rendimenti annuali – {portfolio_title}",
#         panel_width=0.34, gap=0.02, hist_fill=0.95,
#         min_years=2
#     )
#     if fig_annual_hist is not None:
#         _maybe_show(fig_annual_hist)
#         figs.append(fig_annual_hist)

#     # 6) Monthly returns
#     fig_monthly = plot_monthly_returns(
#         pf,
#         eoy=True,
#         title=f"Portfolio {portfolio_title} - Rendimenti mensili (%)",
#         width=vbt_plot_width if mode == "standard" else None
#     )
#     _maybe_show(fig_monthly); figs.append(fig_monthly)

#     fig_monthly_hist = plot_monthly_returns_histogram(
#         pf,
#         title=f"Istogramma dei rendimenti mensili – {portfolio_title}"
#     )
#     _maybe_show(fig_monthly_hist); figs.append(fig_monthly_hist)

#     # 7) Frequenza selezione tickers
#     if mode == "rotational" and sel_tickers is not None:
#         fig_sel_tickers, freq_df, freq_full_df, unselected_info = plot_ticker_frequencies(
#             sel_tickers,
#             start_date=p_start,
#             end_date=p_end,
#             include_prev=True,  # include l'ultima selezione prima dello start (se esiste)
#             universe=universe
#         )
#         _maybe_show(fig_sel_tickers); figs.append(fig_sel_tickers)
#         unselected_df, n_unselected, pct_unselected = unselected_info
#         print(f"{BOLD}{n_unselected}{RESET} tickers su {BOLD}{len(universe)}{RESET} mai selezionati {BOLD}({pct_unselected:.1%}){RESET}")

#     # 8) Performance componenti
#     if mode == "standard" and portfolio_ts is not None:
#         returns = pd.Series({d["symbol"]: d["returns"] for d in portfolio_ts})
#         fig_tickers_perf = plot_total_return_per_ticker(
#             returns, start_date=_ytd_start(), end_date=None
#         )
#         _maybe_show(fig_tickers_perf); figs.append(fig_tickers_perf)
#     elif mode == "rotational":
#         performance = pf.total_return(group_by=False) * 100
#         fig_tickers_perf = plot_total_return_per_ticker(performance[performance != 0])
#         _maybe_show(fig_tickers_perf); figs.append(fig_tickers_perf)

#     # 9–11) Rolling, triangle, contributions
#     if mode == "rotational" or mode == "lazy":
#         if (start_year != curr_year):
    
#             out_roll = plot_cumulative_and_rolling_returns(
#                 pf,
#                 horizons_years=[1, 2, 3, 5],
#                 add_fan_chart=False,
#                 add_heatmap=True,
#                 return_extras=True,
#                 add_horizon_analysis=True,
#             )
    
#             _maybe_show(out_roll["fig"]); figs.append(out_roll["fig"])
    
#             if out_roll["fig_hm"] is not None:
#                 _maybe_show(out_roll["fig_hm"]); figs.append(out_roll["fig_hm"])
    
#             if out_roll["fig_loss"] is not None:
#                 _maybe_show(out_roll["fig_loss"]); figs.append(out_roll["fig_loss"])
    
#             # Triangle
#             fig_triangle, _ = plot_annual_return_triangle(pf, resample_freq="YE")
#             _maybe_show(fig_triangle); figs.append(fig_triangle)

#         fig_assets = build_and_plot_portfolio_contributions(
#             pf,
#             title=portfolio_title,
#             benchmark=benchmark,
#             benchmark_data=benchmark_data,
#             # start_date=plot_start_date,
#             start_date=p_start,
#             # end_date=plot_end_date,
#             end_date=p_end,
#             show_report=show_report
#         )
#         if fig_assets is not None:
#             _maybe_show(fig_assets)
#             figs.append(fig_assets)

#         if (start_year != curr_year):
#             fig_assets_ytd = build_and_plot_portfolio_contributions(
#                 pf,
#                 title=portfolio_title,
#                 benchmark=benchmark,
#                 benchmark_data=benchmark_data,
#                 # start_date=plot_start_date,
#                 start_date=ytd(),
#                 # end_date=plot_end_date,
#                 end_date=p_end,
#                 show_report=show_report
#             )
#             if fig_assets_ytd is not None:
#                 _maybe_show(fig_assets_ytd)
#                 figs.append(fig_assets_ytd)
                
#         # 12) Evoluzione dei pesi degli asset
#         fig_weights = visualize_portfolio_weights(pf=pf, title=portfolio_title)
#         if fig_weights is not None:
#             _maybe_show(fig_weights)
#             figs.append(fig_weights)  
    
#     out_dict = {
#         "figs": figs,
#         "header": header,
#         "stats_df": stats_df,      # <-- FORMATTATO
#         "sintesi_df": sintesi_df,  # <-- FORMATTATO
#     }

#     if mode == "rotational":
    
#         freq_df = (freq_df
#                    .sort_values("Frequenza", ascending=False)
#                    .set_index("Ticker"))

#         out_dict.update({
#             "performance_info": "Informazioni sulla selezione dei titoli:",
#             "performance_tables": {"Sequenza di selezioni": sel_tickers,
#                                    "Frequenze di selezione":freq_df
#                                   }
#         })

#     return out_dict    

    
# def generate_portfolio_performance(
#     pf: 'vbt.Portfolio',
#     portfolio_title: str,
#     pf_b_h: 'vbt.Portfolio',
#     portfolio_ts: list[dict] | None = None,
#     alpha_analysis: bool = True,
#     benchmark: str = 'SPY',
#     plot_start_date: 'pd.Timestamp | str | None' = None,
#     plot_end_date: 'pd.Timestamp | str | None' = None,
#     show_report: bool = True
# ) -> dict:
#     """
#     Adapter STANDARD.
#     Ritorna SEMPRE un dict con:
#       - figs
#       - header
#       - stats_df   (formattato come a video)
#       - sintesi_df (formattato come a video)
#     """

#     return _generate_portfolio_performance_core(
#         pf=pf,
#         portfolio_title=portfolio_title,
#         pf_b_h=pf_b_h,
#         portfolio_ts=portfolio_ts,
#         benchmark=benchmark,
#         alpha_analysis=alpha_analysis,
#         plot_start_date=plot_start_date,
#         plot_end_date=plot_end_date,
#         mode="standard",
#         show_report=show_report
#     )
    
# def generate_rotational_portfolio_performance(
#     pf: 'vbt.Portfolio',
#     portfolio_title: str,
#     sel_tickers: list | None = None,
#     benchmark: str = 'SPY',
#     benchmark_data: 'pd.Series | None' = None,
#     plot_start_date: 'pd.Timestamp | str | None' = None,
#     plot_end_date: 'pd.Timestamp | str | None' = None,
#     method: str | None = None,
#     freq: str | None = None,
#     alpha_analysis: bool = True,
#     show_report: bool = True,
#     universe=[]
# ) -> dict:
#     """
#     Adapter ROTATIONAL.
#     Ritorna SEMPRE un dict con:
#       - figs
#       - header
#       - stats_df   (formattato come a video)
#       - sintesi_df (formattato come a video)
#     """
          
#     return _generate_portfolio_performance_core(
#         pf=pf,
#         portfolio_title=portfolio_title,
#         pf_b_h=None,
#         sel_tickers=sel_tickers,
#         method=method,
#         freq=freq,
#         alpha_analysis=alpha_analysis,
#         benchmark=benchmark,
#         benchmark_data=benchmark_data,
#         plot_start_date=plot_start_date,
#         plot_end_date=plot_end_date,
#         mode="rotational",
#         show_report=show_report,
#         universe=universe
#     )

# def generate_lazy_portfolio_performance(
#     pf: 'vbt.Portfolio',
#     portfolio_title: str,
#     benchmark: str = 'SPY',
#     benchmark_data: 'pd.Series | None' = None,
#     method: str | None = None,
#     freq: str | None = None,
#     alpha_analysis: bool = True,
#     show_report: bool = True,
# ) -> dict:
#     """
#     Adapter ROTATIONAL.
#     Ritorna SEMPRE un dict con:
#       - figs
#       - header
#       - stats_df   (formattato come a video)
#       - sintesi_df (formattato come a video)
#     """
          
#     return _generate_portfolio_performance_core(
#         pf=pf,
#         portfolio_title=portfolio_title,
#         method=method,
#         freq=freq,
#         alpha_analysis=alpha_analysis,
#         benchmark=benchmark,
#         benchmark_data=benchmark_data,
#         mode="lazy",
#         show_report=show_report,
#     )



#
# Portfolio Weights Evolution
#


def _compute_weights_from_pf(pf) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calcola weights e asset_values (incluso CASH) da un vectorbt.Portfolio."""
    if pf is None:
        raise ValueError("pf is None")

    qty = pf.assets()
    if isinstance(qty, pd.Series):
        qty = qty.to_frame()

    prices = pf.close
    if isinstance(prices, pd.Series):
        prices = prices.to_frame()

    qty, prices = qty.align(prices, join="inner", axis=0)
    qty, prices = qty.align(prices, join="inner", axis=1)

    asset_values = qty * prices

    if hasattr(pf, "value"):
        total_value = pf.value().reindex(asset_values.index)
    else:
        total_value = asset_values.sum(axis=1)

    weights = asset_values.div(total_value, axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    if hasattr(pf, "cash"):
        cash = pf.cash().reindex(asset_values.index).fillna(0.0)
        weights["CASH"] = (cash / total_value).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        asset_values["CASH"] = cash

    return weights, asset_values


def _collapse_weights_row(w_row: pd.Series, top_n: int = 12, keep: tuple[str, ...] = ("CASH",)) -> pd.Series:
    """Collassa un vettore pesi in Top-N + OTHER, mantenendo sempre le voci in `keep` se presenti."""
    w = w_row.copy()
    w = w[w.abs() > 0]

    forced = pd.Series(dtype=float)
    for k in keep:
        if k in w.index:
            forced.loc[k] = w.loc[k]
            w = w.drop(index=k)

    w = w.sort_values(ascending=False)
    top = w.iloc[:max(top_n, 0)]
    other = w.iloc[max(top_n, 0):].sum()

    out = pd.concat([top, forced])
    if other > 0:
        out.loc["OTHER"] = other

    return out.sort_values(ascending=False)


def _snapshot_yearly(weights: pd.DataFrame, shift_trading_days: int = 2, current_label: str = "CURRENT") -> pd.DataFrame:
    """
    Snapshot pesi all'inizio di ogni anno (shift di N trading days) + CURRENT (ultimo giorno disponibile).
    """
    idx = weights.index
    years = pd.Index(idx.year).unique()

    snap_dates = []
    for y in years:
        idx_year = idx[idx.year == y]
        if len(idx_year) == 0:
            continue
        pos = min(shift_trading_days, len(idx_year) - 1)  # +2 trading days se shift_trading_days=2
        snap_dates.append(idx_year[pos])

    snap = weights.loc[snap_dates].copy()
    snap.index = pd.Index([d.year for d in snap.index], name="Year")

    last_date = weights.index[-1]
    current_row = weights.loc[[last_date]].copy()
    current_row.index = pd.Index([current_label], name="Year")

    return pd.concat([snap, current_row], axis=0)


def _cash_metrics(cash_w: pd.Series, thresholds=(0.005, 0.01, 0.02, 0.05)) -> pd.DataFrame:
    """Metriche sintetiche su CASH weight (0..1)."""
    cw = cash_w.dropna().astype(float)

    stats = {
        "Cash mean (%)": cw.mean() * 100,
        "Cash median (%)": cw.median() * 100,
        "Cash max (%)": cw.max() * 100,
        "Cash p90 (%)": cw.quantile(0.90) * 100,
        "Cash p95 (%)": cw.quantile(0.95) * 100,
        "Cash p99 (%)": cw.quantile(0.99) * 100,
    }

    for th in thresholds:
        stats[f"Days cash > {th*100:.1f}%"] = int((cw > th).sum())
        stats[f"Pct days cash > {th*100:.1f}%"] = (cw > th).mean() * 100

    # Max streak > 1%
    arr = (cw > 0.01).to_numpy(dtype=bool)
    best = cur = 0
    for v in arr:
        if v:
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    stats["Max consecutive days cash > 1.0%"] = best

    df = pd.DataFrame.from_dict(stats, orient="index", columns=["Value"])
    df["Value"] = df["Value"].astype(float).round(3)
    return df


def _cash_by_year(cash_w: pd.Series) -> pd.DataFrame:
    """Tabella annuale: media/max cash% per anno + CURRENT (ultimo valore come max)."""
    cw = cash_w.dropna().astype(float)

    df = pd.DataFrame({"cash_w": cw})
    df["year"] = df.index.year

    out = df.groupby("year")["cash_w"].agg(["mean", "max"])
    out = out.rename(columns={"mean": "Cash mean (%)", "max": "Cash max (%)"})
    out["Cash mean (%)"] = (out["Cash mean (%)"] * 100).round(2)
    out["Cash max (%)"] = (out["Cash max (%)"] * 100).round(2)
    out.index.name = "Year"

    current = pd.DataFrame(
        {"Cash mean (%)": [np.nan], "Cash max (%)": [round(cw.iloc[-1] * 100, 2)]},
        index=pd.Index(["CURRENT"], name="Year"),
    )
    return pd.concat([out, current], axis=0)


def visualize_portfolio_weights(
    pf,
    title="Portfolio Weights",
    show_report: bool = True,
    vbt_plot_width: int | None = None,
    auto_threshold: int = 12,
    top_n: int = 12,
    shift_trading_days: int = 2,
    fig_height: int = 1150
):
    """
    Auto-mode:
    - Se tickers <= auto_threshold: stacked area full.
    - Altrimenti: stacked area compressa (Top-N + CASH + OTHER).

    Report:
    - Tabella snapshot annuali (inizio anno + CURRENT) via my_display.
    - Cash metrics (tabella) + Cash by Year (tabella) via my_display.

    Figure unica (3 righe x 1 colonna):
    - Row 1: stacked weights evolution
    - Row 2: ONE pie con dropdown per selezione anno/CURRENT (titolo aggiornato)
    - Row 3: CASH (%) over time
    """
    try:
        weights, _ = _compute_weights_from_pf(pf)
    except Exception:
        return None
    if weights.empty:
        return None

    n_cols = weights.shape[1]
    full_mode = n_cols <= auto_threshold

    # Snapshot annuali (+ CURRENT)
    snap = _snapshot_yearly(weights, shift_trading_days=shift_trading_days, current_label="CURRENT")

    # --- Tabella snapshots (auto-mode) ---
    if full_mode:
        snap_table = (snap * 100).round(2)
        weights_plot = weights
    else:
        collapsed_rows = []
        for idx_row in snap.index:
            row_c = _collapse_weights_row(snap.loc[idx_row], top_n=top_n, keep=("CASH",))
            row_c.name = idx_row
            collapsed_rows.append(row_c)

        snap_c = pd.DataFrame(collapsed_rows).fillna(0.0)
        snap_c.index.name = "Year"
        snap_table = (snap_c * 100).round(2)

        # Area: Top-N per peso medio (escludendo CASH)
        mean_w = weights.drop(columns=["CASH"], errors="ignore").mean().sort_values(ascending=False)
        top_cols = list(mean_w.iloc[:top_n].index)

        cols_to_keep = top_cols.copy()
        if "CASH" in weights.columns:
            cols_to_keep.append("CASH")

        w_sub = weights[cols_to_keep].copy()
        other_cols = [c for c in weights.columns if c not in cols_to_keep]
        if len(other_cols) > 0:
            w_sub["OTHER"] = weights[other_cols].sum(axis=1)

        w_sub = w_sub.div(w_sub.sum(axis=1), axis=0).fillna(0.0)
        weights_plot = w_sub

    # --- Report tabelle ---
    if show_report:
        my_display(snap_table, title=f"{title} - Yearly Snapshots")
        if "CASH" in weights.columns:
            my_display(_cash_metrics(weights["CASH"]), title=f"{title} - Cash Metrics")
            my_display(_cash_by_year(weights["CASH"]), title=f"{title} - Cash by Year (mean/max)")

    # --- Pie data per label (Year/CURRENT) ---
    pie_data = {}
    for lbl in snap.index:
        row = snap.loc[lbl]
        if not full_mode:
            row = _collapse_weights_row(row, top_n=top_n, keep=("CASH",))
        else:
            row = row[row.abs() > 0].sort_values(ascending=False)
        pie_data[str(lbl)] = (row.index.astype(str).tolist(), row.values.tolist())

    # Ordine dropdown: anni (crescenti) poi CURRENT
    years_only = sorted([k for k in pie_data.keys() if k != "CURRENT"], key=lambda x: int(x))
    pie_keys = years_only + (["CURRENT"] if "CURRENT" in pie_data else [])

    default_lbl = "CURRENT" if "CURRENT" in pie_data else pie_keys[-1]

    def _pie_title(lbl: str) -> str:
        return f"{title} — Allocation (select year) — showing: {lbl}"

    # --- Figure 3x1 ---
    fig = make_subplots(
        rows=3, cols=1,
        specs=[[{"type": "xy"}],
               [{"type": "domain"}],
               [{"type": "xy"}]],
        row_heights=[0.55, 0.25, 0.20],
        vertical_spacing=0.14,
        subplot_titles=(
            f"{title} — Weights evolution" + ("" if full_mode else f" (Top {top_n} + OTHER)"),
            _pie_title(default_lbl),
            f"{title} — CASH (%) over time"
        )
    )

    # Row 1: stacked area
    for col in weights_plot.columns:
        fig.add_trace(
            go.Scatter(
                x=weights_plot.index,
                y=weights_plot[col],
                mode="lines",
                stackgroup="one",
                name=str(col),
                hovertemplate="%{x|%Y-%m-%d}<br>%{y:.2%}<extra>" + str(col) + "</extra>"
            ),
            row=1, col=1
        )
    fig.update_yaxes(title_text="Weight", tickformat=".0%", row=1, col=1)

    # Row 2: pie
    fig.add_trace(
        go.Pie(
            labels=pie_data[default_lbl][0],
            values=pie_data[default_lbl][1],
            hole=0.45,
            sort=False,
            textinfo="label+percent",
            showlegend=False
        ),
        row=2, col=1
    )

    # Row 3: CASH line
    if "CASH" in weights.columns:
        fig.add_trace(
            go.Scatter(
                x=weights.index,
                y=weights["CASH"] * 100,
                mode="lines",
                name="CASH (%)",
                hovertemplate="%{x|%Y-%m-%d}<br>%{y:.2f}%<extra>CASH</extra>"
            ),
            row=3, col=1
        )
        fig.update_yaxes(title_text="CASH (%)", ticksuffix="%", row=3, col=1)
    else:
        fig.update_yaxes(visible=False, row=3, col=1)
        fig.update_xaxes(visible=False, row=3, col=1)

    # --- trova l'annotation index del titolo del subplot Pie (robusto) ---
    pie_title_annotation_index = None
    if hasattr(fig.layout, "annotations") and fig.layout.annotations:
        for i, ann in enumerate(fig.layout.annotations):
            if isinstance(getattr(ann, "text", None), str) and "Allocation" in ann.text:
                pie_title_annotation_index = i
                break
    if pie_title_annotation_index is None:
        pie_title_annotation_index = 1  # fallback tipico per layout 3x1

    # --- Dropdown: imposta active correttamente sul default_lbl ---
    active_idx = pie_keys.index(default_lbl) if default_lbl in pie_keys else 0

    fig.update_layout(
        updatemenus=[
            dict(
                type="dropdown",
                direction="down",
                x=0.5,
                xanchor="center",
                y=0.53,
                yanchor="top",
                active=active_idx,  # ✅ FIX: allinea selettore al pie iniziale
                buttons=[
                    dict(
                        label=str(lbl),
                        method="update",
                        args=[
                            {"labels": [pie_data[str(lbl)][0]],
                             "values": [pie_data[str(lbl)][1]]},
                            {f"annotations[{pie_title_annotation_index}].text": _pie_title(str(lbl))}
                        ],
                    )
                    for lbl in pie_keys
                ],
            )
        ]
    )

    # Layout sizing
    fig.update_layout(
        autosize=False if vbt_plot_width is not None else True,
        width=int(vbt_plot_width) if vbt_plot_width is not None else None,
        height=int(fig_height),
        title=dict(text=title, x=0.5),
        hovermode="x unified",
        legend_title_text="Ticker",
        margin=dict(l=60, r=60, t=120, b=60)
    )

    return fig
    

###############################################################################
# UTILITY: CREAZIONE DELLA MASCHERA DI OPERATIVITÀ
###############################################################################
def compare_selection_columns(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    column: str = "Top_Tickers",
    label_a: str = "Selezione A",
    label_b: str = "Selezione B",
    compare_only_common_dates: bool = True,
    sort_table_by_diff: bool = False,
    display_table: bool = True,
):
    """
    Confronta due DataFrame contenenti una colonna di selezioni ticker.

    Ogni cella della colonna deve idealmente contenere una lista di ticker,
    ma la funzione gestisce anche:
      - NaN / None
      - stringhe singole
      - tuple / set / np.ndarray / pd.Index

    Parametri
    ---------
    df1, df2 : pd.DataFrame
        DataFrame da confrontare.

    column : str
        Nome della colonna che contiene le selezioni ticker.

    label_a, label_b : str
        Etichette descrittive delle due selezioni.
        Esempio:
            label_a="con risk on/off"
            label_b="senza risk on/off"

    compare_only_common_dates : bool
        Se True, confronta solo le date presenti in entrambi i DataFrame.
        Se False, mantiene l'unione degli indici e converte eventuali NaN in liste vuote.

    sort_table_by_diff : bool
        Se True, la tabella visualizzata viene ordinata per Diff_Count decrescente.

    display_table : bool
        Se True, visualizza la tabella con ace_tools_open.

    Ritorna
    -------
    df_compare : pd.DataFrame
        DataFrame con selezioni e metriche di confronto.
    """

    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    from IPython.display import display

    if column not in df1.columns:
        raise ValueError(f"Colonna '{column}' non presente in df1.")

    if column not in df2.columns:
        raise ValueError(f"Colonna '{column}' non presente in df2.")

    def _to_list(x):
        if x is None:
            return []

        if isinstance(x, float) and np.isnan(x):
            return []

        if isinstance(x, (list, tuple, set, np.ndarray, pd.Index)):
            return [item for item in list(x) if pd.notna(item)]

        if isinstance(x, str):
            return [x]

        return []

    # ------------------------------------------------------------
    # Allineamento indici
    # ------------------------------------------------------------
    if compare_only_common_dates:
        common_index = df1.index.intersection(df2.index)

        df_compare = pd.concat(
            [
                df1.loc[common_index, column],
                df2.loc[common_index, column],
            ],
            axis=1
        )
    else:
        df_compare = pd.concat(
            [
                df1[column],
                df2[column],
            ],
            axis=1
        )

    # Colonne tecniche interne
    df_compare.columns = ["Sel_A", "Sel_B"]

    # ------------------------------------------------------------
    # Normalizzazione celle
    # ------------------------------------------------------------
    df_compare["Sel_A"] = df_compare["Sel_A"].apply(_to_list)
    df_compare["Sel_B"] = df_compare["Sel_B"].apply(_to_list)

    set_a = df_compare["Sel_A"].apply(set)
    set_b = df_compare["Sel_B"].apply(set)

    # ------------------------------------------------------------
    # Metriche di confronto
    # ------------------------------------------------------------
    df_compare["In_Common"] = [
        sorted(a & b) for a, b in zip(set_a, set_b)
    ]

    df_compare[f"Solo in {label_a}"] = [
        sorted(a - b) for a, b in zip(set_a, set_b)
    ]

    df_compare[f"Solo in {label_b}"] = [
        sorted(b - a) for a, b in zip(set_a, set_b)
    ]

    df_compare[f"N {label_a}"] = df_compare["Sel_A"].apply(len)
    df_compare[f"N {label_b}"] = df_compare["Sel_B"].apply(len)

    df_compare["N_Common"] = df_compare["In_Common"].apply(len)
    df_compare[f"N solo {label_a}"] = df_compare[f"Solo in {label_a}"].apply(len)
    df_compare[f"N solo {label_b}"] = df_compare[f"Solo in {label_b}"].apply(len)

    df_compare["Union_Count"] = [
        len(a | b) for a, b in zip(set_a, set_b)
    ]

    df_compare["Diff_Count"] = [
        len(a ^ b) for a, b in zip(set_a, set_b)
    ]

    df_compare["Jaccard"] = [
        len(a & b) / len(a | b) if len(a | b) > 0 else np.nan
        for a, b in zip(set_a, set_b)
    ]

    # ------------------------------------------------------------
    # Grafico 1: Jaccard Similarity
    # ------------------------------------------------------------
    fig_jaccard = go.Figure()

    fig_jaccard.add_trace(
        go.Scatter(
            x=df_compare.index,
            y=df_compare["Jaccard"],
            mode="lines+markers",
            name="Similarità Jaccard",
            hovertemplate=(
                "Data: %{x}<br>"
                "Similarità Jaccard: %{y:.2f}<br>"
                "<extra></extra>"
            )
        )
    )

    fig_jaccard.update_layout(
        title=f"Similarità tra selezioni ({column})",
        xaxis_title="Data di Ribilanciamento",
        yaxis_title="Similarità Jaccard",
        yaxis=dict(range=[0, 1.05]),
        width=1000,
        height=470,
        margin=dict(t=60, b=95),
        annotations=[
            dict(
                text=(
                    "Jaccard Similarity = ticker in comune / ticker totali unici tra le due selezioni. "
                    "Valore 1.00 = selezioni identiche; valore 0.00 = nessun ticker in comune."
                ),
                xref="paper",
                yref="paper",
                x=0,
                y=-0.28,
                showarrow=False,
                align="left",
                font=dict(size=11)
            )
        ]
    )

    display(fig_jaccard)

    # ------------------------------------------------------------
    # Grafico 2: composizione differenze
    # ------------------------------------------------------------
    x_labels = df_compare.index.strftime("%Y-%m-%d")

    fig_stack = go.Figure()

    fig_stack.add_trace(
        go.Bar(
            x=x_labels,
            y=df_compare["N_Common"],
            name="In comune",
            hovertemplate=(
                "Data: %{x}<br>"
                "Ticker comuni: %{y}<br>"
                "<extra></extra>"
            )
        )
    )

    fig_stack.add_trace(
        go.Bar(
            x=x_labels,
            y=df_compare[f"N solo {label_a}"],
            name=f"Solo in {label_a}",
            hovertemplate=(
                "Data: %{x}<br>"
                f"Solo in {label_a}: "
                "%{y}<br>"
                "<extra></extra>"
            )
        )
    )

    fig_stack.add_trace(
        go.Bar(
            x=x_labels,
            y=df_compare[f"N solo {label_b}"],
            name=f"Solo in {label_b}",
            hovertemplate=(
                "Data: %{x}<br>"
                f"Solo in {label_b}: "
                "%{y}<br>"
                "<extra></extra>"
            )
        )
    )

    fig_stack.update_layout(
        title=f"Composizione differenze tra selezioni ({column})",
        barmode="stack",
        xaxis_title="Data di Ribilanciamento",
        yaxis_title="Numero ticker",
        width=1100,
        height=500,
        legend_title="Categoria",
    )

    fig_stack.update_xaxes(
        type="category",
        tickangle=45,
    )

    display(fig_stack)

    # ------------------------------------------------------------
    # Tabella dettagliata con nomi leggibili
    # ------------------------------------------------------------
    df_table = df_compare[
        [
            "Sel_A",
            "Sel_B",
            "In_Common",
            f"Solo in {label_a}",
            f"Solo in {label_b}",
            f"N {label_a}",
            f"N {label_b}",
            "N_Common",
            f"N solo {label_a}",
            f"N solo {label_b}",
            "Diff_Count",
            "Union_Count",
            "Jaccard",
        ]
    ].copy()

    df_table = df_table.rename(
        columns={
            "Sel_A": label_a,
            "Sel_B": label_b,
            "In_Common": "In comune",
            "N_Common": "N in comune",
            "Diff_Count": "N diversi",
            "Union_Count": "N unici totali",
            "Jaccard": "Similarità Jaccard",
        }
    )

    if sort_table_by_diff:
        df_table = df_table.sort_values(
            ["N diversi", "Similarità Jaccard"],
            ascending=[False, True]
        )

    if display_table:
        try:
            import ace_tools_open as tools
            tools.display_dataframe_to_user(
                name=f"Confronto selezioni {column}",
                dataframe=df_table
            )
        except ImportError:
            display(df_table)

    return df_compare
    
# def compare_selection_columns_BAD(df1: pd.DataFrame, df2: pd.DataFrame, column: str = "Top_Tickers"):
#     # Unione dei due DataFrame
#     df_compare = pd.concat([
#         df1[column],
#         df2[column]
#     ], axis=1)

#     # df_compare.columns = ['Sel_A', 'Sel_B']

#     # # Normalizza NaN/scalari → lista vuota (robustezza)
#     # def _to_list(v):
#     #     if v is None or isinstance(v, float):
#     #         return []
#     #     if isinstance(v, (list, tuple, set)):
#     #         return list(v)
#     #     if isinstance(v, str):
#     #         return [v]
#     #     try:
#     #         return list(v)
#     #     except TypeError:
#     #         return []
    
#     # df_compare['Sel_A'] = df_compare['Sel_A'].apply(_to_list)
#     # df_compare['Sel_B'] = df_compare['Sel_B'].apply(_to_list)
    
#     # # Calcolo differenze
#     # df_compare['In_Common']  = df_compare.apply(lambda row: sorted(set(row['Sel_A']) & set(row['Sel_B'])), axis=1)
#     # df_compare['Only_in_A']  = df_compare.apply(lambda row: sorted(set(row['Sel_A']) - set(row['Sel_B'])), axis=1)
#     # df_compare['Only_in_B']  = df_compare.apply(lambda row: sorted(set(row['Sel_B']) - set(row['Sel_A'])), axis=1)
#     # df_compare['Diff_Count'] = df_compare['Only_in_A'].apply(len) + df_compare['Only_in_B'].apply(len)

#     df_compare.columns = ['Sel_A', 'Sel_B']

#     # Calcolo differenze
#     df_compare['In_Common'] = df_compare.apply(lambda row: sorted(set(row['Sel_A']) & set(row['Sel_B'])), axis=1)
#     df_compare['Only_in_A'] = df_compare.apply(lambda row: sorted(set(row['Sel_A']) - set(row['Sel_B'])), axis=1)
#     df_compare['Only_in_B'] = df_compare.apply(lambda row: sorted(set(row['Sel_B']) - set(row['Sel_A'])), axis=1)
#     df_compare['Diff_Count'] = df_compare.apply(lambda row: len(set(row['Sel_A']) ^ set(row['Sel_B'])), axis=1)

#     # Grafico a linee con seaborn
#     plt.figure(figsize=(10, 4))
#     sns.lineplot(data=df_compare['Diff_Count'], marker='o')
#     plt.title(f"Differenze tra selezioni ({column})")
#     plt.xlabel("Data di Ribilanciamento")
#     plt.ylabel("N. Tickers Diversi")
#     plt.xticks(rotation=45)
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()

#     # Grafico Plotly a barre sovrapposte
#     fig = go.Figure()
#     fig.add_trace(go.Bar(
#         x=df_compare.index,
#         y=df_compare['Sel_A'].apply(len),
#         name='Selezione A',
#         marker=dict(color='blue'),
#         opacity=0.6
#     ))
#     fig.add_trace(go.Bar(
#         x=df_compare.index,
#         y=df_compare['Sel_B'].apply(len),
#         name='Selezione B',
#         marker=dict(color='red'),
#         opacity=0.6
#     ))

#     fig.update_layout(
#         title=f"Numero di Tickers selezionati per Data ({column})",
#         barmode='overlay',
#         xaxis_title="Data di Ribilanciamento",
#         yaxis_title="Numero Tickers",
#         width=900,
#         height=400
#     )

#     from IPython.display import display
#     display(fig)

#     import ace_tools_open as tools
#     tools.display_dataframe_to_user(name=f"Confronto {column}", dataframe=df_compare)

#     return df_compare


def find_min_positive_period(returns: pd.Series) -> int:
    """
    Data una Series di rendimenti (ad esempio portfolio.returns()),
    determina il periodo minimo L (numero di periodi consecutivi) tale che 
    per ogni finestra mobile di lunghezza L il rendimento cumulativo sia > 0.

    Il rendimento cumulativo di una finestra si calcola come:
        cumulative_return = (1 + r1) * (1 + r2) * ... * (1 + rL) - 1

    Parametri:
    -----------
    returns : pd.Series
        Serie di rendimenti periodici (es. giornalieri) con indice DateTime.

    Ritorna:
    --------
    L_min : int
        Numero minimo di periodi richiesto affinché ogni finestra mobile
        di quella lunghezza abbia rendimento cumulativo positivo.
        Se non esiste alcun periodo che soddisfi la condizione, ritorna None.
    """
    # Scorriamo i possibili periodi da 1 fino alla lunghezza della serie
    for L in range(1, len(returns) + 1):
        # Calcola il rendimento cumulativo per ogni finestra di lunghezza L
        rolling_cum = returns.rolling(window=L).apply(lambda r: np.prod(1 + r) - 1, raw=True).dropna()
        # Se per ogni finestra il rendimento cumulativo è > 0, abbiamo trovato il minimo L
        if (rolling_cum > 0).all():
            return L
    return None


def my_display(data: pd.DataFrame,
               title: str = ""):
    tools.display_dataframe_to_user(name=title,dataframe=data)


def extract_tickers_from_wikipedia(
    index: str,
    exclude: Iterable[str] | None = None,
    rename: Mapping[str, str] | None = None,
) -> List[str]:
    """
    Estrae i ticker per l'indice specificato usando richieste HTTP con User-Agent,
    auto-rilevamento della tabella e normalizzazione/suffissi dove necessario.

    NEW:
    - exclude: lista/iterabile di ticker da escludere (case-insensitive).
    - rename : mapping {old_ticker: new_ticker} applicato DOPO il postprocess.

    Parametri
    ---------
    index : str
        'sp100','sp500','nasdaq100','ftsemib','dax','eurostoxx50','cac40','ibex35',
        'nikkei','sse50','hangseng','nifty50','kospi200'
    exclude : Iterable[str] | None
        Tickers da escludere dal risultato finale.
    rename : Mapping[str, str] | None
        Rinomina ticker, es. {"BRK.B": "BRK-B"}.

    Ritorna
    -------
    List[str]
        Lista di ticker (Yahoo-style), filtrata e rinominata.
    """
    # --- Config locale (niente globali) ---
    urls = {
        'sp100': "https://en.wikipedia.org/wiki/S%26P_100",
        'sp500': "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        'nasdaq100': "https://en.wikipedia.org/wiki/Nasdaq-100",
        'ftsemib': "https://en.wikipedia.org/wiki/FTSE_MIB",
        'dax': "https://en.wikipedia.org/wiki/DAX",
        'eurostoxx50': "https://en.wikipedia.org/wiki/EURO_STOXX_50",
        'cac40': "https://en.wikipedia.org/wiki/CAC_40",
        'ibex35': "https://en.wikipedia.org/wiki/IBEX_35",
        'nikkei': "https://www.tradingview.com/symbols/TVC-NI225/components",
        'sse50': "https://en.wikipedia.org/wiki/SSE_50_Index",
        'hangseng': "https://en.wikipedia.org/wiki/Hang_Seng_Index",
        'nifty50': "https://en.wikipedia.org/wiki/NIFTY_50",
        'kospi200': "https://en.wikipedia.org/wiki/KOSPI_200",
    }
    candidate_columns = {
        'sp100': ['Symbol', 'Ticker', 'Code'],
        'sp500': ['Symbol', 'Ticker'],
        'nasdaq100': ['Ticker', 'Symbol'],
        'ftsemib': ['Ticker', 'Symbol'],
        'dax': ['Ticker', 'Symbol'],
        'eurostoxx50': ['Ticker', 'Symbol'],
        'cac40': ['Ticker', 'Symbol'],
        'ibex35': ['Ticker', 'Symbol'],
        'nikkei': ['Symbol', 'Code'],
        'sse50': ['Ticker symbol', 'Symbol'],
        'hangseng': ['Ticker', 'Symbol', 'Code'],
        'nifty50': ['Symbol', 'Ticker'],
        'kospi200': ['Symbol', 'Ticker'],
    }
    UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
          "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")

    # --- exclude / rename normalizzati ---
    exclude_set = set()
    if exclude is not None:
        for x in exclude:
            if x is None:
                continue
            s = str(x).strip().upper()
            if s:
                exclude_set.add(s)

    rename_map = {}
    if rename is not None:
        for k, v in rename.items():
            if k is None or v is None:
                continue
            kk = str(k).strip().upper()
            vv = str(v).strip().upper()
            if kk and vv:
                rename_map[kk] = vv

    # --- Helper interni ---
    def http_get(url: str, tries: int = 3, sleep_s: float = 1.0) -> str:
        sess = requests.Session()
        headers = {'User-Agent': UA, 'Accept-Language': 'en-US,en;q=0.9'}
        last_err = None
        for i in range(tries):
            try:
                render_url = url
                if "wikipedia.org" in url:
                    render_url = url + ("?action=render" if "?" not in url else "&action=render")
                r = sess.get(render_url, headers=headers, timeout=20)
                r.raise_for_status()
                return r.text
            except Exception as e:
                last_err = e
                time.sleep(sleep_s * (i + 1))
        raise last_err

    def find_table_with_columns(html: str, cand_cols: List[str]) -> pd.DataFrame:
        from io import StringIO
        tables = pd.read_html(StringIO(html), flavor='lxml')
        low_cands = [c.lower() for c in cand_cols]
        for df in tables:
            cols = [str(c).strip() for c in df.columns]
            cols_low = [c.lower() for c in cols]
            for c in low_cands:
                if c in cols_low:
                    return df
        return tables[0] if tables else None  # type: ignore

    def resolve_column_name(df: pd.DataFrame, cand_cols: List[str]) -> str:
        cols_map = {str(c).strip().lower(): c for c in df.columns}
        for c in cand_cols:
            key = c.lower()
            if key in cols_map:
                return cols_map[key]
        return df.columns[0]

    def clean_symbols(raw_list: List[str]) -> List[str]:
        out = []
        for t in raw_list:
            t = re.sub(r"\[[^\]]*\]", "", str(t))
            t = t.strip().upper()
            if t:
                out.append(t)
        seen, dedup = set(), []
        for t in out:
            if t not in seen:
                seen.add(t); dedup.append(t)
        return dedup

    def postprocess(index_key: str, syms: List[str]) -> List[str]:
        if index_key == 'sse50':
            return [s.replace("SSE:", "").replace("SSE", "").strip() + ".SS" for s in syms]
        if index_key == 'hangseng':
            out = []
            for s in syms:
                s = s.replace("SEHK:", "").strip()
                if s.isdigit() and len(s) <= 5:
                    s = s.zfill(4)
                out.append(s if s.endswith(".HK") else s + ".HK")
            return out
        if index_key == 'nifty50':
            return [s if s.endswith(".NS") else s + ".NS" for s in syms]
        if index_key == 'kospi200':
            return [s if s.endswith(".KS") else s + ".KS" for s in syms]
        if index_key == 'nikkei':
            out = []
            for s in syms:
                code = s[:4]
                if code.isdigit():
                    out.append(f"{code}.T")
            if not out:
                for s in syms:
                    m = re.search(r"\b(\d{4})\b", s)
                    if m:
                        out.append(f"{m.group(1)}.T")
            seen, clean = set(), []
            for x in out:
                x = x.strip().upper()
                if x and x not in seen:
                    seen.add(x); clean.append(x)
            return clean
        return syms

    def apply_rename_and_exclude(syms: List[str]) -> List[str]:
        out = []
        for s in syms:
            su = str(s).strip().upper()
            su = rename_map.get(su, su)
            if su and su not in exclude_set:
                out.append(su)
        return out

    # --- Corpo funzione ---
    if index not in urls:
        raise ValueError(f"Indice non supportato. Scegli tra: {', '.join(sorted(urls.keys()))}")

    try:
        html = http_get(urls[index])
        df = find_table_with_columns(html, candidate_columns.get(index, []))
        if df is None or df.empty:
            return []

        col = resolve_column_name(df, candidate_columns.get(index, []))
        raw = df[col].dropna().astype(str).tolist()

        syms = clean_symbols(raw)
        syms = postprocess(index, syms)
        syms = apply_rename_and_exclude(syms)

        return list(dict.fromkeys(syms))
    except Exception as e:
        print(f"Errore nell'estrazione del ticker per {index}: {e}")
        return []

## Funzioni di grafica

In [ ]:
###############################################################################
# Grafica: funzioni di plot
###############################################################################
def plot_multiple_portfolios(
    portfolios: dict[str, "pd.Series"],
    title: str = None,
    benchmark: str = None,
    benchmark_data: "pd.Series" = None,
    start_date: str = None,
    end_date: str = None,
    base: float = 1.0,   # 1.0=rebased to 1, 0.0=cum return, 100=base 100
) -> "go.Figure":
    """
    Confronta più portafogli e, opzionalmente, un benchmark.

    FIX principali rispetto alla versione precedente:
    - NON droppa i NaN dei returns in input (evita di spostare t0 in avanti)
    - costruisce un indice comune e allinea TUTTE le serie su tale indice
    - impone r[t0] = 0.0 per far partire l'equity-index esattamente dal punto base
    - sceglie t0 coerente con start_date se fornita (così la curva finale combacia con le stats)
    """
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    from functools import reduce

    # -------------------------
    # 0) Helper: normalize index
    # -------------------------
    def _normalize_dt_index(s: pd.Series) -> pd.Series:
        s = s.copy()
        try:
            s.index = pd.DatetimeIndex(s.index)
        except Exception:
            s.index = pd.to_datetime(s.index, errors="coerce")
        try:
            # rimuovi tz se presente
            if getattr(s.index, "tz", None) is not None:
                s.index = s.index.tz_localize(None)
        except Exception:
            pass
        s.index = pd.DatetimeIndex(s.index).normalize()
        return s

    # -------------------------
    # 1) Pulisci i returns dei portafogli (SENZA dropna)
    # -------------------------
    clean: dict[str, pd.Series] = {}
    for name, ret in (portfolios or {}).items():
        if ret is None or getattr(ret, "empty", True):
            continue

        r = ret.copy()

        # se arriva DataFrame a 1 colonna, schiaccia
        if isinstance(r, pd.DataFrame):
            if r.shape[1] == 1:
                r = r.iloc[:, 0]
            else:
                raise ValueError(f"Portafoglio '{name}': returns devono essere pd.Series (non DataFrame multi-colonna)")

        # numeric + pulizia inf
        r = pd.to_numeric(r, errors="coerce").replace([np.inf, -np.inf], np.nan)

        # normalize index + slice
        r = _normalize_dt_index(r)
        if start_date or end_date:
            r = r.loc[start_date:end_date]

        # NON dropna: il primo NaN è normale nei returns
        if not r.empty:
            clean[name] = r

    if not clean:
        raise ValueError("Nessun portafoglio valido da plottare")

    # -------------------------
    # 2) Indice comune solo fra portafogli (intersection)
    # -------------------------
    idxs = [r.index for r in clean.values()]
    port_common_idx = reduce(lambda a, b: a.intersection(b), idxs).sort_values()

    if port_common_idx.empty:
        raise ValueError("Nessuna data comune fra i portafogli")

    # -------------------------
    # 2b) t0 coerente con start_date se fornita
    # -------------------------
    if start_date is not None:
        t0_req = pd.Timestamp(start_date)
        try:
            if getattr(t0_req, "tzinfo", None) is not None:
                t0_req = t0_req.tz_localize(None)
        except Exception:
            pass
        t0_req = pd.Timestamp(t0_req).normalize()

        # prima data disponibile >= richiesta
        valid = port_common_idx[port_common_idx >= t0_req]
        if valid.empty:
            raise ValueError("start_date oltre l'ultima data comune fra i portafogli")
        t0 = valid[0]
    else:
        t0 = port_common_idx[0]

    # -------------------------
    # 3) Carica e riallinea il benchmark (se richiesto)
    # -------------------------
    bench_ret = None
    if benchmark_data is not None:
        br = benchmark_data.copy()

        if isinstance(br, pd.DataFrame):
            if br.shape[1] == 1:
                br = br.iloc[:, 0]
            else:
                raise ValueError("benchmark_data deve essere pd.Series (non DataFrame multi-colonna)")

        br = pd.to_numeric(br, errors="coerce").replace([np.inf, -np.inf], np.nan)
        br = _normalize_dt_index(br)

        # se benchmark_data è PRICE, trasformo in returns; se è già returns, l'utente deve passarli coerenti.
        # Qui mantengo la logica originale: assumo PRICE.
        br = br.pct_change()

        br_aligned = br.reindex(port_common_idx, method="ffill")
        br_aligned = br_aligned.fillna(0.0)
        bench_ret = br_aligned.rename("Benchmark")

    elif benchmark:
        df_bench = download_data(benchmark, start_date, end_date)  # assume esista nel tuo contesto
        br = df_bench.copy()
        if isinstance(br, pd.DataFrame):
            # prova a scegliere la colonna più probabile
            if "Close" in br.columns:
                br = br["Close"]
            else:
                br = br.iloc[:, 0]
        br = pd.to_numeric(br, errors="coerce").replace([np.inf, -np.inf], np.nan)
        br = _normalize_dt_index(br)
        br = br.pct_change()

        br_aligned = br.reindex(port_common_idx, method="ffill")
        br_aligned = br_aligned.fillna(0.0)
        bench_ret = br_aligned.rename(benchmark)

    # -------------------------
    # 4) Costruisci figura
    # -------------------------
    fig = go.Figure()

    # -------------------------
    # 4b) Helper: curva rebased coerente con t0 (e con Total Return)
    # -------------------------
    def _rebased_curve_from_returns(r: pd.Series) -> pd.Series:
        # allinea PRIMA
        r = r.reindex(port_common_idx)

        # forza r[t0] = 0 per partire dal base
        if t0 in r.index:
            r.loc[t0] = 0.0

        # buchi -> 0%
        r = r.fillna(0.0)

        # equity index
        eq = (1.0 + r).cumprod()

        # rebased a 1 su t0
        eq0 = eq.loc[t0]
        if eq0 == 0 or np.isnan(eq0):
            eq0 = 1.0
        reb1 = eq / eq0

        # convert to requested base
        if base == 1.0:
            y = reb1
        elif base == 0.0:
            y = reb1 - 1.0
        else:
            y = reb1 * float(base)

        return y

    # -------------------------
    # 5) Aggiungi portafogli
    # -------------------------
    for name, r in clean.items():
        y = _rebased_curve_from_returns(r)
        fig.add_trace(go.Scatter(
            x=y.index, y=y.values,
            mode="lines", name=f"Portfolio ({name})"
        ))

    # -------------------------
    # 6) Aggiungi benchmark
    # -------------------------
    if bench_ret is not None:
        yb = _rebased_curve_from_returns(bench_ret)
        fig.add_trace(go.Scatter(
            x=yb.index, y=yb.values,
            mode="lines",
            name=f"Benchmark ({benchmark})" if benchmark else "Benchmark",
            line=dict(color="silver"),
            opacity=0.8
        ))

    # -------------------------
    # 7) Linea base coerente
    # -------------------------
    hline_y = 1.0 if base == 1.0 else (0.0 if base == 0.0 else float(base))
    fig.add_hline(
        y=hline_y,
        line=dict(color="red", width=2, dash="dash"),
    )

    # -------------------------
    # 8) Asse Y: formattazione coerente con base
    # -------------------------
    if base == 0.0:
        y_title = "Cumulative Return (%)"
        fig.update_yaxes(tickformat=".1%")
    elif base == 1.0:
        y_title = "Cumulative Return (rebased to 1)"
        fig.update_yaxes(tickformat=".3f")
    else:
        y_title = f"Index (base {base:g})"
        fig.update_yaxes(tickformat=".1f" if float(base) % 1 else ".0f")

    fig.update_layout(
        title=title or ("Performance vs Benchmark" if bench_ret is not None else "Performance Portafogli"),
        xaxis_title="Data",
        yaxis_title=y_title,
        hovermode="x unified",
        height=600,
        xaxis=dict(
            rangeselector=dict(buttons=[
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(step="all", label="All")
            ])
        ),
        legend=dict(
            orientation="v",
            x=1.02,
            y=1,
            xanchor="left",
            yanchor="auto"
        ),
        margin=dict(t=100),
        template="plotly_white"
    )

    return fig
    
    
def plot_multiple_portfolios_R1(
    portfolios: dict[str, pd.Series],
    title: str = None,
    benchmark: str = None,
    benchmark_data: pd.Series = None,
    start_date: str = None,
    end_date: str = None,
    base: float = 1.0,   # <<< NEW: valore base (default 1.0)
) -> go.Figure:
    """
    Confronta più portafogli e, opzionalmente, un benchmark.
    Le curve dei portafogli sono ribasate al primo giorno comune fra tutti i portafogli.

    base:
      - 1.0   => "1 unità investita" (default storico)
      - 0.0   => cumulative return (0 = 0%)
      - 100.0 => indice base 100
    """
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    from functools import reduce

    # 1) Pulisci i returns dei portafogli
    clean = {}
    for name, ret in portfolios.items():
        if ret is None or getattr(ret, "empty", True):
            continue
        # r = ret.dropna().copy()
        r = ret.copy()
        r = r.replace([np.inf, -np.inf], np.nan)
        # NON droppare: il primo NaN dei returns è normale e serve per l'allineamento
        try:
            r.index = r.index.tz_localize(None)
        except Exception:
            pass
        r.index = r.index.normalize()
        if start_date or end_date:
            r = r.loc[start_date:end_date]
        if not r.empty:
            clean[name] = r

    if not clean:
        raise ValueError("Nessun portafoglio valido da plottare")

    # 2) Indice comune solo fra portafogli
    idxs = [r.index for r in clean.values()]
    port_common_idx = reduce(lambda a, b: a.intersection(b), idxs).sort_values()
    if port_common_idx.empty:
        raise ValueError("Nessuna data comune fra i portafogli")

    
    t0 = port_common_idx[0]
    
    # 3) Carica e riallinea il benchmark (se richiesto)
    bench_ret = None
    if benchmark_data is not None:
        br = benchmark_data.pct_change().dropna()
        try:
            br.index = br.index.tz_localize(None)
        except Exception:
            pass
        br.index = br.index.normalize()
        br_aligned = br.reindex(port_common_idx, method="ffill").fillna(0.0)
        bench_ret = br_aligned.rename("Benchmark")
    elif benchmark:
        df_bench = download_data(benchmark, start_date, end_date)
        br = df_bench.pct_change().dropna()
        try:
            br.index = br.index.tz_localize(None)
        except Exception:
            pass
        br.index = br.index.normalize()
        br_aligned = br.reindex(port_common_idx, method="ffill").fillna(0.0)
        bench_ret = br_aligned.rename(benchmark)

    # 4) Costruisci figura
    fig = go.Figure()

    def _rebased_curve_from_returns(r: pd.Series) -> pd.Series:
        # forza Series
        if isinstance(r, pd.DataFrame):
            if r.shape[1] == 1:
                r = r.iloc[:, 0]
            else:
                raise ValueError("plot_multiple_portfolios: returns devono essere pd.Series")
    
        # allinea PRIMA
        r = r.reindex(port_common_idx)
    
        # il primo return deve essere 0 per far partire l'indice da 1 (o base)
        # (tipicamente è NaN perché non esiste t-1)
        if len(r) > 0:
            r.iloc[0] = 0.0
    
        # buchi -> 0% (coerenza)
        r = r.fillna(0.0)
    
        # equity index (parte da 1)
        eq = (1.0 + r).cumprod()
    
        # converti base
        if base == 1.0:
            y = eq
        elif base == 0.0:
            y = eq - 1.0
        else:
            y = eq * float(base)
    
        return y    
    # def _rebased_curve_from_returns(r: pd.Series) -> pd.Series:
    #     full_cum = (1 + r).cumprod()
    #     # rebased to 1 at t0
    #     reb1 = full_cum / full_cum.loc[t0]
    #     # convert to requested base
    #     if base == 1.0:
    #         y = reb1
    #     elif base == 0.0:
    #         y = reb1 - 1.0
    #     else:
    #         y = reb1 * float(base)
    #     return y.reindex(port_common_idx)

    # 5) Aggiungi portafogli
    for name, r in clean.items():
        y = _rebased_curve_from_returns(r)
        fig.add_trace(go.Scatter(
            x=y.index, y=y.values,
            mode="lines", name=f"Portfolio ({name})"
        ))

    # 6) Aggiungi benchmark
    if bench_ret is not None:
        yb = _rebased_curve_from_returns(bench_ret)
        fig.add_trace(go.Scatter(
            x=yb.index, y=yb.values,
            mode="lines",
            name=f"Benchmark ({benchmark})" if benchmark else "Benchmark",
            line=dict(color="silver"),
            opacity=0.8
        ))

    # --- linea base coerente ---
    hline_y = 1.0 if base == 1.0 else (0.0 if base == 0.0 else float(base))
    fig.add_hline(
        y=hline_y,
        line=dict(color="red", width=2, dash="dash"),
    )

    # --- asse Y: formattazione coerente con base ---
    if base == 0.0:
        y_title = "Cumulative Return (%)"
        fig.update_yaxes(tickformat=".1%")
    elif base == 1.0:
        y_title = "Cumulative Return (rebased to 1)"
        fig.update_yaxes(tickformat=".3f")
    else:
        y_title = f"Index (base {base:g})"
        fig.update_yaxes(tickformat=".1f" if float(base) % 1 else ".0f")

    fig.update_layout(
        title=title or ("Performance vs Benchmark" if bench_ret is not None else "Performance Portafogli"),
        xaxis_title="Data",
        yaxis_title=y_title,
        hovermode="x unified",
        # width=vbt_plot_width,
        height=600,
        xaxis=dict(
            rangeselector=dict(buttons=[
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(step="all", label="All")
            ])
        ),
        legend=dict(
            orientation="v",
            x=1.02,
            y=1,
            xanchor="left",
            yanchor="auto"
        ),
        margin=dict(t=100),
        template="plotly_white"
    )

    return fig

def plot_annual_performance(
    portfolios_returns: dict,
    benchmark: str = None,
    benchmark_data: pd.Series = None,
    title: str = None,
    risk_metric: str = "vol",   # "vol" | "downside" | "maxdd"
    trading_days: int = 252
) -> go.Figure:
    """
    Rendimenti annuali (%) da returns + istogramma rischio annuale.

    Regola CORRETTA (intra-year):
      - per ogni anno, compone SOLO i returns il cui giorno precedente è nello stesso anno
        => esclude il primo return dell'anno (cross-year).
    Nessun allineamento tra serie.

    risk_metric:
      - "vol": volatilità annualizzata su returns intra-year
      - "downside": downside deviation annualizzata (solo returns < 0)
      - "maxdd": max drawdown intra-year (equity costruita da returns intra-year)
    """

    data = portfolios_returns.copy()

    # if title is None:
    #     title = "Rendimenti annuali (%) + Rischio annuale"

    if title is None:
        title = f"Rendimenti annuali (%) + Rischio annuale ({risk_metric})"

    # --- benchmark (se fornito come prezzi) -> returns ---
    bench_name = None
    if benchmark_data is not None:
        bh_ret = benchmark_data.pct_change().dropna()
        bench_name = f"Benchmark ({benchmark})" if benchmark else "Benchmark"
        bh_ret.name = bench_name
        data[bench_name] = bh_ret
    elif benchmark:
        try:
            idx_all = pd.DatetimeIndex(
                np.concatenate([s.index.values for s in portfolios_returns.values()
                                if s is not None and len(s) > 0])
            )
            start = idx_all.min().strftime("%Y-%m-%d")
            end   = (idx_all.max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
            bh_close = download_data(benchmark, start_date=start, end_date=end)
            bh_ret   = bh_close.pct_change().dropna()
            bench_name = benchmark
            bh_ret.name = bench_name
            data[bench_name] = bh_ret
        except Exception:
            pass

    def _clean_returns(x):
        if x is None:
            return pd.Series(dtype=float)

        # DataFrame -> mean(axis=1) (compat originale)
        if isinstance(x, pd.DataFrame):
            s = x.mean(axis=1).dropna()
        else:
            s = x.dropna().copy()

        # normalize index
        try:
            if getattr(s.index, "tz", None) is not None:
                s.index = s.index.tz_localize(None)
        except Exception:
            pass
        s.index = pd.to_datetime(s.index).normalize()

        # sort + drop dup dates
        s = s.sort_index()
        if s.index.duplicated().any():
            s = s[~s.index.duplicated(keep="last")]

        # numeric
        s = pd.to_numeric(s, errors="coerce").dropna()
        return s

    def _intra_year_mask(r: pd.Series) -> np.ndarray:
        """True solo per returns il cui giorno precedente è nello stesso anno."""
        yrs = r.index.year
        prev_yrs = pd.Series(yrs, index=r.index).shift(1)
        return (pd.Series(yrs, index=r.index) == prev_yrs).fillna(False).values

    def _annual_intra_year_return(r: pd.Series) -> pd.Series:
        """Compound per anno escludendo il cross-year return (primo return dell'anno)."""
        if r is None or r.empty:
            return pd.Series(dtype=float)

        mask = _intra_year_mask(r)
        r_intra = r[mask]
        if r_intra.empty:
            return pd.Series(dtype=float)

        ann = r_intra.groupby(r_intra.index.year).apply(lambda seg: (1.0 + seg).prod() - 1.0)
        ann.index = ann.index.astype(int)
        return ann.sort_index()

    def _annual_intra_year_risk(r: pd.Series) -> pd.Series:
        """Rischio per anno su returns intra-year."""
        if r is None or r.empty:
            return pd.Series(dtype=float)

        mask = _intra_year_mask(r)
        r_intra = r[mask]
        if r_intra.empty:
            return pd.Series(dtype=float)

        def _year_risk(seg: pd.Series) -> float:
            seg = seg.dropna()
            if len(seg) < 2:
                return np.nan

            if risk_metric == "vol":
                return float(seg.std(ddof=1) * np.sqrt(trading_days))

            if risk_metric == "downside":
                dn = seg[seg < 0]
                if len(dn) < 2:
                    return 0.0
                return float(dn.std(ddof=1) * np.sqrt(trading_days))

            if risk_metric == "maxdd":
                eq = (1.0 + seg).cumprod()
                dd = (eq / eq.cummax()) - 1.0
                return float(dd.min())  # valore negativo (es: -0.18)

            raise ValueError("risk_metric deve essere: 'vol', 'downside' o 'maxdd'")

        out = r_intra.groupby(r_intra.index.year).apply(_year_risk)
        out.index = out.index.astype(int)
        return out.sort_index()

    # --- calcolo annuale per tutti (portafogli + benchmark) ---
    annual_ret = {}
    annual_risk = {}
    for name, series in data.items():
        r = _clean_returns(series)
        annual_ret[name] = _annual_intra_year_return(r)
        annual_risk[name] = _annual_intra_year_risk(r)

    annual_ret_df = pd.DataFrame(annual_ret).sort_index()
    annual_risk_df = pd.DataFrame(annual_risk).sort_index()

    # --- filtro anni presenti in almeno un PORTAFOGLIO (escludi anni solo benchmark) ---
    port_names = list(portfolios_returns.keys())
    port_years = set()
    for nm in port_names:
        if nm in annual_ret_df.columns:
            port_years |= set(annual_ret_df.index[~annual_ret_df[nm].isna()])
    if port_years:
        years_sorted = sorted(port_years)
        annual_ret_df = annual_ret_df.loc[annual_ret_df.index.isin(years_sorted)]
        annual_risk_df = annual_risk_df.loc[annual_risk_df.index.isin(years_sorted)]

    # --- labels rischio ---
    if risk_metric == "vol":
        risk_title = f"Risk (Vol ann., √{trading_days})"
        risk_text_fmt = lambda v: f"{v*100:.2f}%" if pd.notna(v) else ""
    elif risk_metric == "downside":
        risk_title = f"Risk (Downside ann., √{trading_days})"
        risk_text_fmt = lambda v: f"{v*100:.2f}%" if pd.notna(v) else ""
    else:  # maxdd
        risk_title = "Risk (Max Drawdown intra-year)"
        risk_text_fmt = lambda v: f"{v*100:.2f}%" if pd.notna(v) else ""

    # --- plot (2 pannelli) ---
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.12,
        subplot_titles=("Annual Return", risk_title),
    )

    # pannello 1: return
    for col in annual_ret_df.columns:
        y = annual_ret_df[col]
        fig.add_trace(
            go.Bar(
                x=annual_ret_df.index.astype(str),
                y=y,
                name=col if col != bench_name else f"{col}",
                text=(y * 100).round(2).astype(str) + "%",
                textposition="outside",
                opacity=1.0 if col not in [bench_name] else 0.6
            ),
            row=1, col=1
        )

    # pannello 2: risk
    for col in annual_risk_df.columns:
        y = annual_risk_df[col]
        fig.add_trace(
            go.Bar(
                x=annual_risk_df.index.astype(str),
                y=y,
                name=col if col != bench_name else f"{col}",
                text=[risk_text_fmt(v) for v in y.values],
                textposition="outside",
                opacity=1.0 if col not in [bench_name] else 0.6,
                showlegend=False  # evita doppia legenda (già sopra)
            ),
            row=2, col=1
        )
    # if title is None:
    #     title = f"Rendimenti annuali (%) + Rischio annuale ({risk_title})"

    fig.update_layout(
        barmode="group",
        title=title,
        template="plotly_white",
        margin=dict(t=120),
        height=900,
    )
    fig.update_yaxes(title_text="Return", row=1, col=1)
    fig.update_yaxes(title_text="Risk", row=2, col=1)
    fig.update_xaxes(title_text="Year", row=2, col=1)

    return fig

    

def plot_monthly_returns_histogram(
    pf,
    title: str = "Istogramma dei rendimenti mensili",
    top_n: int = 3,
    width: int = 1200,
    height: int = 640,
    *,
    panel_width: float = 0.36,   # larghezza pannello destro (0..1)
    gap: float = 0.02,           # gap tra istogramma e pannello
    hist_fill: float = 0.96,     # frazione dello spazio sinistro occupata dall’istogramma
    left_margin_px: int = 28,    # margine sinistro per label Y
):
    """
    Istogramma dei rendimenti mensili (Plotly) + pannello destro.
    Calcolo MENSILE = intra-month compounding sui returns del portafoglio:
        m = prod(1 + r_intra_month) - 1
    dove r_intra_month esclude SEMPRE il primo return del mese (carry-over dal mese precedente).
    """
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go

    # ------------------------------------------------------------
    # 1) Daily returns ROBUSTI dal value totale (coerente con core)
    # ------------------------------------------------------------
    v = pf.value()
    if isinstance(v, pd.DataFrame):
        v = v.sum(axis=1)
    v = v.dropna().copy()

    try:
        if v.index.tz is not None:
            v.index = v.index.tz_localize(None)
    except Exception:
        pass

    v.index = pd.to_datetime(v.index).normalize()
    v = v.sort_index()
    if v.index.duplicated().any():
        v = v[~v.index.duplicated(keep="last")]

    r = v.pct_change().dropna()
    if r.empty:
        raise ValueError("Nessun rendimento disponibile.")

    # ------------------------------------------------------------
    # 2) Monthly returns INTRA-MONTH (escludi primo return del mese)
    # ------------------------------------------------------------
    monthly_list = []
    for (y, m), rm in r.groupby([r.index.year, r.index.month]):
        # rm contiene TUTTI i daily returns nel mese, incluso il primo (carry-over).
        if len(rm) >= 2:
            rm_intra = rm.iloc[1:]
            ret_m = (1.0 + rm_intra).prod() - 1.0
            monthly_list.append((y, m, float(ret_m)))
        else:
            monthly_list.append((y, m, np.nan))

    monthly = pd.Series(
        [x[2] for x in monthly_list],
        index=pd.PeriodIndex([f"{x[0]}-{x[1]:02d}" for x in monthly_list], freq="M"),
        dtype=float
    ).dropna()

    if monthly.empty:
        raise ValueError("Nessun rendimento mensile disponibile.")

    mean_monthly = float(monthly.mean())

    # ------------------------------------------------------------
    # 3) Statistiche e top/bottom
    # ------------------------------------------------------------
    monthly_named = monthly.copy()
    monthly_named.index = monthly_named.index.astype(str)  # "YYYY-MM"

    years = pd.Index([int(s.split("-")[0]) for s in monthly_named.index])
    start_year, end_year = int(years.min()), int(years.max())

    total_months = int(monthly.shape[0])
    up_months = int((monthly > 0).sum())
    pct_up = 100.0 * up_months / total_months if total_months else np.nan

    top_s = monthly_named.sort_values(ascending=False).head(top_n)
    bot_s = monthly_named.sort_values(ascending=True).head(top_n)

    # bins simmetrici (1%)
    max_abs = float(np.ceil(np.max(np.abs(monthly.values)) * 100.0))
    max_abs = max(max_abs, 6.0)
    xbins = dict(start=-max_abs/100.0, end=max_abs/100.0, size=0.01)

    x_pos = monthly[monthly >= 0].values
    x_neg = monthly[monthly < 0].values

    # ------------------------------------------------------------
    # 4) Istogramma
    # ------------------------------------------------------------
    fig = go.Figure()
    fig.add_histogram(
        x=x_pos, xbins=xbins, marker=dict(color="rgb(34,139,34)"),
        hovertemplate="Mesi: %{y}<extra></extra>", showlegend=False
    )
    fig.add_histogram(
        x=x_neg, xbins=xbins, marker=dict(color="rgb(203,67,53)"),
        hovertemplate="Mesi: %{y}<extra></extra>", showlegend=False
    )
    fig.add_vline(x=0.0, line_width=2, line_dash="solid", line_color="rgba(80,80,80,0.6)")

    fig.update_xaxes(
        tickformat=".0%", title_text="Rendimento mensile (intra-month)",
        zeroline=False, range=[xbins["start"], xbins["end"]],
        showgrid=True, gridcolor="rgba(0,0,0,0.06)"
    )
    fig.update_yaxes(
        title_text="Numero di mesi",
        rangemode="tozero", showgrid=True, gridcolor="rgba(0,0,0,0.06)"
    )
    fig.update_layout(bargap=0.05)

    # ------------------------------------------------------------
    # 5) Pannello destro (paper coords)
    # ------------------------------------------------------------
    GREEN_BG, GREEN_ACC = "#e8f5e9", "#2e7d32"
    RED_BG, RED_ACC = "#fdecea", "#c62828"
    GRAY = "#6b7280"
    IT_MONTHS = ["gennaio","febbraio","marzo","aprile","maggio","giugno",
                 "luglio","agosto","settembre","ottobre","novembre","dicembre"]

    def pct_str(x, d=1): return f"{x*100:.{d}f}%".replace(".", ",")

    def fmt_month(ym: str):
        # ym: "YYYY-MM"
        y, m = ym.split("-")
        return f"{IT_MONTHS[int(m)-1]} {y}"

    left_space = 1.0 - panel_width - gap
    hist_width = max(0.1, left_space * float(hist_fill))
    hist_left = (left_space - hist_width) / 2.0
    hist_right = hist_left + hist_width
    fig.update_xaxes(domain=[hist_left, hist_right])

    px0 = hist_right + gap
    px1 = px0 + panel_width

    def rect(y0, y1, color):
        fig.add_shape(
            type="rect", xref="paper", yref="paper",
            x0=px0, x1=px1, y0=y0, y1=y1,
            fillcolor=color, line=dict(color="rgba(0,0,0,0)")
        )

    def ann(x, y, text, size=14, color="#111", bold=False, anchor="left"):
        fig.add_annotation(
            xref="paper", yref="paper", x=x, y=y,
            xanchor=anchor, yanchor="middle",
            text=(f"<b>{text}</b>" if bold else text),
            showarrow=False, font=dict(size=size, color=color), align="left"
        )

    # riepilogo + media mensile
    rect(0.74, 0.96, GREEN_BG)
    ann(px0+0.02, 0.92, "Il portafoglio ha avuto un rendimento positivo", size=16, color=GREEN_ACC, bold=True)
    ann(
        px0+0.02, 0.86,
        f"durante <b>{up_months}</b> dei <b>{total_months}</b> mesi (<b>{int(round(pct_up))}%</b>) tra il {start_year} e il {end_year}.",
        size=14
    )
    ann(px0+0.02, 0.80, f"Media rendimento mensile: <b>{pct_str(mean_monthly, 2)}</b>", size=14, color="#0f5132")

    def triplet(title_txt, series, y0, y1, bg, accent):
        rect(y0, y1, bg)
        title_y = y1 - 0.05
        month_y = title_y - 0.055
        value_y = month_y - 0.045

        ann(px0 + 0.02, title_y, title_txt, size=16, color=accent, bold=True)

        w = (px1 - px0)
        xs = [px0 + w * 0.17, px0 + w * 0.50, px0 + w * 0.83]

        for i, (idx, val) in enumerate(series.items()):
            if i > 2:
                break
            ann(xs[i], month_y, fmt_month(str(idx)), size=12, anchor="center")
            ann(xs[i], value_y, pct_str(float(val), 1), size=16, color=accent, bold=True, anchor="center")

    triplet("I mesi migliori", top_s, y0=0.48, y1=0.70, bg=GREEN_BG, accent=GREEN_ACC)
    triplet("I mesi peggiori", bot_s, y0=0.22, y1=0.44, bg=RED_BG,   accent=RED_ACC)
    ann(px0+0.02, 0.08, "ℹ️ L'istogramma mostra la frequenza dei rendimenti mensili (intra-month).", size=13, color=GRAY)

    fig.update_layout(
        title=title, title_x=0.5,
        width=width, height=height,
        margin=dict(l=left_margin_px, r=54, t=70, b=48),
        plot_bgcolor="white", paper_bgcolor="white",
    )
    return fig


def plot_year_returns_histogram(
    pf,
    title: str = "Istogramma dei rendimenti annuali",
    top_n: int = 3,
    width: int = 1200,
    height: int = 520,
    *,
    panel_width: float = 0.36,
    gap: float = 0.02,
    hist_fill: float = 0.96,
    left_margin_px: int = 28,
    min_years: int = 2
):
    """
    Istogramma dei rendimenti annuali con pannello testuale a destra (Plotly).
    Calcolo ANNUALE = intra-year compounding sui returns del portafoglio:
        ann = prod(1 + r_intra_year) - 1
    dove r_intra_year esclude SEMPRE il primo return dell'anno (carry-over dall'anno precedente).
    Se il portafoglio ha meno di `min_years` anni, ritorna None.
    """
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go

    # --- daily returns ROBUSTI dal value totale ---
    v = pf.value()
    if isinstance(v, pd.DataFrame):
        v = v.sum(axis=1)
    v = v.dropna().copy()

    try:
        if v.index.tz is not None:
            v.index = v.index.tz_localize(None)
    except Exception:
        pass

    v.index = pd.to_datetime(v.index).normalize()
    v = v.sort_index()
    if v.index.duplicated().any():
        v = v[~v.index.duplicated(keep="last")]

    r = v.pct_change().dropna()

    # --- annual returns INTRA-YEAR (escludi primo return dell'anno) ---
    yearly_list = []
    for y in sorted(r.index.year.unique()):
        ry = r[r.index.year == y]
        if len(ry) >= 2:
            ry_intra = ry.iloc[1:]  # esclude carry-over da anno precedente
            ann = (1.0 + ry_intra).prod() - 1.0
            yearly_list.append((y, float(ann)))

    yearly = pd.Series(dict(yearly_list), dtype=float).sort_index()
    if len(yearly) < min_years:
        return None

    mean_yearly = float(yearly.mean())

    # Serie con indice "YYYY" per etichette
    yearly_named = yearly.copy()
    yearly_named.index = yearly_named.index.astype(str)

    start_year, end_year = int(yearly.index.min()), int(yearly.index.max())
    total_years = int(yearly.shape[0])
    up_years = int((yearly > 0).sum())
    pct_up = 100.0 * up_years / total_years if total_years else np.nan

    # Top/Bottom anni
    top_s = yearly_named.sort_values(ascending=False).head(top_n)
    bot_s = yearly_named.sort_values(ascending=True).head(top_n)

    # Bins simmetrici (2% di passo)
    max_abs = float(np.ceil(np.max(np.abs(yearly.values)) * 100.0))
    max_abs = max(max_abs, 10.0)
    step = 2.0
    max_abs = step * np.ceil(max_abs / step)
    xbins = dict(start=-max_abs/100.0, end=max_abs/100.0, size=step/100.0)

    x_pos = yearly[yearly >= 0].values
    x_neg = yearly[yearly < 0].values

    fig = go.Figure()
    fig.add_histogram(
        x=x_pos, xbins=xbins, marker=dict(color="rgb(34,139,34)"),
        hovertemplate="Anni: %{y}<extra></extra>", showlegend=False
    )
    fig.add_histogram(
        x=x_neg, xbins=xbins, marker=dict(color="rgb(203,67,53)"),
        hovertemplate="Anni: %{y}<extra></extra>", showlegend=False
    )
    fig.add_vline(x=0.0, line_width=2, line_dash="solid", line_color="rgba(80,80,80,0.6)")

    fig.update_xaxes(
        tickformat=".0%", title_text="Rendimento annuale",
        zeroline=False, range=[xbins["start"], xbins["end"]],
        showgrid=True, gridcolor="rgba(0,0,0,0.06)"
    )
    fig.update_yaxes(
        title_text="Numero di anni",
        rangemode="tozero", showgrid=True, gridcolor="rgba(0,0,0,0.06)"
    )
    fig.update_layout(bargap=0.05)

    GREEN_BG, GREEN_ACC = "#e8f5e9", "#2e7d32"
    RED_BG, RED_ACC = "#fdecea", "#c62828"
    GRAY = "#6b7280"

    def pct_str(x, d=1): return f"{x*100:.{d}f}%".replace(".", ",")

    left_space = 1.0 - panel_width - gap
    hist_width = max(0.1, left_space * float(hist_fill))
    hist_left = (left_space - hist_width) / 2.0
    hist_right = hist_left + hist_width
    fig.update_xaxes(domain=[hist_left, hist_right])

    px0 = hist_right + gap
    px1 = px0 + panel_width

    def rect(y0, y1, color):
        fig.add_shape(type="rect", xref="paper", yref="paper",
                      x0=px0, x1=px1, y0=y0, y1=y1,
                      fillcolor=color, line=dict(color="rgba(0,0,0,0)"))

    def ann(x, y, text, size=14, color="#111", bold=False, anchor="left"):
        fig.add_annotation(xref="paper", yref="paper", x=x, y=y,
                           xanchor=anchor, yanchor="middle",
                           text=(f"<b>{text}</b>" if bold else text),
                           showarrow=False, font=dict(size=size, color=color), align="left")

    rect(0.68, 0.94, GREEN_BG)
    ann(px0+0.02, 0.90, "Il portafoglio ha avuto un rendimento positivo", size=16, color=GREEN_ACC, bold=True)
    ann(px0+0.02, 0.84, f"durante <b>{up_years}</b> dei <b>{total_years}</b> anni (<b>{int(round(pct_up))}%</b>) tra il {start_year} e il {end_year}.", size=14)
    ann(px0+0.02, 0.78, f"Media rendimento annuo: <b>{pct_str(mean_yearly, 2)}</b>", size=14, color="#0f5132")

    def triplet(title, series, y0, y1, bg, accent):
        rect(y0, y1, bg)
        title_y = y1 - 0.05
        year_y  = title_y - 0.055
        value_y = year_y  - 0.045
        ann(px0 + 0.02, title_y, title, size=16, color=accent, bold=True)
        w = (px1 - px0)
        xs = [px0 + w*0.17, px0 + w*0.50, px0 + w*0.83]
        for i, (idx, val) in enumerate(series.items()):
            if i > 2: break
            ann(xs[i], year_y, str(idx), size=12, anchor="center")
            ann(xs[i], value_y, pct_str(val, 1), size=16, color=accent, bold=True, anchor="center")

    triplet("Gli anni migliori", top_s, y0=0.42, y1=0.64, bg=GREEN_BG, accent=GREEN_ACC)
    triplet("Gli anni peggiori", bot_s, y0=0.18, y1=0.40, bg=RED_BG,   accent=RED_ACC)

    ann(px0+0.02, 0.08, "ℹ️ L'istogramma mostra la frequenza dei rendimenti annuali (intra-year).", size=13, color=GRAY)

    fig.update_layout(
        title=title, title_x=0.5,
        width=width, height=height,
        margin=dict(l=left_margin_px, r=54, t=64, b=48),
        plot_bgcolor="white", paper_bgcolor="white",
    )
    return fig

    
def plot_monthly_returns(
    pf,
    eoy: bool = True,
    title: str = "Monthly Returns (%)",
    width: int | None = None,
    height: int | None = None,
    auto_height: bool = True,
    cell_h: int = 28,
    min_h: int = 300,
    max_h: int = 900
) -> go.Figure:
    """
    Heatmap 'anno x mese' dei rendimenti mensili (in %) del portafoglio.

    Calcolo su VALUE (robusto):
    - Mensile (intra-month): last_value_month / first_value_month - 1
    - EOY/YTD (intra-year): last_value_year  / first_value_year  - 1

    Nota: così GEN e EOY sono sempre calcolabili anche senza mese/anno precedente.
    """
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go

    # --- value totale robusto ---
    v = pf.value()
    if isinstance(v, pd.DataFrame):
        v = v.sum(axis=1)
    v = v.dropna().copy()

    try:
        if v.index.tz is not None:
            v.index = v.index.tz_localize(None)
    except Exception:
        pass

    v.index = pd.to_datetime(v.index).normalize()
    v = v.sort_index()
    if v.index.duplicated().any():
        v = v[~v.index.duplicated(keep="last")]

    if v.empty:
        raise ValueError("Nessun valore disponibile (pf.value() vuoto).")

    # --- rendimenti mensili intra-month: last/first - 1 ---
    monthly_rows = []
    for (y, m), vm in v.groupby([v.index.year, v.index.month]):
        if len(vm) >= 2:
            ret_m = float(vm.iloc[-1] / vm.iloc[0] - 1.0)
        else:
            ret_m = np.nan
        monthly_rows.append((y, m, ret_m))

    df_m = pd.DataFrame(monthly_rows, columns=["Year", "Month", "Ret"])
    heat = df_m.pivot_table(index="Year", columns="Month", values="Ret", aggfunc="first")

    month_map = {1:"JAN",2:"FEB",3:"MAR",4:"APR",5:"MAY",6:"JUN",
                 7:"JUL",8:"AUG",9:"SEP",10:"OCT",11:"NOV",12:"DEC"}
    all_months = [1,2,3,4,5,6,7,8,9,10,11,12]
    heat = heat.reindex(columns=all_months)
    heat.columns = [month_map[c] for c in heat.columns]

    # anni presenti nel value
    years_idx = pd.Index(sorted(v.index.year.unique()), name="Year")
    heat = heat.reindex(index=years_idx)

    # --- EOY/YTD intra-year: last/first - 1 ---
    if eoy:
        yrets = []
        for y in years_idx:
            vy = v[v.index.year == y]
            if len(vy) >= 2:
                yret = float(vy.iloc[-1] / vy.iloc[0] - 1.0)
            else:
                yret = np.nan
            yrets.append(yret)
        heat["EOY"] = yrets

    # --- dimensioni figura ---
    n_years = len(heat.index)
    if auto_height and height is None:
        height = int(np.clip(120 + cell_h * max(1, n_years), min_h, max_h))

    z = (heat.values * 100.0).astype(float)
    text = np.where(np.isnan(z), "", np.round(z, 2).astype(object))

    fig = go.Figure(
        data=go.Heatmap(
            z=z,
            x=heat.columns.tolist(),
            y=heat.index.astype(str).tolist(),
            colorscale="RdYlGn",
            zmid=0,
            colorbar_title="%",
            text=text,
            texttemplate="%{text}",
            textfont=dict(size=10 if n_years <= 6 else 9 if n_years <= 12 else 8),
            hovertemplate="Year %{y}<br>%{x}: %{z:.2f}%<extra></extra>"
        )
    )

    fig.update_layout(
        title=title,
        width=width, height=height,
        margin=dict(l=50, r=30, t=60, b=40),
        showlegend=False
    )
    fig.update_yaxes(autorange="reversed")
    return fig


#
# Strategies Rotationals
#

def plot_weights_heatmap(df_weights, title="Allocazioni medie (Walk-Forward)"):
    import seaborn as sns, matplotlib.pyplot as plt
    df_avg = df_weights.mean(axis=1).sort_values(ascending=False)
    plt.figure(figsize=(10, 0.4 * len(df_avg)))
    sns.heatmap(df_avg.to_frame().T, cmap="Blues", annot=True, fmt=".1%", cbar=False)
    plt.title(title); plt.yticks([]); plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.show()

def plot_strategy_pie(df_summary, title="Distribuzione strategie selezionate"):
    import matplotlib.pyplot as plt
    counts = df_summary["Method"].value_counts()
    plt.figure(figsize=(6, 6))
    plt.pie(counts, labels=counts.index, autopct="%1.1f%%", startangle=90)
    plt.title(title); plt.tight_layout(); plt.show()

#
# Momentum Rotationals
#
def plot_ticker_frequencies(
    sel_tickers: 'pd.DataFrame',
    start_date: 'pd.Timestamp | str | None' = None,
    end_date: 'pd.Timestamp | str | None' = None,
    include_prev: bool = True,
    universe: 'Iterable[str] | None' = None,
    title: str = "Frequenza di selezione dei titoli",
    width: int = 1000,
    height: int = 600,
):
    """
    Istogramma Plotly dei ticker più ricorrenti (migliorata).

    Parametri
    ---------
    sel_tickers : pd.DataFrame
        DataFrame indicizzato in datetime con colonna 'Top_Tickers' (liste o stringhe come "AAPL,MSFT").
    start_date, end_date : str|Timestamp|None
        Finestra da considerare (inclusiva). Se None -> usa tutto.
    include_prev : bool
        Se True include l'ultima selezione strettamente precedente a start_date (utile per "set iniziale").
    universe : iterable[str] | None
        Lista/Index/set dell'universo completo dei titoli. Se fornito, la funzione calcola tic
        ker mai selezionati e la percentuale di non-selezionati.
    title, width, height : grafica

    Restituisce
    ----------
    fig : plotly.graph_objects.Figure
        Istogramma interattivo.
    freq_df : pd.DataFrame
        DataFrame con colonne ['Ticker','Frequenza'] ordinato per frequenza decrescente.
    freq_full_df : pd.DataFrame
        Se universe fornito: DataFrame con tutte le tickers dell'universo e la frequenza (0=mai selezionato).
        Se universe None: None.
    unselected_df : pd.DataFrame
        Se universe fornito: DataFrame dei tickers mai selezionati e una riga-sintesi con la percentuale.
        Se universe None: None.
    """
    from collections.abc import Iterable

    # universe += ["ZZZ_TEST_NOT_SELECTED"] # test per la selezione

    # -----------------------
    # Validazioni e copia
    # -----------------------
    if sel_tickers is None or sel_tickers.empty:
        raise ValueError("sel_tickers è vuoto: impossibile calcolare le frequenze.")

    if "Top_Tickers" not in sel_tickers.columns:
        raise KeyError("sel_tickers deve contenere la colonna 'Top_Tickers'.")

    df = sel_tickers.copy()

    # --- normalizza indice datetime ---
    try:
        idx = pd.to_datetime(df.index)
    except Exception as e:
        raise TypeError("sel_tickers.index non è convertibile a datetime.") from e

    # rimuovi tz e normalizza (solo data)
    try:
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_localize(None)
    except Exception:
        pass
    df.index = idx.normalize()
    df = df.sort_index()

    # --- parse date bounds ---
    def _to_ts(x):
        if x is None:
            return None
        t = pd.to_datetime(x)
        try:
            if getattr(t, "tz", None) is not None:
                t = t.tz_localize(None)
        except Exception:
            pass
        return pd.Timestamp(t).normalize()

    s = _to_ts(start_date)
    e = _to_ts(end_date)

    # --- filtro finestra ---
    df_win = df
    if s is not None and e is not None:
        if s > e:
            s, e = e, s
        df_win = df.loc[s:e]
    elif s is not None:
        df_win = df.loc[s:]
    elif e is not None:
        df_win = df.loc[:e]

    # --- include prev selection (ultima riga strettamente prima di s) ---
    if include_prev and s is not None:
        # tutte le righe fino a s (inclusive) -> prendo l'ultima con index < s
        df_before = df.loc[:s]
        # rimuovo eventuale riga con index == s perché vogliamo strettamente precedente
        df_before = df_before.loc[df_before.index < s]
        if not df_before.empty:
            prev_row = df_before.iloc[[-1]]
            df_win = pd.concat([prev_row, df_win], axis=0)
            df_win = df_win[~df_win.index.duplicated(keep="last")].sort_index()

    if df_win.empty:
        raise ValueError(
            f"Nessun dato sel_tickers nella finestra richiesta (start={start_date}, end={end_date})."
        )

    # --- esplodi tickers ---
    exploded = df_win["Top_Tickers"].explode()

    # normalizza eventuali stringhe tipo "AAPL,MSFT" e rimuovi NaN
    def _norm(x):
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return None
        if isinstance(x, str):
            # split su virgole o punto e virgola
            if "," in x or ";" in x:
                parts = [p.strip() for p in re.split(r"[,;]+", x) if p.strip()]
                return parts
            # se stringa singola
            return x.strip()
        # se già lista/iterable (ma non stringa), ritorna come è
        if isinstance(x, Iterable) and not isinstance(x, (str, bytes)):
            return list(x)
        return x

    import re
    exploded = exploded.apply(_norm).explode().dropna().astype(str).str.upper().str.strip()

    # Conta frequenze
    freq_series = exploded.value_counts()
    freq_df = freq_series.reset_index()
    freq_df.columns = ["Ticker", "Frequenza"]

    # ----- se fornisco universe: costruisco freq_full_df e unselected_df -----
    freq_full_df = None
    unselected_df = None
    if universe is not None:
        # normalizza universe in lista di stringhe uppercase
        if isinstance(universe, (pd.Index, list, set, tuple)):
            uni_list = [str(x).upper().strip() for x in list(universe)]
        else:
            # se passato un single stringo separato da virgole
            try:
                uni_list = [p.strip().upper() for p in re.split(r"[,;]+", str(universe)) if p.strip()]
            except Exception:
                raise TypeError("Parametro 'universe' non è un iterable riconosciuto.")
        uni_index = pd.Index(sorted(set(uni_list)), name="Ticker")

        # freq_full: merge universo con freq (assegno 0 ai non presenti)
        freq_full_df = pd.DataFrame(index=uni_index).reset_index()
        freq_full_df = freq_full_df.merge(freq_df, on="Ticker", how="left").fillna({"Frequenza": 0})
        freq_full_df["Frequenza"] = freq_full_df["Frequenza"].astype(int)
        freq_full_df = freq_full_df.sort_values("Frequenza", ascending=False).reset_index(drop=True)

        # tickers non selezionati
        unselected = freq_full_df.loc[freq_full_df["Frequenza"] == 0, "Ticker"].tolist()
        pct_unselected = (len(unselected) / len(uni_index)) if len(uni_index) > 0 else np.nan

        unselected_df = pd.DataFrame({
            "Ticker": unselected
        })
        # aggiungo riga di sintesi (facoltativa, utile per visual)
        summary = pd.DataFrame([{
            "Ticker": "<summary>",
            "Frequenza": len(unselected),
            "Pct_unselected": pct_unselected
        }])
        # non concateno la summary alle ticker list automaticamente; lascio separate ma ritorno il valore
        # per comodità aggiungo la percentuale a freq_full_df
        freq_full_df["Pct_universe"] = (freq_full_df["Frequenza"] > 0).astype(int)  # 1 se selezionato almeno 1 volta
        # Aggiungo percentuale colonna per chiarezza (0/1) non la percentuale reale per ticker
        # Fornisco pct_unselected separatamente come float

    # ----- Istogramma -----
    # Se freq_df è vuoto (improbabile qui), creiamo fig vuota altrimenti grafico bar
    if freq_df.empty:
        fig = px.bar(title=title)
        fig.update_layout(width=width, height=height, template="plotly_white")
    else:
        fig = px.bar(
            freq_df,
            x="Ticker",
            y="Frequenza",
            title=title,
            labels={"Frequenza": "Numero di occorrenze"},
            text="Frequenza",
        )
        fig.update_traces(marker_color="blue", textposition="outside")
        fig.update_layout(xaxis_tickangle=-45, template="plotly_white", width=width, height=height)

    # --- return (compatibilità col vecchio API) ---
    return fig, freq_df, freq_full_df, (unselected_df, (len(unselected) if universe is not None else None),
                                        (pct_unselected if universe is not None else None))

    
def plot_total_return_per_ticker(
    returns,
    title: str = "Rendimenti totali per titolo (%)",
    start_date=None,
    end_date=None,
    label_decimals: int = 2,
    highlight_zero: bool = True 
) -> go.Figure:
    """
    Crea un bar chart orizzontale dei rendimenti totali per ticker usando Plotly,
    con etichette percentuali poste all'esterno delle barre.

    Parameters
    ----------
    returns : pd.Series or dict of pd.Series
        - Se è una pd.Series indicizzata con ticker e valori numerici:
          si assume che siano già i rendimenti totali in %.
        - Se è una pd.Series “object” (mapping ticker → pd.Series):
          calcola il total return da ciascuna serie.
    title : str
        Titolo del grafico.
    label_decimals : int
        Numero di decimali da mostrare sulle etichette.
    highlight_zero : bool
        Se True, colora in grigio i rendimenti pari a 0.

    Returns
    -------
    fig : plotly.graph_objects.Figure
    """
    # Se returns è già una serie di floats, salto il calcolo
    if not (returns.dtype == object and isinstance(returns.iloc[0], (pd.Series, list, np.ndarray))):
        total_return = returns.copy()
    else:
        # Applichiamo filtro temporale e normalizzazione
        if start_date is not None:
            returns = pd.Series(
                {sym: ser[ser.index >= start_date] for sym, ser in returns.items()},
                name='returns',
                dtype=object
            )
        if end_date is not None:
            returns = pd.Series(
                {sym: ser[ser.index <= end_date] for sym, ser in returns.items()},
                name='returns',
                dtype=object
            )

        # calcola il total return per ogni ticker
        total_return = pd.Series(
            {symbol: (1 + series).prod() - 1
             for symbol, series in returns.items()},
            name='total_return'
        )

    # Se era in frazione, lo porto a percentuale
    if total_return.abs().max() <= 1:
        total_return = total_return * 100

    # Ordina i rendimenti (escludendo gli zero, se ci sono)
    perf_sorted = total_return[total_return != 0].sort_values()

    # Determina colori
    def get_color(v):
        if v > 0:
            return "green"
        elif v < 0:
            return "red"
        else:
            return "lightgray" if highlight_zero else "black"

    colors = [get_color(v) for v in perf_sorted.values]

    # Prepara le etichette di testo
    texts = [f"{v:.{label_decimals}f}%" for v in perf_sorted.values]

    # Calcola dinamicamente altezza in pixel (min 400px, ~30px per ticker)
    height = max(400, len(perf_sorted) * 30)

    # Estendi i limiti dell'asse X di ±5 punti
    x_min = perf_sorted.min() - 5
    x_max = perf_sorted.max() + 5

    tickers = perf_sorted.index
    values = perf_sorted.values

    # Recupera i nomi aziendali, se disponibili
    company_data = build_company_df_with_cache(tickers)
    def truncate(s, n):
        return s if len(s) <= n else '…' + s[-n:]

    n=70
    company_data['Company'] = company_data['Company'].apply(lambda s: truncate(s, n))

    labels = [
        f"{company_data.loc[t, 'Company'] if t in company_data.index else 'N/D'} ({t})"
        for t in tickers
    ]

    # Costruisci la figura
    fig = go.Figure(go.Bar(
        x=values,
        y=labels,
        orientation='h',
        marker_color=colors,
        text=texts,
        textposition='outside',
        hovertemplate='%{y}: %{x:.2f}%<extra></extra>'
    ))
    
    # Rimuovo il titolo y tradizionale e imposto solo l’asse x
    fig.update_layout(
        title=title,
        xaxis_title="Rendimento (%)",
        yaxis_title=None,
        xaxis=dict(range=[x_min, x_max], showgrid=True, gridcolor='lightgray'),
        margin=dict(l=120, r=40, t=80, b=40),
        height=height
    )
    
    # Mantengo l’ordine delle categorie sull’asse y
    fig.update_yaxes(
        categoryorder='array',
        categoryarray=list(perf_sorted.index),
        showticklabels=True
    )
    
    # Aggiungo l’annotazione in alto a sinistra dentro l’area del plot (paper coords)
    fig.add_annotation(
        xref='paper', yref='paper',
        x=0, y=1.02,               # 0% da sinistra, 102% in alto (leggermente sopra)
        xanchor='left',
        text="Companies (ticker)",
        showarrow=False,
        font=dict(size=12)
    )


    return fig

def _pf_from_equity_curve(eq, *, init_cash=100_000):
    """
    Converte una equity line (Series) in un vbt.Portfolio
    usando from_holding (compatibile col tuo ambiente).
    """
    import pandas as pd
    import numpy as np

    eq = pd.Series(eq).dropna().copy()

    # pulizia index
    try:
        if eq.index.tz is not None:
            eq.index = eq.index.tz_localize(None)
    except Exception:
        pass
    eq.index = pd.to_datetime(eq.index).normalize()
    eq = eq.sort_index()
    eq = eq[~eq.index.duplicated(keep="last")]

    # guard-rail: equity deve essere positiva
    eq = eq.replace([np.inf, -np.inf], np.nan).dropna()
    eq = eq[eq > 0]

    import vectorbt as vbt
    pf_tmp = vbt.Portfolio.from_holding(
        eq,
        init_cash=init_cash,
        freq="D",
        group_by=False
    )
    return pf_tmp


def build_rolling_summaries_table(
    pf,
    *,
    horizons_years=(1, 2, 3, 5),
    annual_trading_days=252,
    risk_free_rate=0.02,
    alpha_analysis=False,
    init_cash=100_000,
    asof_date=None,
):
    import pandas as pd
    import numpy as np
    import vectorbt as vbt

    # --- daily returns base ---
    r = pf.returns()
    if isinstance(r, pd.DataFrame):
        r = r.iloc[:, 0]
    r = r.dropna().copy()
    try:
        if r.index.tz is not None:
            r.index = r.index.tz_localize(None)
    except Exception:
        pass
    r.index = pd.to_datetime(r.index).normalize()
    r = r.sort_index()
    r = r[~r.index.duplicated(keep="last")]

    if r.empty:
        raise ValueError("Rendimenti vuoti.")

    if asof_date is None:
        asof_date = r.index.max()
    else:
        asof_date = pd.to_datetime(asof_date).normalize()

    r = r.loc[:asof_date]
    if r.empty:
        raise ValueError("asof_date fuori range.")
    asof_date = r.index.max()

    # ---- Totale (full history) usando vbt/summary esistente ----
    # (qui va bene usare la tua create_portfolio_summary sul pf totale)
    df_total = create_portfolio_summary(
        pf,
        benchmark_portfolio=None,
        sel_tickers=None,
        alpha_analysis=alpha_analysis,
        risk_free_rate=risk_free_rate,
        show=False,
        run_as_app=False
    )
    rows = {"Totale": df_total["Valore"]}

    # ---- helper: summary rolling "DA RETURNS" (coerente col grafico) ----
    def _summary_from_returns(ret: pd.Series) -> pd.Series:
        ret = ret.dropna()
        n = int(ret.shape[0])
        if n <= 1:
            return pd.Series(dtype=float)

        # coerente col grafico rolling: prod(1+r)-1
        total_ret = float((1.0 + ret).prod() - 1.0)

        # equity curve (base 1)
        eq = (1.0 + ret).cumprod()
        dd = (eq / eq.cummax()) - 1.0
        max_dd = float(abs(dd.min())) if not dd.empty else np.nan

        cagr = float((1.0 + total_ret) ** (annual_trading_days / n) - 1.0)

        vol = float(ret.std(ddof=0) * np.sqrt(annual_trading_days))

        rf_daily = (1 + risk_free_rate) ** (1 / annual_trading_days) - 1
        std = float(ret.std(ddof=0))
        sharpe = float(((ret.mean() - rf_daily) / std) * np.sqrt(annual_trading_days)) if std > 0 else np.nan

        final_value = float(init_cash * (1.0 + total_ret))

        return pd.Series({
            "Periodo": f"{ret.index.min().date().isoformat()} → {ret.index.max().date().isoformat()}",
            "Importo investito (€)": float(init_cash),
            "Valore patrimoniale netto (€)": final_value,
            "Giorni di trading": n,
            "Ritorno totale": total_ret,
            "CAGR": cagr,
            "Max Drawdown": max_dd,
            "Volatilità annua": vol,
            "Rapporto di Sharpe": sharpe,
            # non ha senso per finestre rolling “sintetiche”:
            "Operazioni al mese": np.nan,
            "Market Time Exposure": np.nan,
        })

    # ---- Rolling: ULTIMA finestra, ma:
    # 1) solo se esiste nel grafico (len(r) >= ndays)
    # 2) skip se coincide col Totale (ndays >= len(r))
    n_total = int(len(r))

    for y in horizons_years:
        ndays = int(annual_trading_days * y)

        # se rolling sarebbe uguale al Totale, non mostrarlo
        if ndays >= n_total:
            continue

        # se non ho abbastanza dati, nel grafico rolling è NaN -> skip
        if n_total < ndays:
            continue

        r_win = r.iloc[-ndays:].copy()
        s = _summary_from_returns(r_win)
        if not s.empty:
            rows[f"Rolling {y}y (last window)"] = s

    out = pd.DataFrame(rows).T

    # ---- formatting (uguale al tuo) ----
    wanted = [
        "Periodo",
        "Importo investito (€)",
        "Valore patrimoniale netto (€)",
        "Giorni di trading",
        "Ritorno totale",
        "CAGR",
        "Max Drawdown",
        "Volatilità annua",
        "Rapporto di Sharpe",
        "Operazioni al mese",
        "Market Time Exposure",
    ]
    out = out.reindex(columns=wanted)

    fmt = out.copy()

    pct_cols = ["Ritorno totale", "CAGR", "Max Drawdown", "Volatilità annua", "Market Time Exposure"]
    num_cols = ["Rapporto di Sharpe", "Operazioni al mese"]
    money_cols = ["Importo investito (€)", "Valore patrimoniale netto (€)"]

    for c in pct_cols:
        if c in fmt.columns:
            fmt[c] = fmt[c].apply(lambda x: "n/a" if pd.isna(x) else f"{x*100:.2f}%")

    for c in num_cols:
        if c in fmt.columns:
            fmt[c] = fmt[c].apply(lambda x: "n/a" if pd.isna(x) else f"{x:.2f}")

    for c in money_cols:
        if c in fmt.columns:
            fmt[c] = fmt[c].apply(lambda x: "n/a" if pd.isna(x) else f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".") + "€")

    if "Giorni di trading" in fmt.columns:
        fmt["Giorni di trading"] = fmt["Giorni di trading"].apply(lambda x: "n/a" if pd.isna(x) else str(int(x)))

    return fmt, out
    
def analyze_rolling_horizons(
    roll_cum_df: pd.DataFrame,
    loss_threshold: float = 0.0,             # compat: la vecchia soglia “principale”
    *,
    loss_thresholds: tuple[float, ...] | None = None,  # nuovo: più soglie
    min_obs: int = 30,
    target_prob: float | None = 0.0          # opzionale: “min safe” su soglia 0
) -> dict:
    import numpy as np
    import pandas as pd

    # 1) orizzonti (es: '1y','2y'...)
    cols = [c for c in roll_cum_df.columns if isinstance(c, str) and c.endswith("y")]
    if not cols:
        raise ValueError("roll_cum_df non ha colonne tipo '1y','2y',...")

    horizons = sorted([(int(c[:-1]), c) for c in cols], key=lambda x: x[0])

    # 2) soglie
    if loss_thresholds is None:
        loss_thresholds = (float(loss_threshold),)
    else:
        loss_thresholds = tuple(float(x) for x in loss_thresholds)

    # 3) calcolo probabilità P(R < soglia) per ogni orizzonte e soglia
    prob_rows = []
    for h, col in horizons:
        s = roll_cum_df[col].replace([np.inf, -np.inf], np.nan).dropna()
        n = int(len(s))
        row = {"horizon_years": h, "n_obs": n}
        for thr in loss_thresholds:
            row[f"p_lt_{thr}"] = float((s < thr).mean()) if n >= min_obs else np.nan
        prob_rows.append(row)

    prob_df = pd.DataFrame(prob_rows).set_index("horizon_years").sort_index()

    # 4) “min safe horizon” sulla soglia 0 (se presente e target_prob valorizzato)
    min_safe = None
    if target_prob is not None:
        # cerco la colonna per thr=0.0 (attenzione a float repr)
        # quindi la ricavo cercando la soglia “più vicina a 0”
        thr0 = min(loss_thresholds, key=lambda x: abs(x - 0.0))
        col0 = f"p_lt_{thr0}"
        if abs(thr0) < 1e-12 and col0 in prob_df.columns:
            for h in prob_df.index:
                p = prob_df.loc[h, col0]
                if pd.notna(p) and p <= float(target_prob):
                    min_safe = int(h)
                    break

    return {
        "prob_df": prob_df,                 # matrice delle probabilità
        "loss_thresholds": loss_thresholds,
        "loss_threshold": float(loss_threshold),  # compat
        "target_prob": target_prob,
        "min_safe_horizon": min_safe,
        "min_obs": int(min_obs),
    }



def plot_loss_probability_curve(
    analysis: dict,
    *,
    title: str = "Probabilità di perdita (rolling < soglia) vs orizzonte",
    show_point_labels: bool = True,
    height: int = 420
):

    prob_df: pd.DataFrame = analysis["prob_df"]
    thresholds = analysis["loss_thresholds"]
    min_safe = analysis.get("min_safe_horizon", None)
    target_prob = analysis.get("target_prob", None)

    # --- X NUMERICO (fix: evita asse categorico) ---
    # prob_df.index può essere [1,2,3,5] oppure ["1","2","3","5"] ecc.
    xs = pd.to_numeric(pd.Index(prob_df.index), errors="coerce").astype(float).to_list()

    fig = go.Figure()

    for thr in thresholds:
        col = f"p_lt_{thr}"
        if col not in prob_df.columns:
            continue

        ys = prob_df[col].astype(float).values
        name = f"P(R < {thr:.0%})" if abs(thr) > 1e-12 else "P(R < 0%)"

        fig.add_trace(go.Scatter(
            x=xs, y=ys,
            mode="lines+markers",
            name=name,
            hovertemplate="Orizzonte: %{x}y<br>P: %{y:.2%}<extra></extra>"
        ))

        if show_point_labels:
            fig.add_trace(go.Scatter(
                x=xs, y=ys,
                mode="text",
                text=[f"{v*100:.1f}%" if pd.notna(v) else "" for v in ys],
                textposition="top center",
                showlegend=False,
                hoverinfo="skip"
            ))


    if min_safe is not None:
        fig.add_vline(x=min_safe, line_width=1, line_dash="dot")
    
        if target_prob is not None:
            dx=0.3
            x_text = min_safe + dx

            fig.add_annotation(
                x=x_text,
                y=0.12,          # più alto: 12% dal fondo del pannello
                xref="x",
                yref="paper",
                text=f"Min safe: {min_safe} (P≤{float(target_prob):.0%})",
                showarrow=False,
                yanchor="bottom",
                bgcolor="rgba(255,255,255,0.85)",
                bordercolor="rgba(80,80,80,0.35)",
                borderwidth=0.9,
                font=dict(size=14, color="green")

            )


    fig.update_layout(
        title=title,
        xaxis_title="Orizzonte (anni)",
        yaxis_title="Probabilità",
        yaxis=dict(tickformat=".0%"),
        template="plotly_white",
        height=height,
        hovermode="x unified",
    )

    # FIX CRITICO: asse X lineare (non categorie)
    fig.update_xaxes(type="linear")

    return fig

def compute_rolling_extrema_ranges(roll_cum_df: pd.DataFrame,
                                   daily_index: pd.DatetimeIndex,
                                   annual_trading_days: int = 252) -> pd.DataFrame:
    """
    Per ogni colonna di roll_cum_df (es. '1y','2y',...) trova:
      - il massimo osservato -> (start_date, end_date, return, duration_days)
      - il minimo osservato -> (start_date, end_date, return, duration_days)

    roll_cum_df: DataFrame con index DatetimeIndex e colonne '1y','2y',...
                 i valori sono rolling total returns (es. prod(1+rets)-1).
    daily_index: DatetimeIndex giornaliero corrispondente ai returns usati per il rolling
                 (di solito daily_rets.index, normalizzato a midnight).
    annual_trading_days: numero di trading days/anno usato per calcolare la finestra.
    """
    rows = []
    # normalizza index a DatetimeIndex senza tz
    idx = pd.to_datetime(daily_index).normalize()
    # mapping index -> posizione per lookup rapido
    pos_map = {d: i for i, d in enumerate(idx)}

    for col in roll_cum_df.columns:
        series = roll_cum_df[col].dropna()
        if series.empty:
            rows.append({
                "horizon": col,
                "max_start": pd.NaT, "max_end": pd.NaT, "max_return": np.nan, "max_days": np.nan,
                "min_start": pd.NaT, "min_end": pd.NaT, "min_return": np.nan, "min_days": np.nan,
            })
            continue

        # determina ndays usati nella rolling (es. '1y' -> 1 * annual_trading_days)
        # supporta etichette come '1y' o '2y' o numeriche
        try:
            if isinstance(col, str) and col.endswith("y"):
                years = int(col[:-1])
            else:
                years = int(col)
        except Exception:
            # fallback: stima window dalla differenza di posizioni utili (non ideale)
            # mettiamo il valore default = annual_trading_days
            years = 1

        ndays = int(round(years * annual_trading_days))

        # ---- massimo ----
        max_end = series.idxmax()               # data di fine finestra per il massimo
        max_val = float(series.loc[max_end])
        # posizione corrispondente in daily_index
        pos_end = pos_map.get(pd.to_datetime(max_end).normalize(), None)
        if pos_end is None:
            # se non trovi pos (possibile per differenze minime), usa get_indexer
            pos_end = idx.get_indexer([pd.to_datetime(max_end).normalize()])[0]
        start_pos = pos_end - (ndays - 1)
        if start_pos >= 0:
            max_start = idx[start_pos]
            max_days = (max_end - max_start).days + 1
        else:
            max_start = pd.NaT
            max_days = np.nan

        # ---- minimo ----
        min_end = series.idxmin()
        min_val = float(series.loc[min_end])
        pos_end = pos_map.get(pd.to_datetime(min_end).normalize(), None)
        if pos_end is None:
            pos_end = idx.get_indexer([pd.to_datetime(min_end).normalize()])[0]
        start_pos = pos_end - (ndays - 1)
        if start_pos >= 0:
            min_start = idx[start_pos]
            min_days = (min_end - min_start).days + 1
        else:
            min_start = pd.NaT
            min_days = np.nan

        rows.append({
            "horizon": col,
            "max_start": pd.to_datetime(max_start) if not pd.isna(max_start) else pd.NaT,
            "max_end": pd.to_datetime(max_end),
            "max_return": max_val,
            "max_days": int(max_days) if not np.isnan(max_days) else np.nan,
            "min_start": pd.to_datetime(min_start) if not pd.isna(min_start) else pd.NaT,
            "min_end": pd.to_datetime(min_end),
            "min_return": min_val,
            "min_days": int(min_days) if not np.isnan(min_days) else np.nan,
        })

    out = pd.DataFrame(rows)
    # ordina per horizon se necessario (es. 1y,2y,...)
    def _hkey(x):
        try:
            if isinstance(x, str) and x.endswith("y"):
                return int(x[:-1])
            return int(x)
        except:
            return 999
    out = out.sort_values(by="horizon", key=lambda s: s.map(_hkey)).reset_index(drop=True)
    return out
    
def plot_cumulative_and_rolling_returns(
    pf,
    horizons_years=None,
    annual_trading_days=252,
    title="Rendimenti cumulati e rolling del portafoglio",
    height=700,
    annotate_extrema=True,
    show_controls=True,
    show_rolling_summary: bool = True,
    rolling_summary_risk_free_rate: float = 0.02,
    # --- fan chart ---
    add_fan_chart: bool = True,
    fan_horizon: str = "1y",
    fan_window_days: int = 252,
    fan_percentiles: tuple = (5, 25, 50, 75, 95),
    # --- heatmap ---
    add_heatmap: bool = True,
    return_heatmap: bool = False,
    # --- horizon analysis ---
    add_horizon_analysis: bool = True,
    loss_threshold: float = 0.0,
    min_obs: int = 30,
    # --- loss_probability_vs_horizon ---
    add_loss_prob_curve: bool = True,
    loss_thresholds: tuple = (0.0, -0.05, -0.10),
    loss_target_prob: float | None = 0.0,
    show_loss_point_labels: bool = True,
    # --- NEW: tabella finestre best/worst per rolling ---
    show_windows_table: bool = True,
    # --- return esteso ---
    return_extras: bool = False,
):
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go

    if horizons_years is None:
        horizons_years = [1, 2, 3, 5]

    # -----------------------------
    # Base data
    # -----------------------------
    daily_rets = pf.returns()
    cum_rets_tot = pf.cumulative_returns()

    if isinstance(daily_rets, pd.DataFrame):
        daily_rets = daily_rets.iloc[:, 0]
    if isinstance(cum_rets_tot, pd.DataFrame):
        cum_rets_tot = cum_rets_tot.iloc[:, 0]

    # pulizia minima
    daily_rets = daily_rets.dropna().copy().sort_index()
    cum_rets_tot = cum_rets_tot.dropna().copy().sort_index()

    if daily_rets.empty or cum_rets_tot.empty:
        raise ValueError("Serie returns/cumulative_returns vuote: impossibile plottare rolling.")

    analysis_start = daily_rets.index.min()
    analysis_end = daily_rets.index.max()

    # indice trading effettivo
    idx = daily_rets.index

    # finestra usata per linee/slider (evita sconfini oltre ultimo giorno disponibile)
    idx_window = idx[(idx >= analysis_start) & (idx <= analysis_end)]
    if len(idx_window) == 0:
        raise ValueError("Finestra indice vuota: controlla analysis_start/analysis_end.")

    # -----------------------------
    # Rolling cumulative returns
    # -----------------------------
    roll_cum_dict = {}
    window_days_map = {}  # label -> ndays
    for y in horizons_years:
        ndays = int(annual_trading_days * y)
        label = f"{y}y"
        roll = (1 + daily_rets).rolling(ndays).apply(np.prod, raw=True) - 1
        roll_cum_dict[label] = roll
        window_days_map[label] = ndays

    roll_cum_df = pd.DataFrame(roll_cum_dict, index=daily_rets.index)

    # -----------------------------
    # Main figure
    # -----------------------------
    fig = go.Figure()

    color_map = {
        "Totale": "blue",
        "1y": "orange",
        "2y": "purple",
        "3y": "green",
        "5y": "red",
    }

    # Totale
    cum_slice = cum_rets_tot.loc[analysis_start:analysis_end]
    fig.add_trace(go.Scatter(
        x=cum_slice.index,
        y=cum_slice.values,
        name="Totale",
        mode="lines",
        line=dict(width=2, color=color_map["Totale"]),
        hovertemplate="%{x|%Y-%m-%d}<br>Totale: %{y:.2%}<extra></extra>",
        legendgroup="lg_Totale",
        showlegend=True,
    ))

    # -----------------------------
    # Helper per start-date finestra (trading-days)
    # -----------------------------
    def _start_from_end(end_ts: pd.Timestamp, n: int) -> pd.Timestamp | None:
        """Start della finestra che termina in end_ts e contiene n osservazioni."""
        try:
            pos = idx.get_loc(end_ts)
            if isinstance(pos, slice):
                pos = pos.stop - 1
            start_pos = int(pos) - int(n) + 1
            if start_pos < 0:
                return None
            return idx[start_pos]
        except Exception:
            return None

    # -----------------------------
    # Rolling curves (+ windows table)
    # -----------------------------
    windows_rows = []

    for label in roll_cum_df.columns:
        s = roll_cum_df[label].loc[analysis_start:analysis_end].copy()
        ss = s.dropna()
        if ss.empty:
            # niente dati validi -> non aggiungere nulla (niente legenda)
            continue

        c = color_map.get(label, "gray")
        legend_name = f"Rolling {label}"
        lg = f"lg_{label}"

        # 1) Traccia principale (in legenda)
        fig.add_trace(go.Scatter(
            x=ss.index,
            y=ss.values,
            name=legend_name,
            mode="lines",
            line=dict(width=1.6, color=c),
            hovertemplate="%{x|%Y-%m-%d}<br>" + f"{legend_name}: %{{y:.2%}}<extra></extra>",
            legendgroup=lg,
            showlegend=True,
        ))

        # 2) Area negativa (ancorata al gruppo, non in legenda)
        neg = ss.copy()
        neg[neg > 0] = 0.0
        fig.add_trace(go.Scatter(
            x=neg.index,
            y=neg.values,
            fill="tozeroy",
            mode="none",
            hoverinfo="skip",
            showlegend=False,
            fillcolor="rgba(255,0,0,0.18)",
            legendgroup=lg,
        ))

        # 3) Marker massimo/minimo + tabella finestre
        if annotate_extrema:
            max_date = ss.idxmax()
            min_date = ss.idxmin()
            max_val = float(ss.loc[max_date])
            min_val = float(ss.loc[min_date])

            fig.add_trace(go.Scatter(
                x=[max_date], y=[max_val],
                mode="markers+text",
                text=[f"▲ {max_val:.2%}"],
                textposition="top center",
                marker=dict(color="green", size=8),
                showlegend=False,
                hovertemplate="%{x|%Y-%m-%d}<br>Max: %{y:.2%}<extra></extra>",
                legendgroup=lg,
            ))

            fig.add_trace(go.Scatter(
                x=[min_date], y=[min_val],
                mode="markers+text",
                text=[f"▼ {min_val:.2%}"],
                textposition="bottom center",
                marker=dict(color="red", size=8),
                showlegend=False,
                hovertemplate="%{x|%Y-%m-%d}<br>Min: %{y:.2%}<extra></extra>",
                legendgroup=lg,
            ))

            ndays = int(window_days_map.get(label, annual_trading_days))
            best_start = _start_from_end(max_date, ndays)
            worst_start = _start_from_end(min_date, ndays)

            windows_rows.append({
                "Orizzonte": label,
                "Trading_days": ndays,
                "Best_start": best_start,
                "Best_end": max_date,
                "Best_return": max_val,
                "Worst_start": worst_start,
                "Worst_end": min_date,
                "Worst_return": min_val,
            })

    windows_df = pd.DataFrame(windows_rows)

    # zero line (tagliata alla finestra reale)
    fig.add_trace(go.Scatter(
        x=idx_window,
        y=[0] * len(idx_window),
        mode="lines",
        line=dict(color="red", dash="dot"),
        name="Soglia 0%",
        hoverinfo="skip",
        legendgroup="lg_Zero",
        showlegend=True
    ))

    # -----------------------------
    # Fan chart (rolling percentiles)
    # -----------------------------
    if add_fan_chart and fan_horizon in roll_cum_df.columns:
        sr = roll_cum_df[fan_horizon].loc[analysis_start:analysis_end].dropna()
        if not sr.empty:
            bands = {}
            for p in fan_percentiles:
                bands[p] = sr.rolling(fan_window_days).quantile(p / 100.0)
            bands = pd.DataFrame(bands)

            if {5, 25, 50, 75, 95}.issubset(bands.columns):
                lg_fan = f"lg_fan_{fan_horizon}"

                fig.add_trace(go.Scatter(
                    x=bands.index, y=bands[5],
                    line=dict(width=0),
                    showlegend=True,
                    name=f"{fan_horizon} P5–P95",
                    legendgroup=lg_fan
                ))
                fig.add_trace(go.Scatter(
                    x=bands.index, y=bands[95],
                    fill="tonexty",
                    line=dict(width=0),
                    fillcolor="rgba(120,120,120,0.18)",
                    showlegend=False,
                    legendgroup=lg_fan,
                    hoverinfo="skip"
                ))
                fig.add_trace(go.Scatter(
                    x=bands.index, y=bands[25],
                    line=dict(width=0),
                    showlegend=True,
                    name=f"{fan_horizon} P25–P75",
                    legendgroup=lg_fan
                ))
                fig.add_trace(go.Scatter(
                    x=bands.index, y=bands[75],
                    fill="tonexty",
                    line=dict(width=0),
                    fillcolor="rgba(120,120,120,0.30)",
                    showlegend=False,
                    legendgroup=lg_fan,
                    hoverinfo="skip"
                ))
                fig.add_trace(go.Scatter(
                    x=bands.index, y=bands[50],
                    line=dict(width=1, dash="dot", color="black"),
                    name=f"{fan_horizon} Mediana",
                    legendgroup=lg_fan,
                    showlegend=True,
                    hoverinfo="skip"
                ))

    # -----------------------------
    # Layout (+ togglegroup)
    # -----------------------------
    fig.update_layout(
        title=title,
        height=height,
        hovermode="x unified",
        template="plotly_white",
        yaxis=dict(title="Rendimento cumulato (%)", tickformat=".0%"),
        xaxis=dict(
            title="Data",
            range=[analysis_start, analysis_end],
            autorange=False,
            rangeslider=dict(
                visible=True,
                autorange=False,
                range=[analysis_start, analysis_end]
            ),
        ),
        legend=dict(
            x=0.01, y=0.99,
            bgcolor="rgba(255,255,255,0.7)",
            bordercolor="black",
            borderwidth=1,
            groupclick="togglegroup"
        )
    )

    # -----------------------------
    # Tabella finestre best/worst (display)
    # -----------------------------
    if show_windows_table and windows_df is not None and not windows_df.empty:
        try:
            from IPython.display import display, HTML
    
            df_show = windows_df.copy()
    
            # date pulite
            for c in ["Best_start", "Best_end", "Worst_start", "Worst_end"]:
                df_show[c] = pd.to_datetime(df_show[c]).dt.date
    
            # rendimenti in percentuale
            df_show["Best_return"] = (df_show["Best_return"] * 100).round(2)
            df_show["Worst_return"] = (df_show["Worst_return"] * 100).round(2)
    
            # titolo tabella
            display(HTML(
                "<h4 style='margin-top:15px'>"
                "Finestre rolling migliori e peggiori per orizzonte"
                "</h4>"
                "<p style='color:gray; font-size:12px'>"
                "Per ogni orizzonte (1y, 2y, 3y, …) sono riportati il periodo "
                "che ha generato il rendimento massimo e minimo (Total Return rolling)."
                "</p>"
            ))
    
            display(df_show)
    
        except Exception:
            pass

    # -----------------------------
    # Rolling heatmap
    # -----------------------------
    fig_hm = None
    if add_heatmap:
        hm = roll_cum_df.loc[analysis_start:analysis_end].replace([np.inf, -np.inf], np.nan)

        # se tutto NaN evita crash su nanmin/nanmax
        finite = np.isfinite(hm.values)
        if not finite.any():
            fig_hm = None
        else:
            zmin = float(np.nanmin(hm.values))
            zmax = float(np.nanmax(hm.values))
        
            # --- Colorscale robusta ---
            if zmin >= 0:
                # solo positivi: 0 è il minimo "logico"
                zmin = 0.0
                colorscale = [
                    [0.0, "rgb(255,255,255)"],
                    [1.0, "rgb(0,104,55)"],
                ]
            elif zmax <= 0:
                # solo negativi: 0 è il massimo "logico"
                zmax = 0.0
                colorscale = [
                    [0.0, "rgb(165,0,38)"],
                    [1.0, "rgb(255,255,255)"],
                ]
            else:
                # misto: scala divergente centrata su 0
                p0 = (0.0 - zmin) / (zmax - zmin)
                colorscale = [
                    [0.0, "rgb(165,0,38)"],
                    [p0,  "rgb(255,255,255)"],
                    [1.0, "rgb(0,104,55)"],
                ]

        fig_hm = go.Figure(go.Heatmap(
            z=hm.T.values,
            x=hm.index,
            y=hm.columns,
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            colorbar=dict(title="Rolling Return", tickformat=".0%"),
            hovertemplate="%{x|%Y-%m-%d}<br>%{y}: %{z:.2%}<extra></extra>"
        ))

        fig_hm.update_layout(
            title=dict(
                text=(
                    "Mappa temporale dei rendimenti rolling (Total Return)"
                    "<br><span style='font-size:12px;color:gray'>"
                    "Rendimenti totali osservati ex-post su finestre mobili di diversa durata."
                    "</span>"
                ),
                x=0.5
            ),
            height=420,
            template="plotly_white"
        )

    # -----------------------------
    # Horizon analysis
    # -----------------------------
    analysis = None
    fig_loss = None
    if add_horizon_analysis:
        analysis = analyze_rolling_horizons(
            roll_cum_df.loc[analysis_start:analysis_end],
            loss_threshold=loss_threshold,
            loss_thresholds=loss_thresholds,
            min_obs=min_obs,
            target_prob=loss_target_prob
        )

        if add_loss_prob_curve:
            fig_loss = plot_loss_probability_curve(
                analysis,
                show_point_labels=show_loss_point_labels
            )

    # -----------------------------
    # Return
    # -----------------------------
    if return_extras:
        return {
            "fig": fig,
            "fig_hm": fig_hm,
            "fig_loss": fig_loss,
            "analysis": analysis,
            "roll_cum_df": roll_cum_df,
            "windows_df": windows_df,
        }

    if return_heatmap:
        return fig, fig_hm

    return fig
    
def plot_annual_return_triangle(
    pf,
    resample_freq: str = "YE",
    run_as_app: bool = False
) -> Union[Tuple, Tuple[object, pd.DataFrame, str]]:
    """
    Calcola un "triangolo" di rendimenti medi annuali rolling e lo plotta
    con etichette X colorate: rosso se la colonna ha almeno un rendimento negativo,
    verde altrimenti.

    Returns:
        (fig, triangle_df) oppure (fig, triangle_df, msg) se run_as_app=True
    """
    # 1) Prendi i rendimenti giornalieri e ricostruisci price index
    dr = pf.returns()
    if isinstance(dr, pd.DataFrame):
        dr = dr.iloc[:, 0]
    price_index = (1 + dr).cumprod()

    # 2) Prezzo di fine anno
    alias_map = {"A": "YE", "Y": "YE", "YE": "YE"}
    freq = alias_map.get(resample_freq, "YE")
    yearly_price = price_index.resample(freq).last().to_frame("Price")

    # 3) Log-return annuale
    annual_ret = np.log(yearly_price["Price"] / yearly_price["Price"].shift(1)).dropna().to_frame("Return")

    # --- FIX ROBUSTO: anno di fine finestra corretto (gestisce anche timestamp al 01/01) ---
    annual_ret.index = (pd.to_datetime(annual_ret.index) - pd.Timedelta(days=1)).year
    annual_ret.index.name = "Year"

    # 4) Rolling mean su finestre nY
    total_years = len(annual_ret)
    windows = list(range(total_years, 0, -1))
    for n in windows:
        annual_ret[f"{n}Y"] = annual_ret["Return"].rolling(window=n).mean()
    triangle_df = annual_ret.drop(columns="Return")

    # 5) Orizzonte minimo consigliato
    recommended = None
    for n in sorted(windows):
        vals = triangle_df[f"{n}Y"].dropna()
        if len(vals) > 0 and (vals > 0).all():
            recommended = n
            break

    if recommended:
        if run_as_app:
            msg = (
                "Triangolo dei rendimenti medi annualizzati (CAGR). "
                "Analisi ex-post su finestre discrete di ingresso e uscita. "
                "La valutazione dell’orizzonte minimo di investimento è fornita "
                "dalla mappa rolling e dalla curva di probabilità di perdita."
            )
        else:
            msg = (
                "Triangolo dei rendimenti medi annualizzati (CAGR).\n"
                "Analisi ex-post su finestre discrete di ingresso e uscita.\n"
                "La valutazione dell’orizzonte minimo di investimento è fornita "
                "dalla mappa rolling e dalla curva di probabilità di perdita."
            )
    else:
        msg = (
            "Triangolo dei rendimenti medi annualizzati (CAGR).\n"
            "Analisi ex-post su finestre discrete di ingresso e uscita.\n"
            "La valutazione dell’orizzonte minimo di investimento è fornita "
            "dalla mappa rolling e dalla curva di probabilità di perdita."
        )

    if run_as_app:
        import streamlit as st
        st.markdown(msg,unsafe_allow_html=False)
    else:
        print(msg)

    # 6) Plot con plotly_heatmap_triangle
    fig = plotly_heatmap_triangle(
        triangle_df,
        vmin=-0.2,
        vmax=0.2,
        colorscale='RdYlGn',
        # title="Triangolo dei rendimenti medi annuali (rolling log-return)",
        width=900,
        height=700
    )

    # 7) Colora le tick labels sull'asse X
    cols = list(triangle_df.columns)
    # per ogni col, se almeno un valore < 0 → rosso, else verde
    tick_colors = [
        "red" if (triangle_df[col] < 0).any() else "green"
        for col in cols
    ]
    # creiamo ticktext con span colorato
    tickvals = cols
    ticktext = [
        f"<span style='color:{c}'>{val}</span>"
        for val, c in zip(cols, tick_colors)
    ]
    fig.update_xaxes(
        tickvals=tickvals,
        ticktext=ticktext
    )

    if run_as_app:
        return fig, triangle_df, msg
    else:
        return fig, triangle_df
    
def plotly_heatmap_triangle(
    triangle_df: pd.DataFrame,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    colorscale: str = 'RdYlGn',
    title: Optional[str] = None,   # <-- default None
    width: int = 900,
    height: int = 700
) -> go.Figure:
    # 1) Drop righe/colonne piene di NaN
    df = triangle_df.dropna(axis=0, how='all').dropna(axis=1, how='all')

    # 2) Prepara z
    z = df.values * 100
        
    # 3) Prepara text ma sostituisci NaN con stringa vuota
    text = []
    for i, row in enumerate(z):
        txt_row = []
        end_year = int(df.index[i])
        for j, val in enumerate(row):
            if pd.isna(df.iat[i, j]):
                txt_row.append("")
            else:
                n = int(str(df.columns[j]).replace("Y", ""))
                start_year = end_year - n + 1
                txt_row.append(f"{val:.1f}%<br>{start_year}-{end_year}")
        text.append(txt_row)

    # 4) Etichette assi
    x_labels = [str(col) for col in df.columns]
    y_labels = [str(idx) for idx in df.index]

    # 5) Costruisci heatmap
    heatmap = go.Heatmap(
        z=z,
        x=x_labels,
        y=y_labels,
        text=text,
        texttemplate="%{text}",
        colorscale=colorscale,
        zmin=(vmin * 100) if vmin is not None else None,
        zmax=(vmax * 100) if vmax is not None else None,
        colorbar=dict(title="%")
    )

    fig = go.Figure(data=heatmap)

    # 6) Layout generale

    if title is None:
        plot_title = dict(
            text=(
                "Triangolo dei rendimenti medi annui (CAGR)"
                "<br><span style='font-size:12px;color:gray'>"
                "Rendimenti medi annui calcolati tra date discrete di ingresso e uscita."
                "</span>"
            ),
            x=0.5,
            xanchor="center"
        )
    else:
        plot_title = title

    fig.update_layout(
        # title=title,
        title=plot_title,
        width=width,
        height=height,
        margin=dict(l=100, r=40, t=80, b=80),
    )
    # 7) Asse X
    fig.update_xaxes(
        title_text="Finestra Mobile (anni)",
        ticks="outside",
        tickangle=0,
    )

    # 8) Asse Y spostato a sinistra
    fig.update_yaxes(
        title_text="Anno di Fine Finestra",
        autorange="reversed",
        ticks="outside",
        tickangle=0,
        side="left"
    )

    return fig
    
def build_and_plot_portfolio_contributions(
    portfolio: 'vbt.Portfolio',
    title: str,
    benchmark: str = "SPY",
    benchmark_data: pd.Series | None = None,
    start_date: pd.Timestamp | str | None = None,
    end_date: pd.Timestamp | str | None = None,
    show_report: bool = True,
):
    """
    Costruisce e plotta i contributi al portafoglio.

    - Curve per singolo asset (contributi)
    - Curva aggregata di portafoglio
    - Eventuale benchmark (interno o esterno)

    Robustezza:
    - start_date/end_date accettano str/Timestamp/datetime/None
    - finestra clampata sul range dati disponibile
    - skip pulito se la finestra non interseca i dati

    NOTE CHIAVE:
    - NESSUN download da yfinance.
    - Il benchmark viene SEMPRE iniettato come serie di returns nel dict `portfolios_returns`.
      * Se `benchmark_data` è fornito => usato come PREZZI (Close) esterni.
      * Altrimenti => usato benchmark interno vectorbt: `portfolio.benchmark_returns(...)`.
    - `plot_multiple_portfolios` viene chiamata con benchmark=None, benchmark_data=None per evitare
      qualsiasi interpretazione del benchmark come ticker.
    """

    import pandas as pd
    import numpy as np

    # -----------------------------
    # Utility
    # -----------------------------
    def _to_ts(x):
        if x is None:
            return None
        try:
            return pd.to_datetime(x)
        except Exception:
            return None

    def _normalize_index(s: pd.Series) -> pd.Series:
        s = s.dropna().copy()
        try:
            if getattr(s.index, "tz", None) is not None:
                s.index = s.index.tz_localize(None)
        except Exception:
            pass
        s.index = pd.to_datetime(s.index).normalize()
        s = s[~s.index.duplicated(keep="last")]
        return s.sort_index()

    def _resolve_window(idx: pd.Index, start, end):
        if idx is None or len(idx) == 0:
            return (None, None)

        s = _to_ts(start)
        e = _to_ts(end)

        idx_min = pd.to_datetime(idx.min())
        idx_max = pd.to_datetime(idx.max())

        if s is None:
            s = idx_min
        if e is None:
            e = idx_max

        if s > e:
            s, e = e, s

        # clamp
        if s < idx_min:
            s = idx_min
        if e > idx_max:
            e = idx_max

        if s > idx_max or e < idx_min:
            return (None, None)

        return (s, e)

    def _prices_to_returns(prices: pd.Series, target_idx: pd.DatetimeIndex) -> pd.Series:
        p = _normalize_index(prices)
        r = p.pct_change().dropna()
        target = pd.to_datetime(target_idx).normalize()
        r = r.reindex(target, method="ffill").fillna(0.0)
        return r

    # -----------------------------
    # Returns per-asset
    # -----------------------------
    try:
        asset_returns = portfolio.returns(group_by=False)
    except Exception:
        if show_report:
            print(f"ℹ️  Skip contributions '{title}': impossibile calcolare returns per-asset")
        return None

    if asset_returns is None or getattr(asset_returns, "empty", False):
        if show_report:
            print(f"ℹ️  Skip contributions '{title}': returns per-asset vuoti")
        return None

    # -----------------------------
    # Returns aggregati portafoglio
    # -----------------------------
    try:
        pf_returns = portfolio.returns()
    except Exception:
        if show_report:
            print(f"ℹ️  Skip contributions '{title}': impossibile calcolare portfolio.returns()")
        return None

    if pf_returns is None or getattr(pf_returns, "empty", False):
        if show_report:
            print(f"ℹ️  Skip contributions '{title}': returns portafoglio vuoti")
        return None

    # -----------------------------
    # Finestra robusta
    # -----------------------------
    s, e = _resolve_window(asset_returns.index, start_date, end_date)
    if s is None:
        if show_report:
            dr_min = asset_returns.index.min()
            dr_max = asset_returns.index.max()
            print(
                f"ℹ️  Skip contributions '{title}': finestra fuori range "
                f"(start={start_date}, end={end_date}, data_range={dr_min}→{dr_max})"
            )
        return None

    asset_returns_w = asset_returns.loc[s:e].dropna(how="all")
    pf_returns_w = pf_returns.loc[s:e].dropna()

    if asset_returns_w.empty and pf_returns_w.empty:
        if show_report:
            print(f"ℹ️  Skip contributions '{title}': nessun dato nella finestra {s}→{e}")
        return None

    # -----------------------------
    # Dict per plot (asset + portfolio hero)
    # -----------------------------
    portfolios_returns = {}
    for t in getattr(asset_returns_w, "columns", []):
        portfolios_returns[str(t)] = asset_returns_w[t].dropna()

    # label sentinella per riconoscere SEMPRE il portafoglio
    _PORTFOLIO_LABEL = f"__PORTFOLIO__::{title}"
    portfolios_returns[_PORTFOLIO_LABEL] = pf_returns_w

    # -----------------------------
    # BENCHMARK (interno o esterno) -> SEMPRE come serie returns
    # -----------------------------
    _BENCH_LABEL = f"__BENCH__::{benchmark}".strip() if benchmark else "__BENCH__"

    bench_ret = None
    try:
        if benchmark_data is not None:
            # benchmark esterno: benchmark_data sono PREZZI (Close)
            bench_ret = _prices_to_returns(benchmark_data, pf_returns_w.index)
        else:
            # benchmark interno vectorbt: returns già pronti
            br = portfolio.benchmark_returns(group_by=True)
            if isinstance(br, pd.DataFrame):
                br = br.mean(axis=1)
            br = _normalize_index(br)
            br = br.reindex(pd.to_datetime(pf_returns_w.index).normalize(), method="ffill").fillna(0.0)
            bench_ret = br
    except Exception:
        bench_ret = None

    if bench_ret is not None and not bench_ret.empty:
        portfolios_returns[_BENCH_LABEL] = bench_ret

    # -----------------------------
    # Plot (NESSUN ticker passato!)
    # -----------------------------
    fig = plot_multiple_portfolios(
        portfolios_returns,
        title=f"Contributi al portafoglio: {title}",
        benchmark=None,
        benchmark_data=None,
        start_date=s,
        end_date=e
    )

    # -----------------------------
    # Styling: asset grigi + top-N colorati, portfolio/benchmark hero
    # -----------------------------
    PORTFOLIO_COLOR = "blue"
    BENCHMARK_COLOR = "gray"
    PORTFOLIO_WIDTH = 3.5
    BENCHMARK_WIDTH = 3.0

    ASSET_GRAY = "#C9D1D9"
    ASSET_WIDTH = 1.0
    ASSET_OPACITY = 0.25

    # --- Top-N assets (colorati) ---
    TOPN = 5
    TOPN_WIDTH = 1.8
    TOPN_OPACITY = 0.80
    TOPN_PALETTE = ["#FF6B6B", "#F7B801", "#2EC4B6", "#9B5DE5", "#00BBF9", "#F15BB5"]

    asset_total_ret = (1.0 + asset_returns_w).prod(axis=0) - 1.0
    asset_total_ret = asset_total_ret.replace([np.inf, -np.inf], np.nan).dropna()
    top_assets = asset_total_ret.abs().sort_values(ascending=False).head(TOPN).index.tolist()
    top_color_map = {a: TOPN_PALETTE[i % len(TOPN_PALETTE)] for i, a in enumerate(top_assets)}

    def _is_portfolio_trace(tr):
        n = getattr(tr, "name", "") or ""
        return _PORTFOLIO_LABEL in n

    def _is_benchmark_trace(tr):
        n = getattr(tr, "name", "") or ""
        return _BENCH_LABEL in n

    def _normalize_asset_label(trace_name: str) -> str:
        """
        Normalizza i nomi trace generati da plot_multiple_portfolios.
        Esempi:
          'Portfolio (DTE.DE)' -> 'DTE.DE'
          'Portfolio (DHL.DE (contributo))' -> 'DHL.DE'
        """
        n = (trace_name or "").strip()

        # non toccare le label sentinella
        if n.startswith("__PORTFOLIO__::") or n.startswith("__BENCH__::"):
            return n

        if n.startswith("Portfolio (") and n.endswith(")"):
            n = n[len("Portfolio ("):-1].strip()

        n = n.replace("(contributo)", "").strip()
        return n

    for tr in fig.data:
        name = getattr(tr, "name", "") or ""

        # default: asset grigi
        tr.opacity = ASSET_OPACITY
        if hasattr(tr, "line") and tr.line is not None:
            tr.line.color = ASSET_GRAY
            tr.line.width = ASSET_WIDTH

        # portfolio hero
        if _is_portfolio_trace(tr):
            tr.opacity = 1.0
            if hasattr(tr, "line") and tr.line is not None:
                tr.line.color = PORTFOLIO_COLOR
                tr.line.width = PORTFOLIO_WIDTH
            tr.name = title
            continue

        # benchmark hero
        if _is_benchmark_trace(tr):
            tr.opacity = 1.0
            if hasattr(tr, "line") and tr.line is not None:
                tr.line.color = BENCHMARK_COLOR
                tr.line.width = BENCHMARK_WIDTH
            tr.name = benchmark if benchmark else "Benchmark"
            continue

        # top-N: colorati e più visibili
        asset_key = _normalize_asset_label(name)
        if asset_key in top_color_map:
            tr.opacity = TOPN_OPACITY
            if hasattr(tr, "line") and tr.line is not None:
                tr.line.color = top_color_map[asset_key]
                tr.line.width = TOPN_WIDTH

    return fig
    
def plot_ts_portfolio(
    final_portfolio, 
    portfolio_ts, 
    portfolio_title="Composito", 
    width=1000,
    start_date=None,
    end_date=None,    
    # # nuovo parametro: mostrare le figure a video
    # show_report: bool = True,
):
    """
    Plotta i rendimenti cumulativi dei TS (ricavati da portfolio_ts) e del portafoglio finale,
    normalizzando a 1.0 alla start_date se specificata.

    Parametri:
    - final_portfolio: oggetto vectorbt.Portfolio aggregato
    - portfolio_ts: lista di dict, ognuno con chiave 'portfolio' e 'symbol'
    - portfolio_title: nome da visualizzare per il portafoglio finale
    - width: larghezza grafico in pixel
    - start_date: data iniziale per il filtro (datetime o stringa 'YYYY-MM-DD')
    - end_date: data finale per il filtro (datetime o stringa 'YYYY-MM-DD')
    """
    # 1. Costruzione DataFrame rendimenti cumulati dei TS
    cumulative_intermedi = pd.DataFrame()

    for ts in portfolio_ts:
        symbol = ts["symbol"]
        p = ts["portfolio"]
        cum_returns = 1.0 + p.cumulative_returns()
        cum_returns = cum_returns.copy()

        # Applichiamo filtro temporale e normalizzazione
        if start_date is not None:
            cum_returns = cum_returns[cum_returns.index >= pd.to_datetime(start_date)]
            if not cum_returns.empty:
                cum_returns /= cum_returns.iloc[0]  # normalizza a 1.0

        if end_date is not None:
            cum_returns = cum_returns[cum_returns.index <= pd.to_datetime(end_date)]

        cumulative_intermedi[symbol] = cum_returns

    # 2. Portafoglio finale
    cum_returns_final = 1.0 + final_portfolio.cumulative_returns()
    cum_returns_final = cum_returns_final.copy()

    if start_date is not None:
        cum_returns_final = cum_returns_final[cum_returns_final.index >= pd.to_datetime(start_date)]
        if not cum_returns_final.empty:
            cum_returns_final /= cum_returns_final.iloc[0]  # normalizza a 1.0

    if end_date is not None:
        cum_returns_final = cum_returns_final[cum_returns_final.index <= pd.to_datetime(end_date)]

    # 3. Costruzione grafico
    fig = go.Figure()

    # Tracciati TS
    for col in cumulative_intermedi.columns:
        fig.add_trace(
            go.Scatter(
                x=cumulative_intermedi.index,
                y=cumulative_intermedi[col],
                mode='lines',
                name=f"TS - {col}",
                line=dict(width=1),
                opacity=0.5
            )
        )

    # Tracciato finale
    fig.add_trace(
        go.Scatter(
            x=cum_returns_final.index,
            y=cum_returns_final,
            mode='lines',
            name=f"Portfolio {portfolio_title}",
            line=dict(width=4, color='blue'),
            opacity=1.0
        )
    )
    # 6.1) Aggiungi linea orizzontale rossa a y=1
    fig.add_hline(
        y=1,
        line=dict(color='red', width=2, dash='dash'),
    )

    # Layout
    fig.update_layout(
        title=f"Portfolio {portfolio_title} e Trading System",
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=1, label="1M", step="month", stepmode="backward"),
                    dict(count=3, label="3M", step="month", stepmode="backward"),
                    dict(count=6, label="6M", step="month", stepmode="backward"),
                    dict(count=1, label="YTD", step="year", stepmode="todate"),
                    dict(step="all", label="All")
                ])
            ),
        ),
        width=width,
        height=600,
        template="plotly_white",
        legend=dict(x=1.02, y=1, xanchor='left', yanchor='auto')
    )

    # if show_report: fig.show()

    return fig #, cum_returns_final

## Funzioni di send report via email

In [ ]:
def load_email_credentials(
    secrets_file: str = 'config/tslab_secrets.json',
) -> tuple:
    """
    Carica sender_email e sender_password da un file JSON gitignored.

    Cerca il file in:
    1. <secrets_file> relativo alla CWD corrente
    2. ../../<secrets_file> (per notebook eseguiti da notebooks/runtime/)

    Fallback a variabili di ambiente TSLAB_SENDER_EMAIL /
    TSLAB_SENDER_PASSWORD se il file non e' trovato.

    Parameters
    ----------
    secrets_file : str
        Path relativo al file JSON dei segreti
        (default: 'config/tslab_secrets.json').

    Returns
    -------
    tuple[str, str]
        (sender_email, sender_password)
    """
    import json as _json, os as _os
    for candidate in [secrets_file, _os.path.join('../../', secrets_file)]:
        if _os.path.exists(candidate):
            sec = _json.load(open(candidate))
            return sec['sender_email'], sec['sender_password']
    print(f"WARNING: {secrets_file} non trovato — uso variabili di ambiente.")
    return (
        _os.environ.get('TSLAB_SENDER_EMAIL', ''),
        _os.environ.get('TSLAB_SENDER_PASSWORD', ''),
    )


#
# Generic Send Email report
#
import smtplib
import ssl
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase

def send_report_via_gmail(
    sender_email: str,
    sender_password: str,
    recipient_email: str,
    subject: str,
    body_text: str,
    attachments: list = None ):
    """
    Invia un'email tramite Gmail con testo e, opzionalmente, allegati.
    
    :param sender_email:    L'indirizzo Gmail del mittente (es. "tuo_nome@gmail.com")
    :param sender_password: La password o l'app password generata nelle impostazioni di Google
    :param recipient_email: L'indirizzo del destinatario
    :param subject:         Oggetto dell'email
    :param body_text:       Corpo del messaggio (testo)
    :param attachments:     Lista di path ai file da allegare (opzionale)
    """
    if attachments is None:
        attachments = []

    # Crea il messaggio multipart
    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = recipient_email
    msg['Subject'] = subject

    # Aggiunge il corpo del messaggio come testo
    # msg.attach(MIMEText(body_text, 'plain'))
    msg.attach(MIMEText(body_text, 'html'))

    # Aggiungiamo la parte HTML
    # msg_text = MIMEText(body_html, 'html')

    # Aggiunge gli allegati (se presenti)
    for file_path in attachments:
        try:
            with open(file_path, "rb") as attachment:
                part = MIMEBase("application", "octet-stream")
                part.set_payload(attachment.read())
            encoders.encode_base64(part)
            part.add_header(
                "Content-Disposition",
                f"attachment; filename={file_path.split('/')[-1]}"  # usa solo il nome file
            )
            msg.attach(part)
        except Exception as e:
            print(f"Impossibile allegare il file {file_path}: {e}")
    
    # Avvia la connessione al server SMTP di Gmail (porta 465 = SSL)
    context = ssl.create_default_context()
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
        try:
            print(f"sending mail to: {recipient_email} width subject {subject}")

            server.ehlo()
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
            server.close()
            print(f"mail successfully sent !")
        except :
            print("failed to send mail")
            

def send_email_report(sender_email, sender_password, recipient_email, subject, body_text, attachments):
    """ Invia il report via email. """
    if recipient_email:
        for to in recipient_email.split(','):
            send_report_via_gmail(
                sender_email=sender_email,
                sender_password=sender_password,
                recipient_email=to,
                subject=subject,
                body_text=body_text,
                attachments=attachments
            )
            
def send_portfolio_performance(
    sender_email: str,
    sender_password: str,
    recipient_email: str,
    *,
    assets: dict,
    # html_output: str = "../outputs/report_figures.html",
    html_output: str | None = None,
    figure_titles: list[str] | None = None,
    subject: str | None = None,
):
    # import os
    # import pandas as pd
    import html as _html

    if html_output is None:
        html_output = f"{_TSLAB_OUTPUTS_DIR}/report_figures.html"
        # print(f"html_output: {html_output}")
        
    figs = assets.get("figs", [])
    header = assets.get("header", "Report performance")
    stats_df = assets["stats_df"]        # <-- già formattato (stringhe)
    sintesi_df = assets["sintesi_df"]    # <-- già formattato (stringhe)

    # --- NEW: sezione "Note di lettura performance" ---
    performance_info = assets.get("performance_info", None)
    # supporta:
    # - stringa singola
    # - lista/tuple di stringhe (render come bullet list)
    # - None (sezione omessa)
    performance_tables = assets.get("performance_tables", None)
    # supporta:
    # - dict { "Titolo": pd.DataFrame, ... }
    # - None

    if subject is None:
        subject = f"[TS_LAB] {header}"

    save_figures_to_html_sequential(
        figs=figs,
        output_filename=html_output,
        figure_titles=figure_titles
    )
    attach_name = os.path.basename(html_output)

    def _render_table(df: pd.DataFrame, *, index=True) -> str:
        return (
            df.to_html(index=index, border=0, escape=False)
            .replace('<table ', '<table style="border-collapse:collapse; margin:1em 0;" ')
            .replace('<th>', '<th style="background:#f4f4f4; padding:6px; border:1px solid #ccc;">')
            .replace('<td>', '<td style="padding:6px; border:1px solid #ccc;">')
        )

    def _render_performance_info(info) -> str:
        """Renderizza performance_info in modo robusto."""
        if info is None:
            return ""

        # stringa singola
        if isinstance(info, str):
            txt = _html.escape(info).replace("\n", "<br>")
            return f"<p style='margin:0.6em 0;'>{txt}</p>"

        # lista/tuple di stringhe => bullet list
        if isinstance(info, (list, tuple)):
            items = []
            for x in info:
                if x is None:
                    continue
                s = _html.escape(str(x)).replace("\n", "<br>")
                items.append(f"<li style='margin:0.2em 0;'>{s}</li>")
            if not items:
                return ""
            return "<ul style='margin:0.6em 0 0.6em 1.2em; padding:0;'>" + "".join(items) + "</ul>"

        # fallback: qualunque oggetto -> stringa
        txt = _html.escape(str(info)).replace("\n", "<br>")
        return f"<p style='margin:0.6em 0;'>{txt}</p>"

    def _render_performance_tables(tables) -> str:
        """Renderizza dict di DataFrame (opzionale)."""
        if tables is None:
            return ""
        if not isinstance(tables, dict) or len(tables) == 0:
            return ""

        blocks = []
        for title, df in tables.items():
            if df is None:
                continue
            if not isinstance(df, pd.DataFrame):
                # se arriva qualcosa di diverso, prova render a stringa
                blocks.append(f"<h4 style='margin:0.8em 0 0.2em 0;'>{_html.escape(str(title))}</h4>")
                blocks.append(f"<p>{_html.escape(str(df)).replace(chr(10), '<br>')}</p>")
                continue

            blocks.append(f"<h4 style='margin:0.8em 0 0.2em 0;'>{_html.escape(str(title))}</h4>")
            blocks.append(_render_table(df, index=True))

        return "".join(blocks)

    perf_info_html = _render_performance_info(performance_info)
    perf_tables_html = _render_performance_tables(performance_tables)

    perf_section_html = ""
    if perf_info_html or perf_tables_html:
        perf_section_html = f"""
        <h3>Note di lettura performance</h3>
        {perf_info_html}
        {perf_tables_html}
        """

    html_report = f"""
    <html>
      <head><meta charset="utf-8"></head>
      <body>
        <h2>{header}</h2>

        <h3>Sintesi</h3>
        {_render_table(sintesi_df, index=False)}

        <h3>Dettaglio</h3>
        {_render_table(stats_df, index=True)}

        {perf_section_html}

        <h3>Grafici interattivi</h3>
        <p>
          In allegato <b>{attach_name}</b> con i grafici interattivi (Plotly).
          Aprire il file con un browser.
        </p>
      </body>
    </html>
    """

    send_email_report(
        sender_email,
        sender_password,
        recipient_email,
        subject,
        html_report,
        attachments=[html_output]
    )


def save_figures_to_html_sequential(
    figs: list[Figure],
    output_filename: str,
    figure_titles: list[str] | None = None,
    fig_width: int | None = None,
    fig_height: int | None = None
):
    """
    Salva un elenco di figure Plotly in un unico file HTML, una sotto l’altra.
    Allinea automaticamente `figure_titles` a `figs`, riempiendo o troncando.
    Di default rispetta width/height già in fig.layout. Se vuoi forzarle,
    passale con fig_width/fig_height.

    :param figs:             lista di plotly.graph_objs.Figure
    :param output_filename:  path del file HTML di output
    :param figure_titles:    titoli opzionali anteposti alle figure
                              (verranno pad o truncated)
    :param fig_width:        larghezza px da forzare (None = lascia fig.layout)
    :param fig_height:       altezza px da forzare (None = lascia fig.layout)
    """
    n = len(figs)

    # 1) Normalizza figure_titles alla stessa lunghezza di figs
    if figure_titles is None:
        titles = [""] * n
    else:
        # pad o truncate
        if len(figure_titles) < n:
            titles = figure_titles + [""] * (n - len(figure_titles))
        else:
            titles = figure_titles[:n]

    # 2) Filtra solo le Plotly‐Figure valide, mantenendo i titoli corrispondenti
    valid_pairs = [
        (fig, title) for fig, title in zip(figs, titles)
        if (fig is not None and hasattr(fig, 'data'))
    ]
    if not valid_pairs:
        raise ValueError("Nessuna figura Plotly valida in input")

    # 3) Inizio HTML (includo Plotly.js una sola volta)
    html = [
        "<html>",
        "<head>",
        "  <meta charset='utf-8'/>",
        "  <script src='https://cdn.plot.ly/plotly-latest.min.js'></script>",
        "</head>",
        "<body>"
    ]

    # 4) Loop sulle figure filtrate
    for idx, (fig, title) in enumerate(valid_pairs):
        # Titolo (solo se non vuoto)
        if title:
            html.append(f"<h2>{title}</h2>")

        # Forza width/height se richiesto
        if fig_width is not None or fig_height is not None:
            fig.update_layout(
                width=fig_width  if fig_width  is not None else fig.layout.width,
                height=fig_height if fig_height is not None else fig.layout.height
            )

        # Esporto il frammento HTML
        fragment = fig.to_html(
            full_html=False,
            include_plotlyjs='cdn' if idx == 0 else False
        )
        html.append(fragment)
        html.append("<hr style='margin:40px 0;'/>")

    # 5) Chiudo HTML
    html += ["</body>", "</html>"]

    # 6) Scrivo su file
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write("\n".join(html))

    print(f"HTML Plots salvato in: {output_filename}")
    
def print_dict_kv(d: dict, indent: int = 0):
    """
    Stampa chiave -> valore, con indentazione.
    """
    pad = " " * indent
    for k, v in d.items():
        if isinstance(v, dict):
            print(f"{pad}{k}:")
            print_dict_kv(v, indent + 2)
        else:
            print(f"{pad}{k}: {BOLD}{v}{RESET}")

In [ ]:
# Controllo librerie per funzioni duplicate
print("\n")

prefix=None # tutte le funzioni
# prefix="strategy_" # verifica solo funzioni strategia


# dups = find_duplicate_function_defs_multi("../libs/*functions.ipynb, *strategies.ipynb, *tickers.ipynb, *porfolios.ipynb",prefix=prefix)

patterns = ",".join([
    "../libs/*functions.ipynb",
    "../libs/*strategies.ipynb",
    "../libs/*tickers.ipynb",
    "../libs/*porfolios.ipynb",
])

dups = find_duplicate_function_defs_multi(patterns, prefix=prefix)

print("\nLibreria u_functions importata")